# propagateInside_run

Notebook propre pour tester la propagation. Exécute de haut en bas.

La logique est volontairement comparative :

1. tester un petit baseline homogène 3x3 FD ;
2. tester le même baseline 3x3 FD sur Marmousi ;
3. construire tes opérateurs OPT numériques ;
4. propager avec les opérateurs OPT avec les mêmes sources/plots ;
5. comparer FD vs OPT.

## 1. Setup

In [ ]:
using Pkg
using Statistics
cd(@__DIR__)
Pkg.activate("../")

# GPU packages are optional in this notebook; solves below use CPU sparse matrices.
try
    using Metal
catch err
    @warn "Metal not loaded; continuing on CPU" err
end
ParamFile = "../config/testparam.csv" 
include("../src/batchFiles/batchGPU.jl")
include("../src/commonBatchs.jl")
include("../src/planet1D.jl")
include("../src/GeoPoints.jl")
using .commonBatchs, .planet1D, .GeoPoints

include("../src/flexOPT.jl")
using .flexOPT


## 2. Model And Global Controls

In [ ]:
# Model and run controls.
modelName = "marmousi"
imageFile = "../dataInput/model/random/marmousi.png"
modelDefinitionMethod = "2DimageFile"

Δspace = (1.e2, 1.e2)      # meters
cfl_safety = 0.45
pointsInTime = 3
Nt_debug = 160
store_every_debug = 2
source_amplitude_opt = 1e-6
source_amplitude_fd = 1.0
zoom_radius = 90

model = defineModel(imageFile)
boxGrids = lazyProduceOrLoad("MarmousiCoordInfo", constructLocalBox, model, -3000.0, 0.0, 0.0, 9200.0)
seismicModelMarmousi = lazyProduceOrLoad("seismicModelMarmousi", makeAdHocSeismicModel, model, 1.0, 2.8, 1.5, 5.5, 0.0, 3.2)
models = [seismicModelMarmousi.Vpv .* 1.e3] # MKS: m/s

finite_velocity = filter(isfinite, vec(Float64.(models[1])))
vmin, vmed, vmax = minimum(finite_velocity), median(finite_velocity), maximum(finite_velocity)
Δ = (Δspace..., cfl_safety * minimum(Float64.(Δspace)) / vmax)
Δnum = Δ

spaceShape = size(models[1])
sourcePoint = CartesianIndex(ntuple(d -> cld(spaceShape[d], 2), length(spaceShape))...)

@show spaceShape sourcePoint
@show vmin vmed vmax Δ


## 3. Helpers

In [ ]:
function source_time_samples(Nt, Δt, timePointsUsedForOneStep; wavelet=nothing, t0=12Δt, f0=0.04)
    ntime = Nt + timePointsUsedForOneStep - 1
    t = (0:ntime-1) .* Δt
    if wavelet === nothing
        return Ricker.(t, t0, f0)
    else
        return wavelet.(t)
    end
end

function _normalise_source_weights!(w; normalise=:none)
    if normalise == :none || normalise === false
        return w
    elseif normalise == :sum
        s = sum(w)
        !iszero(s) && (w ./= s)
    elseif normalise == :l1
        s = sum(abs, w)
        !iszero(s) && (w ./= s)
    elseif normalise == :max
        s = maximum(abs, w)
        !iszero(s) && (w ./= s)
    else
        error("normalise should be :none, :sum, :l1, or :max")
    end
    return w
end

function make_sourceFull(preparedLin, spatialWeights, timeSignal; iField=1, amplitude=1.0)
    length(spatialWeights) == preparedLin.NforcePoints ||
        error("spatialWeights should have length preparedLin.NforcePoints=$(preparedLin.NforcePoints)")
    NForceField = hasproperty(preparedLin, :NForceField) ? preparedLin.NForceField : preparedLin.NField
    1 <= iField <= NForceField || error("iField should be between 1 and $(NForceField)")

    T = promote_type(eltype(spatialWeights), eltype(timeSignal), typeof(amplitude))
    sourceFull = zeros(T, preparedLin.NforcePoints, NForceField, length(timeSignal))
    for it in eachindex(timeSignal)
        sourceFull[:, iField, it] .= amplitude .* timeSignal[it] .* spatialWeights
    end
    return sourceFull
end

function point_source_weights(preparedLin, point::CartesianIndex; normalise=:none)
    weights = zeros(Float64, preparedLin.NforcePoints)
    LI = LinearIndices(preparedLin.spaceShape)
    weights[LI[point]] = 1.0
    return _normalise_source_weights!(weights; normalise)
end

function hyperplane_source_weights(preparedLin; fixed=Dict{Int,Int}(), profile=(I -> 1.0), normalise=:sum)
    weights = zeros(Float64, preparedLin.NforcePoints)
    LI = LinearIndices(preparedLin.spaceShape)
    for I in CartesianIndices(preparedLin.spaceShape)
        coords = Tuple(I)
        inside = all(coords[d] == v for (d, v) in fixed)
        if inside
            weights[LI[I]] = profile(I)
        end
    end
    return _normalise_source_weights!(weights; normalise)
end

function mask_source_weights(preparedLin, mask; profile=(I -> 1.0), normalise=:sum)
    weights = zeros(Float64, preparedLin.NforcePoints)
    LI = LinearIndices(preparedLin.spaceShape)

    if mask isa AbstractArray{Bool}
        size(mask) == preparedLin.spaceShape || error("Boolean mask size should be $(preparedLin.spaceShape)")
        for I in CartesianIndices(mask)
            mask[I] && (weights[LI[I]] = profile(I))
        end
    else
        for I in mask
            weights[LI[I]] = profile(I)
        end
    end

    return _normalise_source_weights!(weights; normalise)
end


using LinearAlgebra, SparseArrays
using CairoMakie
CairoMakie.activate!()

function apply_forced_boundary_for_video!(A, b; leftValue=0.0, rightValue=0.0)
    A[1, :] .= 0
    A[1, 1] = 1
    b[1] = leftValue

    A[end, :] .= 0
    A[end, end] = 1
    b[end] = rightValue
    return A, b
end

function propagate_linear_frames(
    preparedLin,
    sourceFull,
    Nt;
    initialCondition=0.0,
    boundaryConditionForced=false,
    store_every=1,
    blowup_limit=Inf,
)
    NField = preparedLin.NField
    NForceField = hasproperty(preparedLin, :NForceField) ? preparedLin.NForceField : preparedLin.NField
    NpointsSpace = preparedLin.NpointsSpace
    timePointsUsedForOneStep = preparedLin.timePointsUsedForOneStep
    NknownTime = max(timePointsUsedForOneStep - 1, 0)

    size(sourceFull, 1) == preparedLin.NforcePoints || error("sourceFull first dimension mismatch")
    size(sourceFull, 2) == NForceField || error("sourceFull second dimension mismatch")
    size(sourceFull, 3) >= Nt + timePointsUsedForOneStep - 1 || error("sourceFull has too few time samples")

    knownField = fill(Float64(initialCondition), size(preparedLin.known_lhs_template))
    knownForce = fill(Float64(initialCondition), size(preparedLin.known_rhs_template))
    unknownField = fill(Float64(initialCondition), NpointsSpace, NField)

    A = sparse(preparedLin.A_template)
    b0 = copy(preparedLin.b_template)
    if boundaryConditionForced
        apply_forced_boundary_for_video!(A, b0; leftValue=0.0, rightValue=0.0)
    end
    factor = lu(A)
    b = copy(preparedLin.b_template)

    frames = Vector{Array{Float64}}()
    for it in 1:Nt
        knownForce .= sourceFull[:, :, it:it+timePointsUsedForOneStep-1]
        knownInputs = vcat(vec(knownField), vec(knownForce))
        preparedLin.b_fun!(b, knownInputs)
        if boundaryConditionForced
            b[1] = 0.0
            b[end] = 0.0
        end

        u = factor \ b
        unknownField .= reshape(real.(u), NpointsSpace, NField)

        umax = maximum(abs, unknownField)
        if !isfinite(umax) || umax > blowup_limit
            @warn "Stopping propagation because the wavefield blew up" it umax blowup_limit
            push!(frames, reshape(copy(unknownField[:, 1]), preparedLin.spaceShape...))
            break
        end

        if it == 1 || it % store_every == 0 || it == Nt
            push!(frames, reshape(copy(unknownField[:, 1]), preparedLin.spaceShape...))
        end

        if NknownTime > 0
            if NknownTime > 1
                knownField[:, :, 1:end-1] .= knownField[:, :, 2:end]
            end
            knownField[:, :, end] .= unknownField
        end
    end
    return frames
end

function _finite_copy(A; fillvalue=0.0)
    B = Array{Float64}(undef, size(A))
    replaced = 0
    for I in eachindex(A)
        v = Float64(real(A[I]))
        if isfinite(v)
            B[I] = v
        else
            B[I] = fillvalue
            replaced += 1
        end
    end
    return B, replaced
end

function finite_frame_diagnostics(frames)
    total_bad = 0
    first_bad = nothing
    finite_max = 0.0
    for (i, frame) in pairs(frames)
        bad_here = count(!isfinite, frame)
        if bad_here > 0 && first_bad === nothing
            first_bad = i
        end
        total_bad += bad_here
        for v in frame
            if isfinite(v)
                finite_max = max(finite_max, abs(Float64(v)))
            end
        end
    end
    return (total_bad=total_bad, first_bad=first_bad, finite_max=finite_max)
end

function _background_for_plot(background, spaceShape)
    background === nothing && return nothing
    bg = Array(background)
    if size(bg) == spaceShape
        clean_bg, replaced = _finite_copy(bg)
    elseif length(bg) == prod(spaceShape)
        clean_bg, replaced = _finite_copy(reshape(bg, spaceShape...))
    else
        @warn "background size $(size(bg)) does not match wavefield spaceShape $spaceShape; ignoring background"
        return nothing
    end
    replaced > 0 && @warn "background had $replaced non-finite values; replaced by 0 for plotting"
    return clean_bg
end

function record_wave_video(
    frames;
    videoFile=joinpath(@__DIR__, "point_source_propagation.mp4"),
    background=nothing,
    sourcePoint=nothing,
    framerate=20,
    title="point source propagation",
    colormap=:balance,
    background_colormap=:grays,
    wave_alpha=0.72,
    clim=nothing,
)
    isempty(frames) && error("frames is empty")
    spaceShape = size(frames[1])
    bg = _background_for_plot(background, spaceShape)
    nd = length(spaceShape)

    diag = finite_frame_diagnostics(frames)
    if diag.total_bad > 0
        @warn "wavefield frames contain $(diag.total_bad) non-finite values; first affected stored frame is $(diag.first_bad). Replacing by 0 for plotting."
    end
    clean_frames = [first(_finite_copy(f)) for f in frames]

    if clim === nothing
        vmax = diag.finite_max
        clim = vmax == 0 ? 1.0 : vmax
    end

    fig = Figure(size=(900, 680))
    ax = Axis(fig[1, 1], aspect=DataAspect(), title=title)

    if nd == 1
        x = 1:spaceShape[1]
        yobs = Observable(vec(clean_frames[1]))
        lines!(ax, x, yobs, color=:dodgerblue3, linewidth=2)
        ylims!(ax, -clim, clim)
        if sourcePoint !== nothing
            vlines!(ax, [Tuple(sourcePoint)[1]], color=:red, linewidth=2)
        end
        record(fig, videoFile, eachindex(frames); framerate=framerate) do iframe
            yobs[] = vec(clean_frames[iframe])
            ax.title = "$title  |  frame $iframe/$(length(frames))"
        end
    elseif nd == 2
        if bg !== nothing
            heatmap!(ax, bg; colormap=background_colormap)
        end
        frame_obs = Observable(clean_frames[1])
        hm = heatmap!(ax, frame_obs; colormap=colormap, colorrange=(-clim, clim), alpha=bg === nothing ? 1.0 : wave_alpha)
        Colorbar(fig[1, 2], hm, label="wavefield")
        if sourcePoint !== nothing
            xy = Tuple(sourcePoint)
            scatter!(ax, [xy[1]], [xy[2]], color=:red, markersize=14)
        end
        record(fig, videoFile, eachindex(frames); framerate=framerate) do iframe
            frame_obs[] = clean_frames[iframe]
            ax.title = "$title  |  frame $iframe/$(length(frames))"
        end
    else
        error("Video helper currently supports 1D/2D frames. For 3D/4D, select a slice before recording.")
    end

    return videoFile
end


function _finite_plot_matrix(A; fillvalue=0.0)
    B = Array{Float64}(undef, size(A))
    nbad = 0
    finite_max = 0.0
    for I in eachindex(A)
        v = Float64(real(A[I]))
        if isfinite(v)
            B[I] = v
            finite_max = max(finite_max, abs(v))
        else
            B[I] = fillvalue
            nbad += 1
        end
    end
    return B, nbad, finite_max
end

function wavefield_snapshot_report(frames)
    rows = NamedTuple[]
    for (i, frame) in pairs(frames)
        _, nbad, finite_max = _finite_plot_matrix(frame)
        push!(rows, (frame=i, nbad=nbad, finite_max=finite_max, minimum=minimum(skipmissing(vec(ifelse.(isfinite.(frame), frame, missing)))), maximum=maximum(skipmissing(vec(ifelse.(isfinite.(frame), frame, missing))))))
    end
    return rows
end


function _snapshot_background_for_plot(background, spaceShape)
    background === nothing && return nothing
    bg = Array(background)
    if size(bg) == spaceShape
        B, replaced, _ = _finite_plot_matrix(bg)
    elseif length(bg) == prod(spaceShape)
        B, replaced, _ = _finite_plot_matrix(reshape(bg, spaceShape...))
    else
        @warn "background size $(size(bg)) does not match wavefield spaceShape $spaceShape; ignoring background"
        return nothing
    end
    replaced > 0 && @warn "background had $replaced non-finite values; replaced by 0 for plotting"
    return B
end

function plot_wave_snapshots(
    frames;
    indices=nothing,
    background=nothing,
    sourcePoint=nothing,
    clim=nothing,
    ncols=3,
    title="wavefield snapshots",
    colormap=:balance,
    background_colormap=:grays,
    wave_alpha=0.72,
)
    isempty(frames) && error("frames is empty")
    if indices === nothing
        indices = unique(round.(Int, range(1, length(frames), length=min(9, length(frames)))))
    end
    indices = collect(indices)

    clean_frames = Array{Float64}[]
    nbad = Int[]
    finite_maxima = Float64[]
    for i in indices
        B, bad, fmax = _finite_plot_matrix(frames[i])
        push!(clean_frames, B)
        push!(nbad, bad)
        push!(finite_maxima, fmax)
    end

    if clim === nothing
        vmax = maximum(finite_maxima)
        clim = vmax == 0 ? 1.0 : vmax
    end

    bg = _snapshot_background_for_plot(background, size(frames[1]))
    n = length(indices)
    nrows = cld(n, ncols)
    fig = Figure(size=(320*ncols, 300*nrows))

    for (k, iframe) in enumerate(indices)
        r = cld(k, ncols)
        c = k - (r - 1) * ncols
        ax = Axis(fig[r, c], aspect=DataAspect(), title="frame $iframe | bad=$(nbad[k])")
        if length(size(frames[1])) == 1
            lines!(ax, vec(clean_frames[k]), color=:dodgerblue3, linewidth=2)
            ylims!(ax, -clim, clim)
            if sourcePoint !== nothing
                vlines!(ax, [Tuple(sourcePoint)[1]], color=:red, linewidth=2)
            end
        elseif length(size(frames[1])) == 2
            if bg !== nothing
                heatmap!(ax, bg; colormap=background_colormap)
            end
            heatmap!(ax, clean_frames[k]; colormap=colormap, colorrange=(-clim, clim), alpha=bg === nothing ? 1.0 : wave_alpha)
            if sourcePoint !== nothing
                xy = Tuple(sourcePoint)
                scatter!(ax, [xy[1]], [xy[2]], color=:red, markersize=10)
            end
        else
            error("Snapshot helper supports 1D/2D frames. Select a slice first for higher dimensions.")
        end
    end

    Label(fig[0, :], title)
    return fig
end


function save_wave_snapshot_pdf(
    frames;
    filename=joinpath(@__DIR__, "wave_snapshots.pdf"),
    indices=nothing,
    background=nothing,
    sourcePoint=nothing,
    clim=nothing,
    ncols=3,
    title="wavefield snapshots",
    kwargs...,
)
    fig = plot_wave_snapshots(
        frames;
        indices=indices,
        background=background,
        sourcePoint=sourcePoint,
        clim=clim,
        ncols=ncols,
        title=title,
        kwargs...,
    )
    save(filename, fig)
    empty!(fig.scene)  # release most displayed scene resources
    GC.gc()
    return filename
end


using Statistics

function cfl_diagnostics(model_velocity, Δnum; cfl_safety=0.45, ppw=10, samples_per_period=20)
    v = vec(Float64.(model_velocity))
    finite_v = filter(isfinite, v)
    dt = Float64(Δnum[end])
    dx = Float64.(Δnum[1:end-1])
    dxmin = minimum(dx)
    dxmax = maximum(dx)
    vmax = maximum(abs, finite_v)
    vmin = minimum(abs, filter(!iszero, finite_v))
    cfl = vmax * dt / dxmin

    # MKS units: velocity [m/s], dx [m], dt [s], frequency [Hz].
    # For a Ricker wavelet in flexOPT, f0 is the dominant/peak frequency.
    fmax_spatial = vmin / (ppw * dxmax)
    fmax_temporal = 1 / (samples_per_period * dt)
    suggested_ricker_f0 = min(fmax_spatial, fmax_temporal)

    return (
        v_min=minimum(finite_v),
        v_median=median(finite_v),
        v_max=maximum(finite_v),
        dt=dt,
        dxmin=dxmin,
        dxmax=dxmax,
        cfl_max=cfl,
        suggested_dt_2D=cfl_safety * dxmin / vmax,
        ppw=ppw,
        samples_per_period=samples_per_period,
        fmax_spatial=fmax_spatial,
        fmax_temporal=fmax_temporal,
        suggested_ricker_f0=suggested_ricker_f0,
        suggested_ricker_t0=1.5 / suggested_ricker_f0,
    )
end

function ricker_time_signal(Nt, Δt; f0, t0=1.5/f0, timePointsUsedForOneStep=1)
    ntime = Nt + timePointsUsedForOneStep - 1
    t = (0:ntime-1) .* Δt
    return t, Ricker.(t, t0, f0)
end

function dominant_frequency_scan(signal, Δt; nfreq=512)
    y = Float64.(signal) .- mean(Float64.(signal))
    n = length(y)
    duration = n * Δt
    fmin = 1 / duration
    fmax = 0.5 / Δt
    freqs = range(fmin, fmax; length=nfreq)
    t = (0:n-1) .* Δt
    spectrum = [abs(sum(y .* exp.(-2im * π * f .* t))) for f in freqs]
    imax = argmax(spectrum)
    return (f_peak=freqs[imax], peak_amplitude=spectrum[imax], frequencies=freqs, spectrum=spectrum)
end

function ricker_diagnostics(model_velocity, Δnum; f0=nothing, Nt=200, timePointsUsedForOneStep=1, ppw=10, samples_per_period=20, t0=nothing)
    cfl = cfl_diagnostics(model_velocity, Δnum; ppw=ppw, samples_per_period=samples_per_period)
    f0_used = f0 === nothing ? cfl.suggested_ricker_f0 : Float64(f0)
    t0_used = t0 === nothing ? 1.5 / f0_used : Float64(t0)
    t, signal = ricker_time_signal(Nt, cfl.dt; f0=f0_used, t0=t0_used, timePointsUsedForOneStep=timePointsUsedForOneStep)
    freqinfo = dominant_frequency_scan(signal, cfl.dt)
    return (
        f0=f0_used,
        t0=t0_used,
        duration=t[end],
        dominant_frequency_scan=freqinfo.f_peak,
        wavelength_min_velocity=cfl.v_min / f0_used,
        points_per_wavelength_min_velocity=(cfl.v_min / f0_used) / cfl.dxmax,
        samples_per_period=1 / (f0_used * cfl.dt),
        max_abs=maximum(abs, signal),
        first_peak_time=t[argmax(abs.(signal))],
        cfl=cfl,
    )
end

function frame_motion_report(frames)
    rows = NamedTuple[]
    previous = nothing
    for (i, frame) in pairs(frames)
        _, nbad, fmax = _finite_plot_matrix(frame)
        dnorm = previous === nothing ? 0.0 : norm(frame .- previous)
        dmax = previous === nothing ? 0.0 : maximum(abs, frame .- previous)
        push!(rows, (frame=i, nbad=nbad, maxabs=fmax, norm=norm(frame), diff_norm=dnorm, diff_max=dmax))
        previous = frame
    end
    return rows
end

function difference_frames(frames)
    length(frames) >= 2 || error("need at least two frames")
    return [frames[i] .- frames[i-1] for i in 2:length(frames)]
end


function _sparse_norms(A)
    return (
        size=size(A),
        nnz=nnz(A),
        maxabs=nnz(A) == 0 ? 0.0 : maximum(abs, nonzeros(A)),
        norm=norm(A),
    )
end

function diagnose_prepared_linear_system(preparedLin)
    println("spaceShape = ", preparedLin.spaceShape)
    println("NpointsSpace = ", preparedLin.NpointsSpace)
    println("NField = ", preparedLin.NField)
    println("NForceField = ", hasproperty(preparedLin, :NForceField) ? preparedLin.NForceField : preparedLin.NField)
    println("timePointsUsedForOneStep = ", preparedLin.timePointsUsedForOneStep)
    println("A_unknown: ", _sparse_norms(preparedLin.A_unknown))
    println("L_known:   ", _sparse_norms(preparedLin.L_known))
    println("R_force:   ", _sparse_norms(preparedLin.R_force))
    if hasproperty(preparedLin, :right_operator)
        println("right_operator.size = ", preparedLin.right_operator.size)
        println("right_operator.nnz = ", length(preparedLin.right_operator.table.vals))
    end
    row_nnz_A = vec(sum(abs.(preparedLin.A_unknown) .> 0; dims=2))
    row_nnz_R = vec(sum(abs.(preparedLin.R_force) .> 0; dims=2))
    println("rows with A entries = ", count(!iszero, row_nnz_A), " / ", length(row_nnz_A))
    println("rows with R entries = ", count(!iszero, row_nnz_R), " / ", length(row_nnz_R))
    return nothing
end

function source_field_response(preparedLin, sourcePoint, timeSignal; amplitude=1.0, it=nothing)
    LI = LinearIndices(preparedLin.spaceShape)
    src_linear = LI[sourcePoint]
    last_start = length(timeSignal) - preparedLin.timePointsUsedForOneStep + 1
    last_start >= 1 || error("timeSignal is too short for timePointsUsedForOneStep=$(preparedLin.timePointsUsedForOneStep)")
    if it === nothing
        it = min(argmax(abs.(timeSignal)), last_start)
    else
        1 <= it <= last_start || error("it should be in 1:$last_start for this timeSignal")
    end

    out = NamedTuple[]
    NForceField = hasproperty(preparedLin, :NForceField) ? preparedLin.NForceField : preparedLin.NField
    for iField in 1:NForceField
        w = zeros(Float64, preparedLin.NforcePoints)
        w[src_linear] = 1.0
        sourceFull = make_sourceFull(preparedLin, w, timeSignal; iField=iField, amplitude=amplitude)
        knownForce = sourceFull[:, :, it:it+preparedLin.timePointsUsedForOneStep-1]
        knownInputs = vcat(vec(zero(preparedLin.known_lhs_template)), vec(knownForce))
        b = copy(preparedLin.b_template)
        preparedLin.b_fun!(b, knownInputs)
        push!(out, (
            iField=iField,
            time_index=it,
            source_norm=norm(knownForce),
            b_norm=norm(b),
            b_maximum=maximum(abs, b),
            b_nonzero=count(!iszero, b),
        ))
    end
    return out
end

function one_step_response(preparedLin, sourceFull; it=1, boundaryConditionForced=false)
    knownField = zero(preparedLin.known_lhs_template)
    knownForce = sourceFull[:, :, it:it+preparedLin.timePointsUsedForOneStep-1]
    knownInputs = vcat(vec(knownField), vec(knownForce))
    b = copy(preparedLin.b_template)
    preparedLin.b_fun!(b, knownInputs)

    A = sparse(preparedLin.A_template)
    if boundaryConditionForced
        apply_forced_boundary_for_video!(A, b; leftValue=0.0, rightValue=0.0)
    end
    u = A \ b
    ufield = reshape(real.(u), preparedLin.NpointsSpace, preparedLin.NField)
    return (
        b_norm=norm(b),
        b_maximum=maximum(abs, b),
        b_nonzero=count(!iszero, b),
        u_norm=norm(u),
        u_maximum=maximum(abs, u),
        u_nonzero=count(!iszero, u),
        ufield=ufield,
    )
end


function _matrix_row_abs_sums(A)
    return vec(sum(abs.(A); dims=2))
end

function operator_growth_diagnostics(preparedLin)
    A = sparse(preparedLin.A_unknown)
    L = sparse(preparedLin.L_known)
    R = sparse(preparedLin.R_force)
    diagA = abs.(diag(A))
    rowA = _matrix_row_abs_sums(A)
    offdiagA = rowA .- diagA
    rowL = _matrix_row_abs_sums(L)
    rowR = _matrix_row_abs_sums(R)
    return (
        A_size=size(A),
        A_nnz=nnz(A),
        A_diag_min=isempty(diagA) ? 0.0 : minimum(diagA),
        A_diag_median=isempty(diagA) ? 0.0 : median(diagA),
        A_diag_max=isempty(diagA) ? 0.0 : maximum(diagA),
        A_offdiag_over_diag_max=maximum(offdiagA ./ max.(diagA, eps(Float64))),
        L_row_sum_max=maximum(rowL),
        R_row_sum_max=maximum(rowR),
        R_over_A_diag_max=maximum(rowR ./ max.(diagA, eps(Float64))),
        condest_dense_sample=size(A,1) <= 5000 ? cond(Matrix(A)) : missing,
    )
end

function homogeneous_zero_test(preparedLin; Nt=20, store_every=1)
    sourceFull_zero = zeros(eltype(preparedLin.known_rhs_template), preparedLin.NforcePoints, preparedLin.NForceField, Nt + preparedLin.timePointsUsedForOneStep - 1)
    frames_zero = propagate_linear_frames(preparedLin, sourceFull_zero, Nt; store_every=store_every, blowup_limit=1e12)
    return wavefield_snapshot_report(frames_zero)
end

function source_amplitude_sweep(preparedLin, sourcePoint, timeSignal; amplitudes=(1e-12, 1e-10, 1e-8, 1e-6), iField=1, Nt=20)
    rows = NamedTuple[]
    w = point_source_weights(preparedLin, sourcePoint)
    for amp in amplitudes
        sourceFull = make_sourceFull(preparedLin, w, timeSignal; iField=iField, amplitude=amp)
        frames = propagate_linear_frames(preparedLin, sourceFull, Nt; store_every=max(1, Nt ÷ 5), blowup_limit=1e30)
        report = wavefield_snapshot_report(frames)
        push!(rows, (amplitude=amp, last=report[end], nframes=length(frames)))
    end
    return rows
end


function _distance_from_source_table(frame, sourcePoint; threshold_fraction=1e-6)
    f = abs.(Float64.(frame))
    fmax = maximum(f)
    threshold = threshold_fraction * max(fmax, eps(Float64))
    active = findall(>(threshold), f)
    if isempty(active)
        return (nactive=0, max_distance=0.0, mean_distance=0.0, fmax=fmax)
    end
    distances = [sqrt(sum((Tuple(I) .- Tuple(sourcePoint)).^2)) for I in active]
    return (
        nactive=length(active),
        max_distance=maximum(distances),
        mean_distance=mean(distances),
        fmax=fmax,
    )
end

function propagation_radius_report(frames, sourcePoint; threshold_fraction=1e-6)
    return [(frame=i, _distance_from_source_table(frame, sourcePoint; threshold_fraction=threshold_fraction)...) for (i, frame) in pairs(frames)]
end

function impulse_sourceFull(preparedLin, sourcePoint; iField=1, amplitude=1.0, Nt=20, impulse_time=1)
    timeSignal = zeros(Float64, Nt + preparedLin.timePointsUsedForOneStep - 1)
    timeSignal[impulse_time] = 1.0
    w = point_source_weights(preparedLin, sourcePoint)
    return make_sourceFull(preparedLin, w, timeSignal; iField=iField, amplitude=amplitude)
end

function impulse_propagation_test(preparedLin, sourcePoint; iField=1, amplitude=1.0, Nt=20, store_every=1, threshold_fraction=1e-8)
    sourceFull = impulse_sourceFull(preparedLin, sourcePoint; iField=iField, amplitude=amplitude, Nt=Nt)
    frames = propagate_linear_frames(preparedLin, sourceFull, Nt; store_every=store_every, blowup_limit=1e20)
    return (
        frames=frames,
        snapshots=wavefield_snapshot_report(frames),
        radius=propagation_radius_report(frames, sourcePoint; threshold_fraction=threshold_fraction),
    )
end


function frame_extrema_report(frames, sourcePoint)
    rows = NamedTuple[]
    for (i, frame) in pairs(frames)
        A = Float64.(frame)
        absA = abs.(A)
        imax = argmax(absA)
        maxpoint = CartesianIndices(size(A))[imax]
        distance = sqrt(sum((Tuple(maxpoint) .- Tuple(sourcePoint)).^2))
        push!(rows, (
            frame=i,
            maxpoint=maxpoint,
            distance_from_source=distance,
            value=A[imax],
            maxabs=absA[imax],
            minimum=minimum(A),
            maximum=maximum(A),
        ))
    end
    return rows
end


function _get_numop(numOps, which::Symbol)
    ops = numOps isa AbstractDict ? numOps["numericalOperators"] : numOps.numericalOperators
    return getproperty(ops, which)
end

function _point_linear_index(op, point)
    νWhole = op.geometry.νWhole
    if point isa Integer
        1 <= point <= length(νWhole) || error("point integer should be in 1:$(length(νWhole))")
        return Int(point), νWhole[Int(point)]
    elseif point isa CartesianIndex
        point in νWhole || error("point $point is not in op.geometry.νWhole")
        pointLinear = LinearIndices(νWhole)[point]
        return Int(pointLinear), point
    else
        error("point should be an Int linear point index or a CartesianIndex")
    end
end

function _time_role(iT, activeTimePoints)
    if iT == activeTimePoints
        return :future_unknown
    elseif iT == activeTimePoints - 1
        return :present_known
    else
        return Symbol("past_known_tminus$(activeTimePoints - iT)")
    end
end

function coefficient_rows_at_point(op, point; iExpr=1, atol=0.0)
    pointLinear, pointCI = _point_linear_index(op, point)
    nPoints = length(op.geometry.νWhole)
    activeTimePoints = op.geometry.activeTimePoints
    spaceShape = Tuple(op.geometry.wholeRegionPointsSpace)
    Nexpr = op.size[1] ÷ nPoints
    Nfield = op.size[2] ÷ (nPoints * activeTimePoints)

    1 <= iExpr <= Nexpr || error("iExpr should be in 1:$Nexpr")

    residualLI = LinearIndices((Nexpr, nPoints))
    targetRow = residualLI[iExpr, pointLinear]
    fieldTimeSpaceCI = CartesianIndices((Nfield, activeTimePoints, spaceShape...))

    rows = NamedTuple[]
    for k in eachindex(op.table.vals)
        Int(op.table.rows[k]) == targetRow || continue
        val = op.table.vals[k]
        abs(val) > atol || continue

        ci = fieldTimeSpaceCI[Int(op.table.cols[k])]
        iField = ci[1]
        iT = ci[2]
        neighbour = CartesianIndex(Tuple(ci)[3:end])
        offset = Tuple(neighbour - pointCI)
        timeRole = _time_role(iT, activeTimePoints)

        push!(rows, (
            k = k,
            residual_row = targetRow,
            expr = iExpr,
            point = pointCI,
            field = iField,
            time_slot = iT,
            time_role = timeRole,
            neighbour = neighbour,
            offset = offset,
            coef = val,
            abscoef = abs(val),
        ))
    end

    sort!(rows, by = r -> (r.time_slot, r.field, r.offset))
    return rows
end

function inspect_operator_coefficients_at_point(numOps, point; which=:residual, iExpr=1, atol=0.0)
    op = _get_numop(numOps, which)
    rows = coefficient_rows_at_point(op, point; iExpr=iExpr, atol=atol)
    println("operator = ", which)
    println("point = ", point, ", iExpr = ", iExpr, ", ncoef = ", length(rows))
    for r in rows
        println(
            "t=", r.time_slot,
            " (", r.time_role, ")",
            " field=", r.field,
            " offset=", r.offset,
            " neighbour=", r.neighbour,
            " coef=", r.coef,
        )
    end
    return rows
end

function coefficient_table_by_time(rows)
    slots = sort(unique(getproperty.(rows, :time_slot)))
    return [(
        time_slot=s,
        time_role=first(r.time_role for r in rows if r.time_slot == s),
        n=length(filter(r -> r.time_slot == s, rows)),
        sumcoef=sum(r.coef for r in rows if r.time_slot == s),
        sumabs=sum(r.abscoef for r in rows if r.time_slot == s),
    ) for s in slots]
end

function coefficient_rows_for_role(rows, role::Symbol)
    return filter(r -> r.time_role == role, rows)
end


function source_time_samples(Nt, Δt, timePointsUsedForOneStep; wavelet=nothing, t0=12Δt, f0=0.04)
    ntime = Nt + timePointsUsedForOneStep - 1
    t = (0:ntime-1) .* Δt
    if wavelet === nothing
        return Ricker.(t, t0, f0)
    else
        return wavelet.(t)
    end
end

function _normalise_source_weights!(w; normalise=:none)
    if normalise == :none || normalise === false
        return w
    elseif normalise == :sum
        s = sum(w)
        !iszero(s) && (w ./= s)
    elseif normalise == :l1
        s = sum(abs, w)
        !iszero(s) && (w ./= s)
    elseif normalise == :max
        s = maximum(abs, w)
        !iszero(s) && (w ./= s)
    else
        error("normalise should be :none, :sum, :l1, or :max")
    end
    return w
end

function make_sourceFull(preparedLin, spatialWeights, timeSignal; iField=1, amplitude=1.0)
    length(spatialWeights) == preparedLin.NforcePoints ||
        error("spatialWeights should have length preparedLin.NforcePoints=$(preparedLin.NforcePoints)")
    NForceField = hasproperty(preparedLin, :NForceField) ? preparedLin.NForceField : preparedLin.NField
    1 <= iField <= NForceField || error("iField should be between 1 and $(NForceField)")

    T = promote_type(eltype(spatialWeights), eltype(timeSignal), typeof(amplitude))
    sourceFull = zeros(T, preparedLin.NforcePoints, NForceField, length(timeSignal))
    for it in eachindex(timeSignal)
        sourceFull[:, iField, it] .= amplitude .* timeSignal[it] .* spatialWeights
    end
    return sourceFull
end

function point_source_weights(preparedLin, point::CartesianIndex; normalise=:none)
    weights = zeros(Float64, preparedLin.NforcePoints)
    LI = LinearIndices(preparedLin.spaceShape)
    weights[LI[point]] = 1.0
    return _normalise_source_weights!(weights; normalise)
end

function hyperplane_source_weights(preparedLin; fixed=Dict{Int,Int}(), profile=(I -> 1.0), normalise=:sum)
    weights = zeros(Float64, preparedLin.NforcePoints)
    LI = LinearIndices(preparedLin.spaceShape)
    for I in CartesianIndices(preparedLin.spaceShape)
        coords = Tuple(I)
        inside = all(coords[d] == v for (d, v) in fixed)
        if inside
            weights[LI[I]] = profile(I)
        end
    end
    return _normalise_source_weights!(weights; normalise)
end

function mask_source_weights(preparedLin, mask; profile=(I -> 1.0), normalise=:sum)
    weights = zeros(Float64, preparedLin.NforcePoints)
    LI = LinearIndices(preparedLin.spaceShape)

    if mask isa AbstractArray{Bool}
        size(mask) == preparedLin.spaceShape || error("Boolean mask size should be $(preparedLin.spaceShape)")
        for I in CartesianIndices(mask)
            mask[I] && (weights[LI[I]] = profile(I))
        end
    else
        for I in mask
            weights[LI[I]] = profile(I)
        end
    end

    return _normalise_source_weights!(weights; normalise)
end


using LinearAlgebra, SparseArrays
using CairoMakie
CairoMakie.activate!()

function apply_forced_boundary_for_video!(A, b; leftValue=0.0, rightValue=0.0)
    A[1, :] .= 0
    A[1, 1] = 1
    b[1] = leftValue

    A[end, :] .= 0
    A[end, end] = 1
    b[end] = rightValue
    return A, b
end

function propagate_linear_frames(
    preparedLin,
    sourceFull,
    Nt;
    initialCondition=0.0,
    boundaryConditionForced=false,
    store_every=1,
    blowup_limit=Inf,
)
    NField = preparedLin.NField
    NForceField = hasproperty(preparedLin, :NForceField) ? preparedLin.NForceField : preparedLin.NField
    NpointsSpace = preparedLin.NpointsSpace
    timePointsUsedForOneStep = preparedLin.timePointsUsedForOneStep
    NknownTime = max(timePointsUsedForOneStep - 1, 0)

    size(sourceFull, 1) == preparedLin.NforcePoints || error("sourceFull first dimension mismatch")
    size(sourceFull, 2) == NForceField || error("sourceFull second dimension mismatch")
    size(sourceFull, 3) >= Nt + timePointsUsedForOneStep - 1 || error("sourceFull has too few time samples")

    knownField = fill(Float64(initialCondition), size(preparedLin.known_lhs_template))
    knownForce = fill(Float64(initialCondition), size(preparedLin.known_rhs_template))
    unknownField = fill(Float64(initialCondition), NpointsSpace, NField)

    A = sparse(preparedLin.A_template)
    b0 = copy(preparedLin.b_template)
    if boundaryConditionForced
        apply_forced_boundary_for_video!(A, b0; leftValue=0.0, rightValue=0.0)
    end
    factor = lu(A)
    b = copy(preparedLin.b_template)

    frames = Vector{Array{Float64}}()
    for it in 1:Nt
        knownForce .= sourceFull[:, :, it:it+timePointsUsedForOneStep-1]
        knownInputs = vcat(vec(knownField), vec(knownForce))
        preparedLin.b_fun!(b, knownInputs)
        if boundaryConditionForced
            b[1] = 0.0
            b[end] = 0.0
        end

        u = factor \ b
        unknownField .= reshape(real.(u), NpointsSpace, NField)

        umax = maximum(abs, unknownField)
        if !isfinite(umax) || umax > blowup_limit
            @warn "Stopping propagation because the wavefield blew up" it umax blowup_limit
            push!(frames, reshape(copy(unknownField[:, 1]), preparedLin.spaceShape...))
            break
        end

        if it == 1 || it % store_every == 0 || it == Nt
            push!(frames, reshape(copy(unknownField[:, 1]), preparedLin.spaceShape...))
        end

        if NknownTime > 0
            if NknownTime > 1
                knownField[:, :, 1:end-1] .= knownField[:, :, 2:end]
            end
            knownField[:, :, end] .= unknownField
        end
    end
    return frames
end

function _finite_copy(A; fillvalue=0.0)
    B = Array{Float64}(undef, size(A))
    replaced = 0
    for I in eachindex(A)
        v = Float64(real(A[I]))
        if isfinite(v)
            B[I] = v
        else
            B[I] = fillvalue
            replaced += 1
        end
    end
    return B, replaced
end

function finite_frame_diagnostics(frames)
    total_bad = 0
    first_bad = nothing
    finite_max = 0.0
    for (i, frame) in pairs(frames)
        bad_here = count(!isfinite, frame)
        if bad_here > 0 && first_bad === nothing
            first_bad = i
        end
        total_bad += bad_here
        for v in frame
            if isfinite(v)
                finite_max = max(finite_max, abs(Float64(v)))
            end
        end
    end
    return (total_bad=total_bad, first_bad=first_bad, finite_max=finite_max)
end

function _background_for_plot(background, spaceShape)
    background === nothing && return nothing
    bg = Array(background)
    if size(bg) == spaceShape
        clean_bg, replaced = _finite_copy(bg)
    elseif length(bg) == prod(spaceShape)
        clean_bg, replaced = _finite_copy(reshape(bg, spaceShape...))
    else
        @warn "background size $(size(bg)) does not match wavefield spaceShape $spaceShape; ignoring background"
        return nothing
    end
    replaced > 0 && @warn "background had $replaced non-finite values; replaced by 0 for plotting"
    return clean_bg
end

function record_wave_video(
    frames;
    videoFile=joinpath(@__DIR__, "point_source_propagation.mp4"),
    background=nothing,
    sourcePoint=nothing,
    framerate=20,
    title="point source propagation",
    colormap=:balance,
    background_colormap=:grays,
    wave_alpha=0.72,
    clim=nothing,
)
    isempty(frames) && error("frames is empty")
    spaceShape = size(frames[1])
    bg = _background_for_plot(background, spaceShape)
    nd = length(spaceShape)

    diag = finite_frame_diagnostics(frames)
    if diag.total_bad > 0
        @warn "wavefield frames contain $(diag.total_bad) non-finite values; first affected stored frame is $(diag.first_bad). Replacing by 0 for plotting."
    end
    clean_frames = [first(_finite_copy(f)) for f in frames]

    if clim === nothing
        vmax = diag.finite_max
        clim = vmax == 0 ? 1.0 : vmax
    end

    fig = Figure(size=(900, 680))
    ax = Axis(fig[1, 1], aspect=DataAspect(), title=title)

    if nd == 1
        x = 1:spaceShape[1]
        yobs = Observable(vec(clean_frames[1]))
        lines!(ax, x, yobs, color=:dodgerblue3, linewidth=2)
        ylims!(ax, -clim, clim)
        if sourcePoint !== nothing
            vlines!(ax, [Tuple(sourcePoint)[1]], color=:red, linewidth=2)
        end
        record(fig, videoFile, eachindex(frames); framerate=framerate) do iframe
            yobs[] = vec(clean_frames[iframe])
            ax.title = "$title  |  frame $iframe/$(length(frames))"
        end
    elseif nd == 2
        if bg !== nothing
            heatmap!(ax, bg; colormap=background_colormap)
        end
        frame_obs = Observable(clean_frames[1])
        hm = heatmap!(ax, frame_obs; colormap=colormap, colorrange=(-clim, clim), alpha=bg === nothing ? 1.0 : wave_alpha)
        Colorbar(fig[1, 2], hm, label="wavefield")
        if sourcePoint !== nothing
            xy = Tuple(sourcePoint)
            scatter!(ax, [xy[1]], [xy[2]], color=:red, markersize=14)
        end
        record(fig, videoFile, eachindex(frames); framerate=framerate) do iframe
            frame_obs[] = clean_frames[iframe]
            ax.title = "$title  |  frame $iframe/$(length(frames))"
        end
    else
        error("Video helper currently supports 1D/2D frames. For 3D/4D, select a slice before recording.")
    end

    return videoFile
end


function _finite_plot_matrix(A; fillvalue=0.0)
    B = Array{Float64}(undef, size(A))
    nbad = 0
    finite_max = 0.0
    for I in eachindex(A)
        v = Float64(real(A[I]))
        if isfinite(v)
            B[I] = v
            finite_max = max(finite_max, abs(v))
        else
            B[I] = fillvalue
            nbad += 1
        end
    end
    return B, nbad, finite_max
end

function wavefield_snapshot_report(frames)
    rows = NamedTuple[]
    for (i, frame) in pairs(frames)
        _, nbad, finite_max = _finite_plot_matrix(frame)
        push!(rows, (frame=i, nbad=nbad, finite_max=finite_max, minimum=minimum(skipmissing(vec(ifelse.(isfinite.(frame), frame, missing)))), maximum=maximum(skipmissing(vec(ifelse.(isfinite.(frame), frame, missing))))))
    end
    return rows
end


function _snapshot_background_for_plot(background, spaceShape)
    background === nothing && return nothing
    bg = Array(background)
    if size(bg) == spaceShape
        B, replaced, _ = _finite_plot_matrix(bg)
    elseif length(bg) == prod(spaceShape)
        B, replaced, _ = _finite_plot_matrix(reshape(bg, spaceShape...))
    else
        @warn "background size $(size(bg)) does not match wavefield spaceShape $spaceShape; ignoring background"
        return nothing
    end
    replaced > 0 && @warn "background had $replaced non-finite values; replaced by 0 for plotting"
    return B
end

function plot_wave_snapshots(
    frames;
    indices=nothing,
    background=nothing,
    sourcePoint=nothing,
    clim=nothing,
    ncols=3,
    title="wavefield snapshots",
    colormap=:balance,
    background_colormap=:grays,
    wave_alpha=0.72,
)
    isempty(frames) && error("frames is empty")
    if indices === nothing
        indices = unique(round.(Int, range(1, length(frames), length=min(9, length(frames)))))
    end
    indices = collect(indices)

    clean_frames = Array{Float64}[]
    nbad = Int[]
    finite_maxima = Float64[]
    for i in indices
        B, bad, fmax = _finite_plot_matrix(frames[i])
        push!(clean_frames, B)
        push!(nbad, bad)
        push!(finite_maxima, fmax)
    end

    if clim === nothing
        vmax = maximum(finite_maxima)
        clim = vmax == 0 ? 1.0 : vmax
    end

    bg = _snapshot_background_for_plot(background, size(frames[1]))
    n = length(indices)
    nrows = cld(n, ncols)
    fig = Figure(size=(320*ncols, 300*nrows))

    for (k, iframe) in enumerate(indices)
        r = cld(k, ncols)
        c = k - (r - 1) * ncols
        ax = Axis(fig[r, c], aspect=DataAspect(), title="frame $iframe | bad=$(nbad[k])")
        if length(size(frames[1])) == 1
            lines!(ax, vec(clean_frames[k]), color=:dodgerblue3, linewidth=2)
            ylims!(ax, -clim, clim)
            if sourcePoint !== nothing
                vlines!(ax, [Tuple(sourcePoint)[1]], color=:red, linewidth=2)
            end
        elseif length(size(frames[1])) == 2
            if bg !== nothing
                heatmap!(ax, bg; colormap=background_colormap)
            end
            heatmap!(ax, clean_frames[k]; colormap=colormap, colorrange=(-clim, clim), alpha=bg === nothing ? 1.0 : wave_alpha)
            if sourcePoint !== nothing
                xy = Tuple(sourcePoint)
                scatter!(ax, [xy[1]], [xy[2]], color=:red, markersize=10)
            end
        else
            error("Snapshot helper supports 1D/2D frames. Select a slice first for higher dimensions.")
        end
    end

    Label(fig[0, :], title)
    return fig
end


function save_wave_snapshot_pdf(
    frames;
    filename=joinpath(@__DIR__, "wave_snapshots.pdf"),
    indices=nothing,
    background=nothing,
    sourcePoint=nothing,
    clim=nothing,
    ncols=3,
    title="wavefield snapshots",
    kwargs...,
)
    fig = plot_wave_snapshots(
        frames;
        indices=indices,
        background=background,
        sourcePoint=sourcePoint,
        clim=clim,
        ncols=ncols,
        title=title,
        kwargs...,
    )
    save(filename, fig)
    empty!(fig.scene)  # release most displayed scene resources
    GC.gc()
    return filename
end


using Statistics

function cfl_diagnostics(model_velocity, Δnum; cfl_safety=0.45, ppw=10, samples_per_period=20)
    v = vec(Float64.(model_velocity))
    finite_v = filter(isfinite, v)
    dt = Float64(Δnum[end])
    dx = Float64.(Δnum[1:end-1])
    dxmin = minimum(dx)
    dxmax = maximum(dx)
    vmax = maximum(abs, finite_v)
    vmin = minimum(abs, filter(!iszero, finite_v))
    cfl = vmax * dt / dxmin

    # MKS units: velocity [m/s], dx [m], dt [s], frequency [Hz].
    # For a Ricker wavelet in flexOPT, f0 is the dominant/peak frequency.
    fmax_spatial = vmin / (ppw * dxmax)
    fmax_temporal = 1 / (samples_per_period * dt)
    suggested_ricker_f0 = min(fmax_spatial, fmax_temporal)

    return (
        v_min=minimum(finite_v),
        v_median=median(finite_v),
        v_max=maximum(finite_v),
        dt=dt,
        dxmin=dxmin,
        dxmax=dxmax,
        cfl_max=cfl,
        suggested_dt_2D=cfl_safety * dxmin / vmax,
        ppw=ppw,
        samples_per_period=samples_per_period,
        fmax_spatial=fmax_spatial,
        fmax_temporal=fmax_temporal,
        suggested_ricker_f0=suggested_ricker_f0,
        suggested_ricker_t0=1.5 / suggested_ricker_f0,
    )
end

function ricker_time_signal(Nt, Δt; f0, t0=1.5/f0, timePointsUsedForOneStep=1)
    ntime = Nt + timePointsUsedForOneStep - 1
    t = (0:ntime-1) .* Δt
    return t, Ricker.(t, t0, f0)
end

function dominant_frequency_scan(signal, Δt; nfreq=512)
    y = Float64.(signal) .- mean(Float64.(signal))
    n = length(y)
    duration = n * Δt
    fmin = 1 / duration
    fmax = 0.5 / Δt
    freqs = range(fmin, fmax; length=nfreq)
    t = (0:n-1) .* Δt
    spectrum = [abs(sum(y .* exp.(-2im * π * f .* t))) for f in freqs]
    imax = argmax(spectrum)
    return (f_peak=freqs[imax], peak_amplitude=spectrum[imax], frequencies=freqs, spectrum=spectrum)
end

function ricker_diagnostics(model_velocity, Δnum; f0=nothing, Nt=200, timePointsUsedForOneStep=1, ppw=10, samples_per_period=20, t0=nothing)
    cfl = cfl_diagnostics(model_velocity, Δnum; ppw=ppw, samples_per_period=samples_per_period)
    f0_used = f0 === nothing ? cfl.suggested_ricker_f0 : Float64(f0)
    t0_used = t0 === nothing ? 1.5 / f0_used : Float64(t0)
    t, signal = ricker_time_signal(Nt, cfl.dt; f0=f0_used, t0=t0_used, timePointsUsedForOneStep=timePointsUsedForOneStep)
    freqinfo = dominant_frequency_scan(signal, cfl.dt)
    return (
        f0=f0_used,
        t0=t0_used,
        duration=t[end],
        dominant_frequency_scan=freqinfo.f_peak,
        wavelength_min_velocity=cfl.v_min / f0_used,
        points_per_wavelength_min_velocity=(cfl.v_min / f0_used) / cfl.dxmax,
        samples_per_period=1 / (f0_used * cfl.dt),
        max_abs=maximum(abs, signal),
        first_peak_time=t[argmax(abs.(signal))],
        cfl=cfl,
    )
end

function frame_motion_report(frames)
    rows = NamedTuple[]
    previous = nothing
    for (i, frame) in pairs(frames)
        _, nbad, fmax = _finite_plot_matrix(frame)
        dnorm = previous === nothing ? 0.0 : norm(frame .- previous)
        dmax = previous === nothing ? 0.0 : maximum(abs, frame .- previous)
        push!(rows, (frame=i, nbad=nbad, maxabs=fmax, norm=norm(frame), diff_norm=dnorm, diff_max=dmax))
        previous = frame
    end
    return rows
end

function difference_frames(frames)
    length(frames) >= 2 || error("need at least two frames")
    return [frames[i] .- frames[i-1] for i in 2:length(frames)]
end


function _sparse_norms(A)
    return (
        size=size(A),
        nnz=nnz(A),
        maxabs=nnz(A) == 0 ? 0.0 : maximum(abs, nonzeros(A)),
        norm=norm(A),
    )
end

function diagnose_prepared_linear_system(preparedLin)
    println("spaceShape = ", preparedLin.spaceShape)
    println("NpointsSpace = ", preparedLin.NpointsSpace)
    println("NField = ", preparedLin.NField)
    println("NForceField = ", hasproperty(preparedLin, :NForceField) ? preparedLin.NForceField : preparedLin.NField)
    println("timePointsUsedForOneStep = ", preparedLin.timePointsUsedForOneStep)
    println("A_unknown: ", _sparse_norms(preparedLin.A_unknown))
    println("L_known:   ", _sparse_norms(preparedLin.L_known))
    println("R_force:   ", _sparse_norms(preparedLin.R_force))
    if hasproperty(preparedLin, :right_operator)
        println("right_operator.size = ", preparedLin.right_operator.size)
        println("right_operator.nnz = ", length(preparedLin.right_operator.table.vals))
    end
    row_nnz_A = vec(sum(abs.(preparedLin.A_unknown) .> 0; dims=2))
    row_nnz_R = vec(sum(abs.(preparedLin.R_force) .> 0; dims=2))
    println("rows with A entries = ", count(!iszero, row_nnz_A), " / ", length(row_nnz_A))
    println("rows with R entries = ", count(!iszero, row_nnz_R), " / ", length(row_nnz_R))
    return nothing
end

function source_field_response(preparedLin, sourcePoint, timeSignal; amplitude=1.0, it=nothing)
    LI = LinearIndices(preparedLin.spaceShape)
    src_linear = LI[sourcePoint]
    last_start = length(timeSignal) - preparedLin.timePointsUsedForOneStep + 1
    last_start >= 1 || error("timeSignal is too short for timePointsUsedForOneStep=$(preparedLin.timePointsUsedForOneStep)")
    if it === nothing
        it = min(argmax(abs.(timeSignal)), last_start)
    else
        1 <= it <= last_start || error("it should be in 1:$last_start for this timeSignal")
    end

    out = NamedTuple[]
    NForceField = hasproperty(preparedLin, :NForceField) ? preparedLin.NForceField : preparedLin.NField
    for iField in 1:NForceField
        w = zeros(Float64, preparedLin.NforcePoints)
        w[src_linear] = 1.0
        sourceFull = make_sourceFull(preparedLin, w, timeSignal; iField=iField, amplitude=amplitude)
        knownForce = sourceFull[:, :, it:it+preparedLin.timePointsUsedForOneStep-1]
        knownInputs = vcat(vec(zero(preparedLin.known_lhs_template)), vec(knownForce))
        b = copy(preparedLin.b_template)
        preparedLin.b_fun!(b, knownInputs)
        push!(out, (
            iField=iField,
            time_index=it,
            source_norm=norm(knownForce),
            b_norm=norm(b),
            b_maximum=maximum(abs, b),
            b_nonzero=count(!iszero, b),
        ))
    end
    return out
end

function one_step_response(preparedLin, sourceFull; it=1, boundaryConditionForced=false)
    knownField = zero(preparedLin.known_lhs_template)
    knownForce = sourceFull[:, :, it:it+preparedLin.timePointsUsedForOneStep-1]
    knownInputs = vcat(vec(knownField), vec(knownForce))
    b = copy(preparedLin.b_template)
    preparedLin.b_fun!(b, knownInputs)

    A = sparse(preparedLin.A_template)
    if boundaryConditionForced
        apply_forced_boundary_for_video!(A, b; leftValue=0.0, rightValue=0.0)
    end
    u = A \ b
    ufield = reshape(real.(u), preparedLin.NpointsSpace, preparedLin.NField)
    return (
        b_norm=norm(b),
        b_maximum=maximum(abs, b),
        b_nonzero=count(!iszero, b),
        u_norm=norm(u),
        u_maximum=maximum(abs, u),
        u_nonzero=count(!iszero, u),
        ufield=ufield,
    )
end


function _matrix_row_abs_sums(A)
    return vec(sum(abs.(A); dims=2))
end

function operator_growth_diagnostics(preparedLin)
    A = sparse(preparedLin.A_unknown)
    L = sparse(preparedLin.L_known)
    R = sparse(preparedLin.R_force)
    diagA = abs.(diag(A))
    rowA = _matrix_row_abs_sums(A)
    offdiagA = rowA .- diagA
    rowL = _matrix_row_abs_sums(L)
    rowR = _matrix_row_abs_sums(R)
    return (
        A_size=size(A),
        A_nnz=nnz(A),
        A_diag_min=isempty(diagA) ? 0.0 : minimum(diagA),
        A_diag_median=isempty(diagA) ? 0.0 : median(diagA),
        A_diag_max=isempty(diagA) ? 0.0 : maximum(diagA),
        A_offdiag_over_diag_max=maximum(offdiagA ./ max.(diagA, eps(Float64))),
        L_row_sum_max=maximum(rowL),
        R_row_sum_max=maximum(rowR),
        R_over_A_diag_max=maximum(rowR ./ max.(diagA, eps(Float64))),
        condest_dense_sample=size(A,1) <= 5000 ? cond(Matrix(A)) : missing,
    )
end

function homogeneous_zero_test(preparedLin; Nt=20, store_every=1)
    sourceFull_zero = zeros(eltype(preparedLin.known_rhs_template), preparedLin.NforcePoints, preparedLin.NForceField, Nt + preparedLin.timePointsUsedForOneStep - 1)
    frames_zero = propagate_linear_frames(preparedLin, sourceFull_zero, Nt; store_every=store_every, blowup_limit=1e12)
    return wavefield_snapshot_report(frames_zero)
end

function source_amplitude_sweep(preparedLin, sourcePoint, timeSignal; amplitudes=(1e-12, 1e-10, 1e-8, 1e-6), iField=1, Nt=20)
    rows = NamedTuple[]
    w = point_source_weights(preparedLin, sourcePoint)
    for amp in amplitudes
        sourceFull = make_sourceFull(preparedLin, w, timeSignal; iField=iField, amplitude=amp)
        frames = propagate_linear_frames(preparedLin, sourceFull, Nt; store_every=max(1, Nt ÷ 5), blowup_limit=1e30)
        report = wavefield_snapshot_report(frames)
        push!(rows, (amplitude=amp, last=report[end], nframes=length(frames)))
    end
    return rows
end


function _distance_from_source_table(frame, sourcePoint; threshold_fraction=1e-6)
    f = abs.(Float64.(frame))
    fmax = maximum(f)
    threshold = threshold_fraction * max(fmax, eps(Float64))
    active = findall(>(threshold), f)
    if isempty(active)
        return (nactive=0, max_distance=0.0, mean_distance=0.0, fmax=fmax)
    end
    distances = [sqrt(sum((Tuple(I) .- Tuple(sourcePoint)).^2)) for I in active]
    return (
        nactive=length(active),
        max_distance=maximum(distances),
        mean_distance=mean(distances),
        fmax=fmax,
    )
end

function propagation_radius_report(frames, sourcePoint; threshold_fraction=1e-6)
    return [(frame=i, _distance_from_source_table(frame, sourcePoint; threshold_fraction=threshold_fraction)...) for (i, frame) in pairs(frames)]
end

function impulse_sourceFull(preparedLin, sourcePoint; iField=1, amplitude=1.0, Nt=20, impulse_time=1)
    timeSignal = zeros(Float64, Nt + preparedLin.timePointsUsedForOneStep - 1)
    timeSignal[impulse_time] = 1.0
    w = point_source_weights(preparedLin, sourcePoint)
    return make_sourceFull(preparedLin, w, timeSignal; iField=iField, amplitude=amplitude)
end

function impulse_propagation_test(preparedLin, sourcePoint; iField=1, amplitude=1.0, Nt=20, store_every=1, threshold_fraction=1e-8)
    sourceFull = impulse_sourceFull(preparedLin, sourcePoint; iField=iField, amplitude=amplitude, Nt=Nt)
    frames = propagate_linear_frames(preparedLin, sourceFull, Nt; store_every=store_every, blowup_limit=1e20)
    return (
        frames=frames,
        snapshots=wavefield_snapshot_report(frames),
        radius=propagation_radius_report(frames, sourcePoint; threshold_fraction=threshold_fraction),
    )
end


function frame_extrema_report(frames, sourcePoint)
    rows = NamedTuple[]
    for (i, frame) in pairs(frames)
        A = Float64.(frame)
        absA = abs.(A)
        imax = argmax(absA)
        maxpoint = CartesianIndices(size(A))[imax]
        distance = sqrt(sum((Tuple(maxpoint) .- Tuple(sourcePoint)).^2))
        push!(rows, (
            frame=i,
            maxpoint=maxpoint,
            distance_from_source=distance,
            value=A[imax],
            maxabs=absA[imax],
            minimum=minimum(A),
            maximum=maximum(A),
        ))
    end
    return rows
end


function _get_numop(numOps, which::Symbol)
    ops = numOps isa AbstractDict ? numOps["numericalOperators"] : numOps.numericalOperators
    return getproperty(ops, which)
end

function _point_linear_index(op, point)
    νWhole = op.geometry.νWhole
    if point isa Integer
        1 <= point <= length(νWhole) || error("point integer should be in 1:$(length(νWhole))")
        return Int(point), νWhole[Int(point)]
    elseif point isa CartesianIndex
        point in νWhole || error("point $point is not in op.geometry.νWhole")
        pointLinear = LinearIndices(νWhole)[point]
        return Int(pointLinear), point
    else
        error("point should be an Int linear point index or a CartesianIndex")
    end
end

function _time_role(iT, activeTimePoints)
    if iT == activeTimePoints
        return :future_unknown
    elseif iT == activeTimePoints - 1
        return :present_known
    else
        return Symbol("past_known_tminus$(activeTimePoints - iT)")
    end
end

function coefficient_rows_at_point(op, point; iExpr=1, atol=0.0)
    pointLinear, pointCI = _point_linear_index(op, point)
    nPoints = length(op.geometry.νWhole)
    activeTimePoints = op.geometry.activeTimePoints
    spaceShape = Tuple(op.geometry.wholeRegionPointsSpace)
    Nexpr = op.size[1] ÷ nPoints
    Nfield = op.size[2] ÷ (nPoints * activeTimePoints)

    1 <= iExpr <= Nexpr || error("iExpr should be in 1:$Nexpr")

    residualLI = LinearIndices((Nexpr, nPoints))
    targetRow = residualLI[iExpr, pointLinear]
    fieldTimeSpaceCI = CartesianIndices((Nfield, activeTimePoints, spaceShape...))

    rows = NamedTuple[]
    for k in eachindex(op.table.vals)
        Int(op.table.rows[k]) == targetRow || continue
        val = op.table.vals[k]
        abs(val) > atol || continue

        ci = fieldTimeSpaceCI[Int(op.table.cols[k])]
        iField = ci[1]
        iT = ci[2]
        neighbour = CartesianIndex(Tuple(ci)[3:end])
        offset = Tuple(neighbour - pointCI)
        timeRole = _time_role(iT, activeTimePoints)

        push!(rows, (
            k = k,
            residual_row = targetRow,
            expr = iExpr,
            point = pointCI,
            field = iField,
            time_slot = iT,
            time_role = timeRole,
            neighbour = neighbour,
            offset = offset,
            coef = val,
            abscoef = abs(val),
        ))
    end

    sort!(rows, by = r -> (r.time_slot, r.field, r.offset))
    return rows
end

function inspect_operator_coefficients_at_point(numOps, point; which=:residual, iExpr=1, atol=0.0)
    op = _get_numop(numOps, which)
    rows = coefficient_rows_at_point(op, point; iExpr=iExpr, atol=atol)
    println("operator = ", which)
    println("point = ", point, ", iExpr = ", iExpr, ", ncoef = ", length(rows))
    for r in rows
        println(
            "t=", r.time_slot,
            " (", r.time_role, ")",
            " field=", r.field,
            " offset=", r.offset,
            " neighbour=", r.neighbour,
            " coef=", r.coef,
        )
    end
    return rows
end

function coefficient_table_by_time(rows)
    slots = sort(unique(getproperty.(rows, :time_slot)))
    return [(
        time_slot=s,
        time_role=first(r.time_role for r in rows if r.time_slot == s),
        n=length(filter(r -> r.time_slot == s, rows)),
        sumcoef=sum(r.coef for r in rows if r.time_slot == s),
        sumabs=sum(r.abscoef for r in rows if r.time_slot == s),
    ) for s in slots]
end

function coefficient_rows_for_role(rows, role::Symbol)
    return filter(r -> r.time_role == role, rows)
end


function source_time_samples(Nt, Δt, timePointsUsedForOneStep; wavelet=nothing, t0=12Δt, f0=0.04)
    ntime = Nt + timePointsUsedForOneStep - 1
    t = (0:ntime-1) .* Δt
    if wavelet === nothing
        return Ricker.(t, t0, f0)
    else
        return wavelet.(t)
    end
end

function _normalise_source_weights!(w; normalise=:none)
    if normalise == :none || normalise === false
        return w
    elseif normalise == :sum
        s = sum(w)
        !iszero(s) && (w ./= s)
    elseif normalise == :l1
        s = sum(abs, w)
        !iszero(s) && (w ./= s)
    elseif normalise == :max
        s = maximum(abs, w)
        !iszero(s) && (w ./= s)
    else
        error("normalise should be :none, :sum, :l1, or :max")
    end
    return w
end

function make_sourceFull(preparedLin, spatialWeights, timeSignal; iField=1, amplitude=1.0)
    length(spatialWeights) == preparedLin.NforcePoints ||
        error("spatialWeights should have length preparedLin.NforcePoints=$(preparedLin.NforcePoints)")
    NForceField = hasproperty(preparedLin, :NForceField) ? preparedLin.NForceField : preparedLin.NField
    1 <= iField <= NForceField || error("iField should be between 1 and $(NForceField)")

    T = promote_type(eltype(spatialWeights), eltype(timeSignal), typeof(amplitude))
    sourceFull = zeros(T, preparedLin.NforcePoints, NForceField, length(timeSignal))
    for it in eachindex(timeSignal)
        sourceFull[:, iField, it] .= amplitude .* timeSignal[it] .* spatialWeights
    end
    return sourceFull
end

function point_source_weights(preparedLin, point::CartesianIndex; normalise=:none)
    weights = zeros(Float64, preparedLin.NforcePoints)
    LI = LinearIndices(preparedLin.spaceShape)
    weights[LI[point]] = 1.0
    return _normalise_source_weights!(weights; normalise)
end

function hyperplane_source_weights(preparedLin; fixed=Dict{Int,Int}(), profile=(I -> 1.0), normalise=:sum)
    weights = zeros(Float64, preparedLin.NforcePoints)
    LI = LinearIndices(preparedLin.spaceShape)
    for I in CartesianIndices(preparedLin.spaceShape)
        coords = Tuple(I)
        inside = all(coords[d] == v for (d, v) in fixed)
        if inside
            weights[LI[I]] = profile(I)
        end
    end
    return _normalise_source_weights!(weights; normalise)
end

function mask_source_weights(preparedLin, mask; profile=(I -> 1.0), normalise=:sum)
    weights = zeros(Float64, preparedLin.NforcePoints)
    LI = LinearIndices(preparedLin.spaceShape)

    if mask isa AbstractArray{Bool}
        size(mask) == preparedLin.spaceShape || error("Boolean mask size should be $(preparedLin.spaceShape)")
        for I in CartesianIndices(mask)
            mask[I] && (weights[LI[I]] = profile(I))
        end
    else
        for I in mask
            weights[LI[I]] = profile(I)
        end
    end

    return _normalise_source_weights!(weights; normalise)
end


using LinearAlgebra, SparseArrays
using CairoMakie
CairoMakie.activate!()

function apply_forced_boundary_for_video!(A, b; leftValue=0.0, rightValue=0.0)
    A[1, :] .= 0
    A[1, 1] = 1
    b[1] = leftValue

    A[end, :] .= 0
    A[end, end] = 1
    b[end] = rightValue
    return A, b
end

function propagate_linear_frames(
    preparedLin,
    sourceFull,
    Nt;
    initialCondition=0.0,
    boundaryConditionForced=false,
    store_every=1,
    blowup_limit=Inf,
)
    NField = preparedLin.NField
    NForceField = hasproperty(preparedLin, :NForceField) ? preparedLin.NForceField : preparedLin.NField
    NpointsSpace = preparedLin.NpointsSpace
    timePointsUsedForOneStep = preparedLin.timePointsUsedForOneStep
    NknownTime = max(timePointsUsedForOneStep - 1, 0)

    size(sourceFull, 1) == preparedLin.NforcePoints || error("sourceFull first dimension mismatch")
    size(sourceFull, 2) == NForceField || error("sourceFull second dimension mismatch")
    size(sourceFull, 3) >= Nt + timePointsUsedForOneStep - 1 || error("sourceFull has too few time samples")

    knownField = fill(Float64(initialCondition), size(preparedLin.known_lhs_template))
    knownForce = fill(Float64(initialCondition), size(preparedLin.known_rhs_template))
    unknownField = fill(Float64(initialCondition), NpointsSpace, NField)

    A = sparse(preparedLin.A_template)
    b0 = copy(preparedLin.b_template)
    if boundaryConditionForced
        apply_forced_boundary_for_video!(A, b0; leftValue=0.0, rightValue=0.0)
    end
    factor = lu(A)
    b = copy(preparedLin.b_template)

    frames = Vector{Array{Float64}}()
    for it in 1:Nt
        knownForce .= sourceFull[:, :, it:it+timePointsUsedForOneStep-1]
        knownInputs = vcat(vec(knownField), vec(knownForce))
        preparedLin.b_fun!(b, knownInputs)
        if boundaryConditionForced
            b[1] = 0.0
            b[end] = 0.0
        end

        u = factor \ b
        unknownField .= reshape(real.(u), NpointsSpace, NField)

        umax = maximum(abs, unknownField)
        if !isfinite(umax) || umax > blowup_limit
            @warn "Stopping propagation because the wavefield blew up" it umax blowup_limit
            push!(frames, reshape(copy(unknownField[:, 1]), preparedLin.spaceShape...))
            break
        end

        if it == 1 || it % store_every == 0 || it == Nt
            push!(frames, reshape(copy(unknownField[:, 1]), preparedLin.spaceShape...))
        end

        if NknownTime > 0
            if NknownTime > 1
                knownField[:, :, 1:end-1] .= knownField[:, :, 2:end]
            end
            knownField[:, :, end] .= unknownField
        end
    end
    return frames
end

function _finite_copy(A; fillvalue=0.0)
    B = Array{Float64}(undef, size(A))
    replaced = 0
    for I in eachindex(A)
        v = Float64(real(A[I]))
        if isfinite(v)
            B[I] = v
        else
            B[I] = fillvalue
            replaced += 1
        end
    end
    return B, replaced
end

function finite_frame_diagnostics(frames)
    total_bad = 0
    first_bad = nothing
    finite_max = 0.0
    for (i, frame) in pairs(frames)
        bad_here = count(!isfinite, frame)
        if bad_here > 0 && first_bad === nothing
            first_bad = i
        end
        total_bad += bad_here
        for v in frame
            if isfinite(v)
                finite_max = max(finite_max, abs(Float64(v)))
            end
        end
    end
    return (total_bad=total_bad, first_bad=first_bad, finite_max=finite_max)
end

function _background_for_plot(background, spaceShape)
    background === nothing && return nothing
    bg = Array(background)
    if size(bg) == spaceShape
        clean_bg, replaced = _finite_copy(bg)
    elseif length(bg) == prod(spaceShape)
        clean_bg, replaced = _finite_copy(reshape(bg, spaceShape...))
    else
        @warn "background size $(size(bg)) does not match wavefield spaceShape $spaceShape; ignoring background"
        return nothing
    end
    replaced > 0 && @warn "background had $replaced non-finite values; replaced by 0 for plotting"
    return clean_bg
end

function record_wave_video(
    frames;
    videoFile=joinpath(@__DIR__, "point_source_propagation.mp4"),
    background=nothing,
    sourcePoint=nothing,
    framerate=20,
    title="point source propagation",
    colormap=:balance,
    background_colormap=:grays,
    wave_alpha=0.72,
    clim=nothing,
)
    isempty(frames) && error("frames is empty")
    spaceShape = size(frames[1])
    bg = _background_for_plot(background, spaceShape)
    nd = length(spaceShape)

    diag = finite_frame_diagnostics(frames)
    if diag.total_bad > 0
        @warn "wavefield frames contain $(diag.total_bad) non-finite values; first affected stored frame is $(diag.first_bad). Replacing by 0 for plotting."
    end
    clean_frames = [first(_finite_copy(f)) for f in frames]

    if clim === nothing
        vmax = diag.finite_max
        clim = vmax == 0 ? 1.0 : vmax
    end

    fig = Figure(size=(900, 680))
    ax = Axis(fig[1, 1], aspect=DataAspect(), title=title)

    if nd == 1
        x = 1:spaceShape[1]
        yobs = Observable(vec(clean_frames[1]))
        lines!(ax, x, yobs, color=:dodgerblue3, linewidth=2)
        ylims!(ax, -clim, clim)
        if sourcePoint !== nothing
            vlines!(ax, [Tuple(sourcePoint)[1]], color=:red, linewidth=2)
        end
        record(fig, videoFile, eachindex(frames); framerate=framerate) do iframe
            yobs[] = vec(clean_frames[iframe])
            ax.title = "$title  |  frame $iframe/$(length(frames))"
        end
    elseif nd == 2
        if bg !== nothing
            heatmap!(ax, bg; colormap=background_colormap)
        end
        frame_obs = Observable(clean_frames[1])
        hm = heatmap!(ax, frame_obs; colormap=colormap, colorrange=(-clim, clim), alpha=bg === nothing ? 1.0 : wave_alpha)
        Colorbar(fig[1, 2], hm, label="wavefield")
        if sourcePoint !== nothing
            xy = Tuple(sourcePoint)
            scatter!(ax, [xy[1]], [xy[2]], color=:red, markersize=14)
        end
        record(fig, videoFile, eachindex(frames); framerate=framerate) do iframe
            frame_obs[] = clean_frames[iframe]
            ax.title = "$title  |  frame $iframe/$(length(frames))"
        end
    else
        error("Video helper currently supports 1D/2D frames. For 3D/4D, select a slice before recording.")
    end

    return videoFile
end


function _finite_plot_matrix(A; fillvalue=0.0)
    B = Array{Float64}(undef, size(A))
    nbad = 0
    finite_max = 0.0
    for I in eachindex(A)
        v = Float64(real(A[I]))
        if isfinite(v)
            B[I] = v
            finite_max = max(finite_max, abs(v))
        else
            B[I] = fillvalue
            nbad += 1
        end
    end
    return B, nbad, finite_max
end

function wavefield_snapshot_report(frames)
    rows = NamedTuple[]
    for (i, frame) in pairs(frames)
        _, nbad, finite_max = _finite_plot_matrix(frame)
        push!(rows, (frame=i, nbad=nbad, finite_max=finite_max, minimum=minimum(skipmissing(vec(ifelse.(isfinite.(frame), frame, missing)))), maximum=maximum(skipmissing(vec(ifelse.(isfinite.(frame), frame, missing))))))
    end
    return rows
end


function _snapshot_background_for_plot(background, spaceShape)
    background === nothing && return nothing
    bg = Array(background)
    if size(bg) == spaceShape
        B, replaced, _ = _finite_plot_matrix(bg)
    elseif length(bg) == prod(spaceShape)
        B, replaced, _ = _finite_plot_matrix(reshape(bg, spaceShape...))
    else
        @warn "background size $(size(bg)) does not match wavefield spaceShape $spaceShape; ignoring background"
        return nothing
    end
    replaced > 0 && @warn "background had $replaced non-finite values; replaced by 0 for plotting"
    return B
end

function plot_wave_snapshots(
    frames;
    indices=nothing,
    background=nothing,
    sourcePoint=nothing,
    clim=nothing,
    ncols=3,
    title="wavefield snapshots",
    colormap=:balance,
    background_colormap=:grays,
    wave_alpha=0.72,
)
    isempty(frames) && error("frames is empty")
    if indices === nothing
        indices = unique(round.(Int, range(1, length(frames), length=min(9, length(frames)))))
    end
    indices = collect(indices)

    clean_frames = Array{Float64}[]
    nbad = Int[]
    finite_maxima = Float64[]
    for i in indices
        B, bad, fmax = _finite_plot_matrix(frames[i])
        push!(clean_frames, B)
        push!(nbad, bad)
        push!(finite_maxima, fmax)
    end

    if clim === nothing
        vmax = maximum(finite_maxima)
        clim = vmax == 0 ? 1.0 : vmax
    end

    bg = _snapshot_background_for_plot(background, size(frames[1]))
    n = length(indices)
    nrows = cld(n, ncols)
    fig = Figure(size=(320*ncols, 300*nrows))

    for (k, iframe) in enumerate(indices)
        r = cld(k, ncols)
        c = k - (r - 1) * ncols
        ax = Axis(fig[r, c], aspect=DataAspect(), title="frame $iframe | bad=$(nbad[k])")
        if length(size(frames[1])) == 1
            lines!(ax, vec(clean_frames[k]), color=:dodgerblue3, linewidth=2)
            ylims!(ax, -clim, clim)
            if sourcePoint !== nothing
                vlines!(ax, [Tuple(sourcePoint)[1]], color=:red, linewidth=2)
            end
        elseif length(size(frames[1])) == 2
            if bg !== nothing
                heatmap!(ax, bg; colormap=background_colormap)
            end
            heatmap!(ax, clean_frames[k]; colormap=colormap, colorrange=(-clim, clim), alpha=bg === nothing ? 1.0 : wave_alpha)
            if sourcePoint !== nothing
                xy = Tuple(sourcePoint)
                scatter!(ax, [xy[1]], [xy[2]], color=:red, markersize=10)
            end
        else
            error("Snapshot helper supports 1D/2D frames. Select a slice first for higher dimensions.")
        end
    end

    Label(fig[0, :], title)
    return fig
end


function save_wave_snapshot_pdf(
    frames;
    filename=joinpath(@__DIR__, "wave_snapshots.pdf"),
    indices=nothing,
    background=nothing,
    sourcePoint=nothing,
    clim=nothing,
    ncols=3,
    title="wavefield snapshots",
    kwargs...,
)
    fig = plot_wave_snapshots(
        frames;
        indices=indices,
        background=background,
        sourcePoint=sourcePoint,
        clim=clim,
        ncols=ncols,
        title=title,
        kwargs...,
    )
    save(filename, fig)
    empty!(fig.scene)  # release most displayed scene resources
    GC.gc()
    return filename
end


using Statistics

function cfl_diagnostics(model_velocity, Δnum; cfl_safety=0.45, ppw=10, samples_per_period=20)
    v = vec(Float64.(model_velocity))
    finite_v = filter(isfinite, v)
    dt = Float64(Δnum[end])
    dx = Float64.(Δnum[1:end-1])
    dxmin = minimum(dx)
    dxmax = maximum(dx)
    vmax = maximum(abs, finite_v)
    vmin = minimum(abs, filter(!iszero, finite_v))
    cfl = vmax * dt / dxmin

    # MKS units: velocity [m/s], dx [m], dt [s], frequency [Hz].
    # For a Ricker wavelet in flexOPT, f0 is the dominant/peak frequency.
    fmax_spatial = vmin / (ppw * dxmax)
    fmax_temporal = 1 / (samples_per_period * dt)
    suggested_ricker_f0 = min(fmax_spatial, fmax_temporal)

    return (
        v_min=minimum(finite_v),
        v_median=median(finite_v),
        v_max=maximum(finite_v),
        dt=dt,
        dxmin=dxmin,
        dxmax=dxmax,
        cfl_max=cfl,
        suggested_dt_2D=cfl_safety * dxmin / vmax,
        ppw=ppw,
        samples_per_period=samples_per_period,
        fmax_spatial=fmax_spatial,
        fmax_temporal=fmax_temporal,
        suggested_ricker_f0=suggested_ricker_f0,
        suggested_ricker_t0=1.5 / suggested_ricker_f0,
    )
end

function ricker_time_signal(Nt, Δt; f0, t0=1.5/f0, timePointsUsedForOneStep=1)
    ntime = Nt + timePointsUsedForOneStep - 1
    t = (0:ntime-1) .* Δt
    return t, Ricker.(t, t0, f0)
end

function dominant_frequency_scan(signal, Δt; nfreq=512)
    y = Float64.(signal) .- mean(Float64.(signal))
    n = length(y)
    duration = n * Δt
    fmin = 1 / duration
    fmax = 0.5 / Δt
    freqs = range(fmin, fmax; length=nfreq)
    t = (0:n-1) .* Δt
    spectrum = [abs(sum(y .* exp.(-2im * π * f .* t))) for f in freqs]
    imax = argmax(spectrum)
    return (f_peak=freqs[imax], peak_amplitude=spectrum[imax], frequencies=freqs, spectrum=spectrum)
end

function ricker_diagnostics(model_velocity, Δnum; f0=nothing, Nt=200, timePointsUsedForOneStep=1, ppw=10, samples_per_period=20, t0=nothing)
    cfl = cfl_diagnostics(model_velocity, Δnum; ppw=ppw, samples_per_period=samples_per_period)
    f0_used = f0 === nothing ? cfl.suggested_ricker_f0 : Float64(f0)
    t0_used = t0 === nothing ? 1.5 / f0_used : Float64(t0)
    t, signal = ricker_time_signal(Nt, cfl.dt; f0=f0_used, t0=t0_used, timePointsUsedForOneStep=timePointsUsedForOneStep)
    freqinfo = dominant_frequency_scan(signal, cfl.dt)
    return (
        f0=f0_used,
        t0=t0_used,
        duration=t[end],
        dominant_frequency_scan=freqinfo.f_peak,
        wavelength_min_velocity=cfl.v_min / f0_used,
        points_per_wavelength_min_velocity=(cfl.v_min / f0_used) / cfl.dxmax,
        samples_per_period=1 / (f0_used * cfl.dt),
        max_abs=maximum(abs, signal),
        first_peak_time=t[argmax(abs.(signal))],
        cfl=cfl,
    )
end

function frame_motion_report(frames)
    rows = NamedTuple[]
    previous = nothing
    for (i, frame) in pairs(frames)
        _, nbad, fmax = _finite_plot_matrix(frame)
        dnorm = previous === nothing ? 0.0 : norm(frame .- previous)
        dmax = previous === nothing ? 0.0 : maximum(abs, frame .- previous)
        push!(rows, (frame=i, nbad=nbad, maxabs=fmax, norm=norm(frame), diff_norm=dnorm, diff_max=dmax))
        previous = frame
    end
    return rows
end

function difference_frames(frames)
    length(frames) >= 2 || error("need at least two frames")
    return [frames[i] .- frames[i-1] for i in 2:length(frames)]
end


function _sparse_norms(A)
    return (
        size=size(A),
        nnz=nnz(A),
        maxabs=nnz(A) == 0 ? 0.0 : maximum(abs, nonzeros(A)),
        norm=norm(A),
    )
end

function diagnose_prepared_linear_system(preparedLin)
    println("spaceShape = ", preparedLin.spaceShape)
    println("NpointsSpace = ", preparedLin.NpointsSpace)
    println("NField = ", preparedLin.NField)
    println("NForceField = ", hasproperty(preparedLin, :NForceField) ? preparedLin.NForceField : preparedLin.NField)
    println("timePointsUsedForOneStep = ", preparedLin.timePointsUsedForOneStep)
    println("A_unknown: ", _sparse_norms(preparedLin.A_unknown))
    println("L_known:   ", _sparse_norms(preparedLin.L_known))
    println("R_force:   ", _sparse_norms(preparedLin.R_force))
    if hasproperty(preparedLin, :right_operator)
        println("right_operator.size = ", preparedLin.right_operator.size)
        println("right_operator.nnz = ", length(preparedLin.right_operator.table.vals))
    end
    row_nnz_A = vec(sum(abs.(preparedLin.A_unknown) .> 0; dims=2))
    row_nnz_R = vec(sum(abs.(preparedLin.R_force) .> 0; dims=2))
    println("rows with A entries = ", count(!iszero, row_nnz_A), " / ", length(row_nnz_A))
    println("rows with R entries = ", count(!iszero, row_nnz_R), " / ", length(row_nnz_R))
    return nothing
end

function source_field_response(preparedLin, sourcePoint, timeSignal; amplitude=1.0, it=nothing)
    LI = LinearIndices(preparedLin.spaceShape)
    src_linear = LI[sourcePoint]
    last_start = length(timeSignal) - preparedLin.timePointsUsedForOneStep + 1
    last_start >= 1 || error("timeSignal is too short for timePointsUsedForOneStep=$(preparedLin.timePointsUsedForOneStep)")
    if it === nothing
        it = min(argmax(abs.(timeSignal)), last_start)
    else
        1 <= it <= last_start || error("it should be in 1:$last_start for this timeSignal")
    end

    out = NamedTuple[]
    NForceField = hasproperty(preparedLin, :NForceField) ? preparedLin.NForceField : preparedLin.NField
    for iField in 1:NForceField
        w = zeros(Float64, preparedLin.NforcePoints)
        w[src_linear] = 1.0
        sourceFull = make_sourceFull(preparedLin, w, timeSignal; iField=iField, amplitude=amplitude)
        knownForce = sourceFull[:, :, it:it+preparedLin.timePointsUsedForOneStep-1]
        knownInputs = vcat(vec(zero(preparedLin.known_lhs_template)), vec(knownForce))
        b = copy(preparedLin.b_template)
        preparedLin.b_fun!(b, knownInputs)
        push!(out, (
            iField=iField,
            time_index=it,
            source_norm=norm(knownForce),
            b_norm=norm(b),
            b_maximum=maximum(abs, b),
            b_nonzero=count(!iszero, b),
        ))
    end
    return out
end

function one_step_response(preparedLin, sourceFull; it=1, boundaryConditionForced=false)
    knownField = zero(preparedLin.known_lhs_template)
    knownForce = sourceFull[:, :, it:it+preparedLin.timePointsUsedForOneStep-1]
    knownInputs = vcat(vec(knownField), vec(knownForce))
    b = copy(preparedLin.b_template)
    preparedLin.b_fun!(b, knownInputs)

    A = sparse(preparedLin.A_template)
    if boundaryConditionForced
        apply_forced_boundary_for_video!(A, b; leftValue=0.0, rightValue=0.0)
    end
    u = A \ b
    ufield = reshape(real.(u), preparedLin.NpointsSpace, preparedLin.NField)
    return (
        b_norm=norm(b),
        b_maximum=maximum(abs, b),
        b_nonzero=count(!iszero, b),
        u_norm=norm(u),
        u_maximum=maximum(abs, u),
        u_nonzero=count(!iszero, u),
        ufield=ufield,
    )
end


function _matrix_row_abs_sums(A)
    return vec(sum(abs.(A); dims=2))
end

function operator_growth_diagnostics(preparedLin)
    A = sparse(preparedLin.A_unknown)
    L = sparse(preparedLin.L_known)
    R = sparse(preparedLin.R_force)
    diagA = abs.(diag(A))
    rowA = _matrix_row_abs_sums(A)
    offdiagA = rowA .- diagA
    rowL = _matrix_row_abs_sums(L)
    rowR = _matrix_row_abs_sums(R)
    return (
        A_size=size(A),
        A_nnz=nnz(A),
        A_diag_min=isempty(diagA) ? 0.0 : minimum(diagA),
        A_diag_median=isempty(diagA) ? 0.0 : median(diagA),
        A_diag_max=isempty(diagA) ? 0.0 : maximum(diagA),
        A_offdiag_over_diag_max=maximum(offdiagA ./ max.(diagA, eps(Float64))),
        L_row_sum_max=maximum(rowL),
        R_row_sum_max=maximum(rowR),
        R_over_A_diag_max=maximum(rowR ./ max.(diagA, eps(Float64))),
        condest_dense_sample=size(A,1) <= 5000 ? cond(Matrix(A)) : missing,
    )
end

function homogeneous_zero_test(preparedLin; Nt=20, store_every=1)
    sourceFull_zero = zeros(eltype(preparedLin.known_rhs_template), preparedLin.NforcePoints, preparedLin.NForceField, Nt + preparedLin.timePointsUsedForOneStep - 1)
    frames_zero = propagate_linear_frames(preparedLin, sourceFull_zero, Nt; store_every=store_every, blowup_limit=1e12)
    return wavefield_snapshot_report(frames_zero)
end

function source_amplitude_sweep(preparedLin, sourcePoint, timeSignal; amplitudes=(1e-12, 1e-10, 1e-8, 1e-6), iField=1, Nt=20)
    rows = NamedTuple[]
    w = point_source_weights(preparedLin, sourcePoint)
    for amp in amplitudes
        sourceFull = make_sourceFull(preparedLin, w, timeSignal; iField=iField, amplitude=amp)
        frames = propagate_linear_frames(preparedLin, sourceFull, Nt; store_every=max(1, Nt ÷ 5), blowup_limit=1e30)
        report = wavefield_snapshot_report(frames)
        push!(rows, (amplitude=amp, last=report[end], nframes=length(frames)))
    end
    return rows
end


function _distance_from_source_table(frame, sourcePoint; threshold_fraction=1e-6)
    f = abs.(Float64.(frame))
    fmax = maximum(f)
    threshold = threshold_fraction * max(fmax, eps(Float64))
    active = findall(>(threshold), f)
    if isempty(active)
        return (nactive=0, max_distance=0.0, mean_distance=0.0, fmax=fmax)
    end
    distances = [sqrt(sum((Tuple(I) .- Tuple(sourcePoint)).^2)) for I in active]
    return (
        nactive=length(active),
        max_distance=maximum(distances),
        mean_distance=mean(distances),
        fmax=fmax,
    )
end

function propagation_radius_report(frames, sourcePoint; threshold_fraction=1e-6)
    return [(frame=i, _distance_from_source_table(frame, sourcePoint; threshold_fraction=threshold_fraction)...) for (i, frame) in pairs(frames)]
end

function impulse_sourceFull(preparedLin, sourcePoint; iField=1, amplitude=1.0, Nt=20, impulse_time=1)
    timeSignal = zeros(Float64, Nt + preparedLin.timePointsUsedForOneStep - 1)
    timeSignal[impulse_time] = 1.0
    w = point_source_weights(preparedLin, sourcePoint)
    return make_sourceFull(preparedLin, w, timeSignal; iField=iField, amplitude=amplitude)
end

function impulse_propagation_test(preparedLin, sourcePoint; iField=1, amplitude=1.0, Nt=20, store_every=1, threshold_fraction=1e-8)
    sourceFull = impulse_sourceFull(preparedLin, sourcePoint; iField=iField, amplitude=amplitude, Nt=Nt)
    frames = propagate_linear_frames(preparedLin, sourceFull, Nt; store_every=store_every, blowup_limit=1e20)
    return (
        frames=frames,
        snapshots=wavefield_snapshot_report(frames),
        radius=propagation_radius_report(frames, sourcePoint; threshold_fraction=threshold_fraction),
    )
end


function frame_extrema_report(frames, sourcePoint)
    rows = NamedTuple[]
    for (i, frame) in pairs(frames)
        A = Float64.(frame)
        absA = abs.(A)
        imax = argmax(absA)
        maxpoint = CartesianIndices(size(A))[imax]
        distance = sqrt(sum((Tuple(maxpoint) .- Tuple(sourcePoint)).^2))
        push!(rows, (
            frame=i,
            maxpoint=maxpoint,
            distance_from_source=distance,
            value=A[imax],
            maxabs=absA[imax],
            minimum=minimum(A),
            maximum=maximum(A),
        ))
    end
    return rows
end


function _get_numop(numOps, which::Symbol)
    ops = numOps isa AbstractDict ? numOps["numericalOperators"] : numOps.numericalOperators
    return getproperty(ops, which)
end

function _point_linear_index(op, point)
    νWhole = op.geometry.νWhole
    if point isa Integer
        1 <= point <= length(νWhole) || error("point integer should be in 1:$(length(νWhole))")
        return Int(point), νWhole[Int(point)]
    elseif point isa CartesianIndex
        point in νWhole || error("point $point is not in op.geometry.νWhole")
        pointLinear = LinearIndices(νWhole)[point]
        return Int(pointLinear), point
    else
        error("point should be an Int linear point index or a CartesianIndex")
    end
end

function _time_role(iT, activeTimePoints)
    if iT == activeTimePoints
        return :future_unknown
    elseif iT == activeTimePoints - 1
        return :present_known
    else
        return Symbol("past_known_tminus$(activeTimePoints - iT)")
    end
end

function coefficient_rows_at_point(op, point; iExpr=1, atol=0.0)
    pointLinear, pointCI = _point_linear_index(op, point)
    nPoints = length(op.geometry.νWhole)
    activeTimePoints = op.geometry.activeTimePoints
    spaceShape = Tuple(op.geometry.wholeRegionPointsSpace)
    Nexpr = op.size[1] ÷ nPoints
    Nfield = op.size[2] ÷ (nPoints * activeTimePoints)

    1 <= iExpr <= Nexpr || error("iExpr should be in 1:$Nexpr")

    residualLI = LinearIndices((Nexpr, nPoints))
    targetRow = residualLI[iExpr, pointLinear]
    fieldTimeSpaceCI = CartesianIndices((Nfield, activeTimePoints, spaceShape...))

    rows = NamedTuple[]
    for k in eachindex(op.table.vals)
        Int(op.table.rows[k]) == targetRow || continue
        val = op.table.vals[k]
        abs(val) > atol || continue

        ci = fieldTimeSpaceCI[Int(op.table.cols[k])]
        iField = ci[1]
        iT = ci[2]
        neighbour = CartesianIndex(Tuple(ci)[3:end])
        offset = Tuple(neighbour - pointCI)
        timeRole = _time_role(iT, activeTimePoints)

        push!(rows, (
            k = k,
            residual_row = targetRow,
            expr = iExpr,
            point = pointCI,
            field = iField,
            time_slot = iT,
            time_role = timeRole,
            neighbour = neighbour,
            offset = offset,
            coef = val,
            abscoef = abs(val),
        ))
    end

    sort!(rows, by = r -> (r.time_slot, r.field, r.offset))
    return rows
end

function inspect_operator_coefficients_at_point(numOps, point; which=:residual, iExpr=1, atol=0.0)
    op = _get_numop(numOps, which)
    rows = coefficient_rows_at_point(op, point; iExpr=iExpr, atol=atol)
    println("operator = ", which)
    println("point = ", point, ", iExpr = ", iExpr, ", ncoef = ", length(rows))
    for r in rows
        println(
            "t=", r.time_slot,
            " (", r.time_role, ")",
            " field=", r.field,
            " offset=", r.offset,
            " neighbour=", r.neighbour,
            " coef=", r.coef,
        )
    end
    return rows
end

function coefficient_table_by_time(rows)
    slots = sort(unique(getproperty.(rows, :time_slot)))
    return [(
        time_slot=s,
        time_role=first(r.time_role for r in rows if r.time_slot == s),
        n=length(filter(r -> r.time_slot == s, rows)),
        sumcoef=sum(r.coef for r in rows if r.time_slot == s),
        sumabs=sum(r.abscoef for r in rows if r.time_slot == s),
    ) for s in slots]
end

function coefficient_rows_for_role(rows, role::Symbol)
    return filter(r -> r.time_role == role, rows)
end


function source_time_samples(Nt, Δt, timePointsUsedForOneStep; wavelet=nothing, t0=12Δt, f0=0.04)
    ntime = Nt + timePointsUsedForOneStep - 1
    t = (0:ntime-1) .* Δt
    if wavelet === nothing
        return Ricker.(t, t0, f0)
    else
        return wavelet.(t)
    end
end

function _normalise_source_weights!(w; normalise=:none)
    if normalise == :none || normalise === false
        return w
    elseif normalise == :sum
        s = sum(w)
        !iszero(s) && (w ./= s)
    elseif normalise == :l1
        s = sum(abs, w)
        !iszero(s) && (w ./= s)
    elseif normalise == :max
        s = maximum(abs, w)
        !iszero(s) && (w ./= s)
    else
        error("normalise should be :none, :sum, :l1, or :max")
    end
    return w
end

function make_sourceFull(preparedLin, spatialWeights, timeSignal; iField=1, amplitude=1.0)
    length(spatialWeights) == preparedLin.NforcePoints ||
        error("spatialWeights should have length preparedLin.NforcePoints=$(preparedLin.NforcePoints)")
    NForceField = hasproperty(preparedLin, :NForceField) ? preparedLin.NForceField : preparedLin.NField
    1 <= iField <= NForceField || error("iField should be between 1 and $(NForceField)")

    T = promote_type(eltype(spatialWeights), eltype(timeSignal), typeof(amplitude))
    sourceFull = zeros(T, preparedLin.NforcePoints, NForceField, length(timeSignal))
    for it in eachindex(timeSignal)
        sourceFull[:, iField, it] .= amplitude .* timeSignal[it] .* spatialWeights
    end
    return sourceFull
end

function point_source_weights(preparedLin, point::CartesianIndex; normalise=:none)
    weights = zeros(Float64, preparedLin.NforcePoints)
    LI = LinearIndices(preparedLin.spaceShape)
    weights[LI[point]] = 1.0
    return _normalise_source_weights!(weights; normalise)
end

function hyperplane_source_weights(preparedLin; fixed=Dict{Int,Int}(), profile=(I -> 1.0), normalise=:sum)
    weights = zeros(Float64, preparedLin.NforcePoints)
    LI = LinearIndices(preparedLin.spaceShape)
    for I in CartesianIndices(preparedLin.spaceShape)
        coords = Tuple(I)
        inside = all(coords[d] == v for (d, v) in fixed)
        if inside
            weights[LI[I]] = profile(I)
        end
    end
    return _normalise_source_weights!(weights; normalise)
end

function mask_source_weights(preparedLin, mask; profile=(I -> 1.0), normalise=:sum)
    weights = zeros(Float64, preparedLin.NforcePoints)
    LI = LinearIndices(preparedLin.spaceShape)

    if mask isa AbstractArray{Bool}
        size(mask) == preparedLin.spaceShape || error("Boolean mask size should be $(preparedLin.spaceShape)")
        for I in CartesianIndices(mask)
            mask[I] && (weights[LI[I]] = profile(I))
        end
    else
        for I in mask
            weights[LI[I]] = profile(I)
        end
    end

    return _normalise_source_weights!(weights; normalise)
end


using LinearAlgebra, SparseArrays
using CairoMakie
CairoMakie.activate!()

function apply_forced_boundary_for_video!(A, b; leftValue=0.0, rightValue=0.0)
    A[1, :] .= 0
    A[1, 1] = 1
    b[1] = leftValue

    A[end, :] .= 0
    A[end, end] = 1
    b[end] = rightValue
    return A, b
end

function propagate_linear_frames(
    preparedLin,
    sourceFull,
    Nt;
    initialCondition=0.0,
    boundaryConditionForced=false,
    store_every=1,
    blowup_limit=Inf,
)
    NField = preparedLin.NField
    NForceField = hasproperty(preparedLin, :NForceField) ? preparedLin.NForceField : preparedLin.NField
    NpointsSpace = preparedLin.NpointsSpace
    timePointsUsedForOneStep = preparedLin.timePointsUsedForOneStep
    NknownTime = max(timePointsUsedForOneStep - 1, 0)

    size(sourceFull, 1) == preparedLin.NforcePoints || error("sourceFull first dimension mismatch")
    size(sourceFull, 2) == NForceField || error("sourceFull second dimension mismatch")
    size(sourceFull, 3) >= Nt + timePointsUsedForOneStep - 1 || error("sourceFull has too few time samples")

    knownField = fill(Float64(initialCondition), size(preparedLin.known_lhs_template))
    knownForce = fill(Float64(initialCondition), size(preparedLin.known_rhs_template))
    unknownField = fill(Float64(initialCondition), NpointsSpace, NField)

    A = sparse(preparedLin.A_template)
    b0 = copy(preparedLin.b_template)
    if boundaryConditionForced
        apply_forced_boundary_for_video!(A, b0; leftValue=0.0, rightValue=0.0)
    end
    factor = lu(A)
    b = copy(preparedLin.b_template)

    frames = Vector{Array{Float64}}()
    for it in 1:Nt
        knownForce .= sourceFull[:, :, it:it+timePointsUsedForOneStep-1]
        knownInputs = vcat(vec(knownField), vec(knownForce))
        preparedLin.b_fun!(b, knownInputs)
        if boundaryConditionForced
            b[1] = 0.0
            b[end] = 0.0
        end

        u = factor \ b
        unknownField .= reshape(real.(u), NpointsSpace, NField)

        umax = maximum(abs, unknownField)
        if !isfinite(umax) || umax > blowup_limit
            @warn "Stopping propagation because the wavefield blew up" it umax blowup_limit
            push!(frames, reshape(copy(unknownField[:, 1]), preparedLin.spaceShape...))
            break
        end

        if it == 1 || it % store_every == 0 || it == Nt
            push!(frames, reshape(copy(unknownField[:, 1]), preparedLin.spaceShape...))
        end

        if NknownTime > 0
            if NknownTime > 1
                knownField[:, :, 1:end-1] .= knownField[:, :, 2:end]
            end
            knownField[:, :, end] .= unknownField
        end
    end
    return frames
end

function _finite_copy(A; fillvalue=0.0)
    B = Array{Float64}(undef, size(A))
    replaced = 0
    for I in eachindex(A)
        v = Float64(real(A[I]))
        if isfinite(v)
            B[I] = v
        else
            B[I] = fillvalue
            replaced += 1
        end
    end
    return B, replaced
end

function finite_frame_diagnostics(frames)
    total_bad = 0
    first_bad = nothing
    finite_max = 0.0
    for (i, frame) in pairs(frames)
        bad_here = count(!isfinite, frame)
        if bad_here > 0 && first_bad === nothing
            first_bad = i
        end
        total_bad += bad_here
        for v in frame
            if isfinite(v)
                finite_max = max(finite_max, abs(Float64(v)))
            end
        end
    end
    return (total_bad=total_bad, first_bad=first_bad, finite_max=finite_max)
end

function _background_for_plot(background, spaceShape)
    background === nothing && return nothing
    bg = Array(background)
    if size(bg) == spaceShape
        clean_bg, replaced = _finite_copy(bg)
    elseif length(bg) == prod(spaceShape)
        clean_bg, replaced = _finite_copy(reshape(bg, spaceShape...))
    else
        @warn "background size $(size(bg)) does not match wavefield spaceShape $spaceShape; ignoring background"
        return nothing
    end
    replaced > 0 && @warn "background had $replaced non-finite values; replaced by 0 for plotting"
    return clean_bg
end

function record_wave_video(
    frames;
    videoFile=joinpath(@__DIR__, "point_source_propagation.mp4"),
    background=nothing,
    sourcePoint=nothing,
    framerate=20,
    title="point source propagation",
    colormap=:balance,
    background_colormap=:grays,
    wave_alpha=0.72,
    clim=nothing,
)
    isempty(frames) && error("frames is empty")
    spaceShape = size(frames[1])
    bg = _background_for_plot(background, spaceShape)
    nd = length(spaceShape)

    diag = finite_frame_diagnostics(frames)
    if diag.total_bad > 0
        @warn "wavefield frames contain $(diag.total_bad) non-finite values; first affected stored frame is $(diag.first_bad). Replacing by 0 for plotting."
    end
    clean_frames = [first(_finite_copy(f)) for f in frames]

    if clim === nothing
        vmax = diag.finite_max
        clim = vmax == 0 ? 1.0 : vmax
    end

    fig = Figure(size=(900, 680))
    ax = Axis(fig[1, 1], aspect=DataAspect(), title=title)

    if nd == 1
        x = 1:spaceShape[1]
        yobs = Observable(vec(clean_frames[1]))
        lines!(ax, x, yobs, color=:dodgerblue3, linewidth=2)
        ylims!(ax, -clim, clim)
        if sourcePoint !== nothing
            vlines!(ax, [Tuple(sourcePoint)[1]], color=:red, linewidth=2)
        end
        record(fig, videoFile, eachindex(frames); framerate=framerate) do iframe
            yobs[] = vec(clean_frames[iframe])
            ax.title = "$title  |  frame $iframe/$(length(frames))"
        end
    elseif nd == 2
        if bg !== nothing
            heatmap!(ax, bg; colormap=background_colormap)
        end
        frame_obs = Observable(clean_frames[1])
        hm = heatmap!(ax, frame_obs; colormap=colormap, colorrange=(-clim, clim), alpha=bg === nothing ? 1.0 : wave_alpha)
        Colorbar(fig[1, 2], hm, label="wavefield")
        if sourcePoint !== nothing
            xy = Tuple(sourcePoint)
            scatter!(ax, [xy[1]], [xy[2]], color=:red, markersize=14)
        end
        record(fig, videoFile, eachindex(frames); framerate=framerate) do iframe
            frame_obs[] = clean_frames[iframe]
            ax.title = "$title  |  frame $iframe/$(length(frames))"
        end
    else
        error("Video helper currently supports 1D/2D frames. For 3D/4D, select a slice before recording.")
    end

    return videoFile
end


function _finite_plot_matrix(A; fillvalue=0.0)
    B = Array{Float64}(undef, size(A))
    nbad = 0
    finite_max = 0.0
    for I in eachindex(A)
        v = Float64(real(A[I]))
        if isfinite(v)
            B[I] = v
            finite_max = max(finite_max, abs(v))
        else
            B[I] = fillvalue
            nbad += 1
        end
    end
    return B, nbad, finite_max
end

function wavefield_snapshot_report(frames)
    rows = NamedTuple[]
    for (i, frame) in pairs(frames)
        _, nbad, finite_max = _finite_plot_matrix(frame)
        push!(rows, (frame=i, nbad=nbad, finite_max=finite_max, minimum=minimum(skipmissing(vec(ifelse.(isfinite.(frame), frame, missing)))), maximum=maximum(skipmissing(vec(ifelse.(isfinite.(frame), frame, missing))))))
    end
    return rows
end


function _snapshot_background_for_plot(background, spaceShape)
    background === nothing && return nothing
    bg = Array(background)
    if size(bg) == spaceShape
        B, replaced, _ = _finite_plot_matrix(bg)
    elseif length(bg) == prod(spaceShape)
        B, replaced, _ = _finite_plot_matrix(reshape(bg, spaceShape...))
    else
        @warn "background size $(size(bg)) does not match wavefield spaceShape $spaceShape; ignoring background"
        return nothing
    end
    replaced > 0 && @warn "background had $replaced non-finite values; replaced by 0 for plotting"
    return B
end

function plot_wave_snapshots(
    frames;
    indices=nothing,
    background=nothing,
    sourcePoint=nothing,
    clim=nothing,
    ncols=3,
    title="wavefield snapshots",
    colormap=:balance,
    background_colormap=:grays,
    wave_alpha=0.72,
)
    isempty(frames) && error("frames is empty")
    if indices === nothing
        indices = unique(round.(Int, range(1, length(frames), length=min(9, length(frames)))))
    end
    indices = collect(indices)

    clean_frames = Array{Float64}[]
    nbad = Int[]
    finite_maxima = Float64[]
    for i in indices
        B, bad, fmax = _finite_plot_matrix(frames[i])
        push!(clean_frames, B)
        push!(nbad, bad)
        push!(finite_maxima, fmax)
    end

    if clim === nothing
        vmax = maximum(finite_maxima)
        clim = vmax == 0 ? 1.0 : vmax
    end

    bg = _snapshot_background_for_plot(background, size(frames[1]))
    n = length(indices)
    nrows = cld(n, ncols)
    fig = Figure(size=(320*ncols, 300*nrows))

    for (k, iframe) in enumerate(indices)
        r = cld(k, ncols)
        c = k - (r - 1) * ncols
        ax = Axis(fig[r, c], aspect=DataAspect(), title="frame $iframe | bad=$(nbad[k])")
        if length(size(frames[1])) == 1
            lines!(ax, vec(clean_frames[k]), color=:dodgerblue3, linewidth=2)
            ylims!(ax, -clim, clim)
            if sourcePoint !== nothing
                vlines!(ax, [Tuple(sourcePoint)[1]], color=:red, linewidth=2)
            end
        elseif length(size(frames[1])) == 2
            if bg !== nothing
                heatmap!(ax, bg; colormap=background_colormap)
            end
            heatmap!(ax, clean_frames[k]; colormap=colormap, colorrange=(-clim, clim), alpha=bg === nothing ? 1.0 : wave_alpha)
            if sourcePoint !== nothing
                xy = Tuple(sourcePoint)
                scatter!(ax, [xy[1]], [xy[2]], color=:red, markersize=10)
            end
        else
            error("Snapshot helper supports 1D/2D frames. Select a slice first for higher dimensions.")
        end
    end

    Label(fig[0, :], title)
    return fig
end


function save_wave_snapshot_pdf(
    frames;
    filename=joinpath(@__DIR__, "wave_snapshots.pdf"),
    indices=nothing,
    background=nothing,
    sourcePoint=nothing,
    clim=nothing,
    ncols=3,
    title="wavefield snapshots",
    kwargs...,
)
    fig = plot_wave_snapshots(
        frames;
        indices=indices,
        background=background,
        sourcePoint=sourcePoint,
        clim=clim,
        ncols=ncols,
        title=title,
        kwargs...,
    )
    save(filename, fig)
    empty!(fig.scene)  # release most displayed scene resources
    GC.gc()
    return filename
end


using Statistics

function cfl_diagnostics(model_velocity, Δnum; cfl_safety=0.45, ppw=10, samples_per_period=20)
    v = vec(Float64.(model_velocity))
    finite_v = filter(isfinite, v)
    dt = Float64(Δnum[end])
    dx = Float64.(Δnum[1:end-1])
    dxmin = minimum(dx)
    dxmax = maximum(dx)
    vmax = maximum(abs, finite_v)
    vmin = minimum(abs, filter(!iszero, finite_v))
    cfl = vmax * dt / dxmin

    # MKS units: velocity [m/s], dx [m], dt [s], frequency [Hz].
    # For a Ricker wavelet in flexOPT, f0 is the dominant/peak frequency.
    fmax_spatial = vmin / (ppw * dxmax)
    fmax_temporal = 1 / (samples_per_period * dt)
    suggested_ricker_f0 = min(fmax_spatial, fmax_temporal)

    return (
        v_min=minimum(finite_v),
        v_median=median(finite_v),
        v_max=maximum(finite_v),
        dt=dt,
        dxmin=dxmin,
        dxmax=dxmax,
        cfl_max=cfl,
        suggested_dt_2D=cfl_safety * dxmin / vmax,
        ppw=ppw,
        samples_per_period=samples_per_period,
        fmax_spatial=fmax_spatial,
        fmax_temporal=fmax_temporal,
        suggested_ricker_f0=suggested_ricker_f0,
        suggested_ricker_t0=1.5 / suggested_ricker_f0,
    )
end

function ricker_time_signal(Nt, Δt; f0, t0=1.5/f0, timePointsUsedForOneStep=1)
    ntime = Nt + timePointsUsedForOneStep - 1
    t = (0:ntime-1) .* Δt
    return t, Ricker.(t, t0, f0)
end

function dominant_frequency_scan(signal, Δt; nfreq=512)
    y = Float64.(signal) .- mean(Float64.(signal))
    n = length(y)
    duration = n * Δt
    fmin = 1 / duration
    fmax = 0.5 / Δt
    freqs = range(fmin, fmax; length=nfreq)
    t = (0:n-1) .* Δt
    spectrum = [abs(sum(y .* exp.(-2im * π * f .* t))) for f in freqs]
    imax = argmax(spectrum)
    return (f_peak=freqs[imax], peak_amplitude=spectrum[imax], frequencies=freqs, spectrum=spectrum)
end

function ricker_diagnostics(model_velocity, Δnum; f0=nothing, Nt=200, timePointsUsedForOneStep=1, ppw=10, samples_per_period=20, t0=nothing)
    cfl = cfl_diagnostics(model_velocity, Δnum; ppw=ppw, samples_per_period=samples_per_period)
    f0_used = f0 === nothing ? cfl.suggested_ricker_f0 : Float64(f0)
    t0_used = t0 === nothing ? 1.5 / f0_used : Float64(t0)
    t, signal = ricker_time_signal(Nt, cfl.dt; f0=f0_used, t0=t0_used, timePointsUsedForOneStep=timePointsUsedForOneStep)
    freqinfo = dominant_frequency_scan(signal, cfl.dt)
    return (
        f0=f0_used,
        t0=t0_used,
        duration=t[end],
        dominant_frequency_scan=freqinfo.f_peak,
        wavelength_min_velocity=cfl.v_min / f0_used,
        points_per_wavelength_min_velocity=(cfl.v_min / f0_used) / cfl.dxmax,
        samples_per_period=1 / (f0_used * cfl.dt),
        max_abs=maximum(abs, signal),
        first_peak_time=t[argmax(abs.(signal))],
        cfl=cfl,
    )
end

function frame_motion_report(frames)
    rows = NamedTuple[]
    previous = nothing
    for (i, frame) in pairs(frames)
        _, nbad, fmax = _finite_plot_matrix(frame)
        dnorm = previous === nothing ? 0.0 : norm(frame .- previous)
        dmax = previous === nothing ? 0.0 : maximum(abs, frame .- previous)
        push!(rows, (frame=i, nbad=nbad, maxabs=fmax, norm=norm(frame), diff_norm=dnorm, diff_max=dmax))
        previous = frame
    end
    return rows
end

function difference_frames(frames)
    length(frames) >= 2 || error("need at least two frames")
    return [frames[i] .- frames[i-1] for i in 2:length(frames)]
end


function _sparse_norms(A)
    return (
        size=size(A),
        nnz=nnz(A),
        maxabs=nnz(A) == 0 ? 0.0 : maximum(abs, nonzeros(A)),
        norm=norm(A),
    )
end

function diagnose_prepared_linear_system(preparedLin)
    println("spaceShape = ", preparedLin.spaceShape)
    println("NpointsSpace = ", preparedLin.NpointsSpace)
    println("NField = ", preparedLin.NField)
    println("NForceField = ", hasproperty(preparedLin, :NForceField) ? preparedLin.NForceField : preparedLin.NField)
    println("timePointsUsedForOneStep = ", preparedLin.timePointsUsedForOneStep)
    println("A_unknown: ", _sparse_norms(preparedLin.A_unknown))
    println("L_known:   ", _sparse_norms(preparedLin.L_known))
    println("R_force:   ", _sparse_norms(preparedLin.R_force))
    if hasproperty(preparedLin, :right_operator)
        println("right_operator.size = ", preparedLin.right_operator.size)
        println("right_operator.nnz = ", length(preparedLin.right_operator.table.vals))
    end
    row_nnz_A = vec(sum(abs.(preparedLin.A_unknown) .> 0; dims=2))
    row_nnz_R = vec(sum(abs.(preparedLin.R_force) .> 0; dims=2))
    println("rows with A entries = ", count(!iszero, row_nnz_A), " / ", length(row_nnz_A))
    println("rows with R entries = ", count(!iszero, row_nnz_R), " / ", length(row_nnz_R))
    return nothing
end

function source_field_response(preparedLin, sourcePoint, timeSignal; amplitude=1.0, it=nothing)
    LI = LinearIndices(preparedLin.spaceShape)
    src_linear = LI[sourcePoint]
    last_start = length(timeSignal) - preparedLin.timePointsUsedForOneStep + 1
    last_start >= 1 || error("timeSignal is too short for timePointsUsedForOneStep=$(preparedLin.timePointsUsedForOneStep)")
    if it === nothing
        it = min(argmax(abs.(timeSignal)), last_start)
    else
        1 <= it <= last_start || error("it should be in 1:$last_start for this timeSignal")
    end

    out = NamedTuple[]
    NForceField = hasproperty(preparedLin, :NForceField) ? preparedLin.NForceField : preparedLin.NField
    for iField in 1:NForceField
        w = zeros(Float64, preparedLin.NforcePoints)
        w[src_linear] = 1.0
        sourceFull = make_sourceFull(preparedLin, w, timeSignal; iField=iField, amplitude=amplitude)
        knownForce = sourceFull[:, :, it:it+preparedLin.timePointsUsedForOneStep-1]
        knownInputs = vcat(vec(zero(preparedLin.known_lhs_template)), vec(knownForce))
        b = copy(preparedLin.b_template)
        preparedLin.b_fun!(b, knownInputs)
        push!(out, (
            iField=iField,
            time_index=it,
            source_norm=norm(knownForce),
            b_norm=norm(b),
            b_maximum=maximum(abs, b),
            b_nonzero=count(!iszero, b),
        ))
    end
    return out
end

function one_step_response(preparedLin, sourceFull; it=1, boundaryConditionForced=false)
    knownField = zero(preparedLin.known_lhs_template)
    knownForce = sourceFull[:, :, it:it+preparedLin.timePointsUsedForOneStep-1]
    knownInputs = vcat(vec(knownField), vec(knownForce))
    b = copy(preparedLin.b_template)
    preparedLin.b_fun!(b, knownInputs)

    A = sparse(preparedLin.A_template)
    if boundaryConditionForced
        apply_forced_boundary_for_video!(A, b; leftValue=0.0, rightValue=0.0)
    end
    u = A \ b
    ufield = reshape(real.(u), preparedLin.NpointsSpace, preparedLin.NField)
    return (
        b_norm=norm(b),
        b_maximum=maximum(abs, b),
        b_nonzero=count(!iszero, b),
        u_norm=norm(u),
        u_maximum=maximum(abs, u),
        u_nonzero=count(!iszero, u),
        ufield=ufield,
    )
end


function _matrix_row_abs_sums(A)
    return vec(sum(abs.(A); dims=2))
end

function operator_growth_diagnostics(preparedLin)
    A = sparse(preparedLin.A_unknown)
    L = sparse(preparedLin.L_known)
    R = sparse(preparedLin.R_force)
    diagA = abs.(diag(A))
    rowA = _matrix_row_abs_sums(A)
    offdiagA = rowA .- diagA
    rowL = _matrix_row_abs_sums(L)
    rowR = _matrix_row_abs_sums(R)
    return (
        A_size=size(A),
        A_nnz=nnz(A),
        A_diag_min=isempty(diagA) ? 0.0 : minimum(diagA),
        A_diag_median=isempty(diagA) ? 0.0 : median(diagA),
        A_diag_max=isempty(diagA) ? 0.0 : maximum(diagA),
        A_offdiag_over_diag_max=maximum(offdiagA ./ max.(diagA, eps(Float64))),
        L_row_sum_max=maximum(rowL),
        R_row_sum_max=maximum(rowR),
        R_over_A_diag_max=maximum(rowR ./ max.(diagA, eps(Float64))),
        condest_dense_sample=size(A,1) <= 5000 ? cond(Matrix(A)) : missing,
    )
end

function homogeneous_zero_test(preparedLin; Nt=20, store_every=1)
    sourceFull_zero = zeros(eltype(preparedLin.known_rhs_template), preparedLin.NforcePoints, preparedLin.NForceField, Nt + preparedLin.timePointsUsedForOneStep - 1)
    frames_zero = propagate_linear_frames(preparedLin, sourceFull_zero, Nt; store_every=store_every, blowup_limit=1e12)
    return wavefield_snapshot_report(frames_zero)
end

function source_amplitude_sweep(preparedLin, sourcePoint, timeSignal; amplitudes=(1e-12, 1e-10, 1e-8, 1e-6), iField=1, Nt=20)
    rows = NamedTuple[]
    w = point_source_weights(preparedLin, sourcePoint)
    for amp in amplitudes
        sourceFull = make_sourceFull(preparedLin, w, timeSignal; iField=iField, amplitude=amp)
        frames = propagate_linear_frames(preparedLin, sourceFull, Nt; store_every=max(1, Nt ÷ 5), blowup_limit=1e30)
        report = wavefield_snapshot_report(frames)
        push!(rows, (amplitude=amp, last=report[end], nframes=length(frames)))
    end
    return rows
end


function _distance_from_source_table(frame, sourcePoint; threshold_fraction=1e-6)
    f = abs.(Float64.(frame))
    fmax = maximum(f)
    threshold = threshold_fraction * max(fmax, eps(Float64))
    active = findall(>(threshold), f)
    if isempty(active)
        return (nactive=0, max_distance=0.0, mean_distance=0.0, fmax=fmax)
    end
    distances = [sqrt(sum((Tuple(I) .- Tuple(sourcePoint)).^2)) for I in active]
    return (
        nactive=length(active),
        max_distance=maximum(distances),
        mean_distance=mean(distances),
        fmax=fmax,
    )
end

function propagation_radius_report(frames, sourcePoint; threshold_fraction=1e-6)
    return [(frame=i, _distance_from_source_table(frame, sourcePoint; threshold_fraction=threshold_fraction)...) for (i, frame) in pairs(frames)]
end

function impulse_sourceFull(preparedLin, sourcePoint; iField=1, amplitude=1.0, Nt=20, impulse_time=1)
    timeSignal = zeros(Float64, Nt + preparedLin.timePointsUsedForOneStep - 1)
    timeSignal[impulse_time] = 1.0
    w = point_source_weights(preparedLin, sourcePoint)
    return make_sourceFull(preparedLin, w, timeSignal; iField=iField, amplitude=amplitude)
end

function impulse_propagation_test(preparedLin, sourcePoint; iField=1, amplitude=1.0, Nt=20, store_every=1, threshold_fraction=1e-8)
    sourceFull = impulse_sourceFull(preparedLin, sourcePoint; iField=iField, amplitude=amplitude, Nt=Nt)
    frames = propagate_linear_frames(preparedLin, sourceFull, Nt; store_every=store_every, blowup_limit=1e20)
    return (
        frames=frames,
        snapshots=wavefield_snapshot_report(frames),
        radius=propagation_radius_report(frames, sourcePoint; threshold_fraction=threshold_fraction),
    )
end


function frame_extrema_report(frames, sourcePoint)
    rows = NamedTuple[]
    for (i, frame) in pairs(frames)
        A = Float64.(frame)
        absA = abs.(A)
        imax = argmax(absA)
        maxpoint = CartesianIndices(size(A))[imax]
        distance = sqrt(sum((Tuple(maxpoint) .- Tuple(sourcePoint)).^2))
        push!(rows, (
            frame=i,
            maxpoint=maxpoint,
            distance_from_source=distance,
            value=A[imax],
            maxabs=absA[imax],
            minimum=minimum(A),
            maximum=maximum(A),
        ))
    end
    return rows
end


function _get_numop(numOps, which::Symbol)
    ops = numOps isa AbstractDict ? numOps["numericalOperators"] : numOps.numericalOperators
    return getproperty(ops, which)
end

function _point_linear_index(op, point)
    νWhole = op.geometry.νWhole
    if point isa Integer
        1 <= point <= length(νWhole) || error("point integer should be in 1:$(length(νWhole))")
        return Int(point), νWhole[Int(point)]
    elseif point isa CartesianIndex
        point in νWhole || error("point $point is not in op.geometry.νWhole")
        pointLinear = LinearIndices(νWhole)[point]
        return Int(pointLinear), point
    else
        error("point should be an Int linear point index or a CartesianIndex")
    end
end

function _time_role(iT, activeTimePoints)
    if iT == activeTimePoints
        return :future_unknown
    elseif iT == activeTimePoints - 1
        return :present_known
    else
        return Symbol("past_known_tminus$(activeTimePoints - iT)")
    end
end

function coefficient_rows_at_point(op, point; iExpr=1, atol=0.0)
    pointLinear, pointCI = _point_linear_index(op, point)
    nPoints = length(op.geometry.νWhole)
    activeTimePoints = op.geometry.activeTimePoints
    spaceShape = Tuple(op.geometry.wholeRegionPointsSpace)
    Nexpr = op.size[1] ÷ nPoints
    Nfield = op.size[2] ÷ (nPoints * activeTimePoints)

    1 <= iExpr <= Nexpr || error("iExpr should be in 1:$Nexpr")

    residualLI = LinearIndices((Nexpr, nPoints))
    targetRow = residualLI[iExpr, pointLinear]
    fieldTimeSpaceCI = CartesianIndices((Nfield, activeTimePoints, spaceShape...))

    rows = NamedTuple[]
    for k in eachindex(op.table.vals)
        Int(op.table.rows[k]) == targetRow || continue
        val = op.table.vals[k]
        abs(val) > atol || continue

        ci = fieldTimeSpaceCI[Int(op.table.cols[k])]
        iField = ci[1]
        iT = ci[2]
        neighbour = CartesianIndex(Tuple(ci)[3:end])
        offset = Tuple(neighbour - pointCI)
        timeRole = _time_role(iT, activeTimePoints)

        push!(rows, (
            k = k,
            residual_row = targetRow,
            expr = iExpr,
            point = pointCI,
            field = iField,
            time_slot = iT,
            time_role = timeRole,
            neighbour = neighbour,
            offset = offset,
            coef = val,
            abscoef = abs(val),
        ))
    end

    sort!(rows, by = r -> (r.time_slot, r.field, r.offset))
    return rows
end

function inspect_operator_coefficients_at_point(numOps, point; which=:residual, iExpr=1, atol=0.0)
    op = _get_numop(numOps, which)
    rows = coefficient_rows_at_point(op, point; iExpr=iExpr, atol=atol)
    println("operator = ", which)
    println("point = ", point, ", iExpr = ", iExpr, ", ncoef = ", length(rows))
    for r in rows
        println(
            "t=", r.time_slot,
            " (", r.time_role, ")",
            " field=", r.field,
            " offset=", r.offset,
            " neighbour=", r.neighbour,
            " coef=", r.coef,
        )
    end
    return rows
end

function coefficient_table_by_time(rows)
    slots = sort(unique(getproperty.(rows, :time_slot)))
    return [(
        time_slot=s,
        time_role=first(r.time_role for r in rows if r.time_slot == s),
        n=length(filter(r -> r.time_slot == s, rows)),
        sumcoef=sum(r.coef for r in rows if r.time_slot == s),
        sumabs=sum(r.abscoef for r in rows if r.time_slot == s),
    ) for s in slots]
end

function coefficient_rows_for_role(rows, role::Symbol)
    return filter(r -> r.time_role == role, rows)
end


function source_time_samples(Nt, Δt, timePointsUsedForOneStep; wavelet=nothing, t0=12Δt, f0=0.04)
    ntime = Nt + timePointsUsedForOneStep - 1
    t = (0:ntime-1) .* Δt
    if wavelet === nothing
        return Ricker.(t, t0, f0)
    else
        return wavelet.(t)
    end
end

function _normalise_source_weights!(w; normalise=:none)
    if normalise == :none || normalise === false
        return w
    elseif normalise == :sum
        s = sum(w)
        !iszero(s) && (w ./= s)
    elseif normalise == :l1
        s = sum(abs, w)
        !iszero(s) && (w ./= s)
    elseif normalise == :max
        s = maximum(abs, w)
        !iszero(s) && (w ./= s)
    else
        error("normalise should be :none, :sum, :l1, or :max")
    end
    return w
end

function make_sourceFull(preparedLin, spatialWeights, timeSignal; iField=1, amplitude=1.0)
    length(spatialWeights) == preparedLin.NforcePoints ||
        error("spatialWeights should have length preparedLin.NforcePoints=$(preparedLin.NforcePoints)")
    NForceField = hasproperty(preparedLin, :NForceField) ? preparedLin.NForceField : preparedLin.NField
    1 <= iField <= NForceField || error("iField should be between 1 and $(NForceField)")

    T = promote_type(eltype(spatialWeights), eltype(timeSignal), typeof(amplitude))
    sourceFull = zeros(T, preparedLin.NforcePoints, NForceField, length(timeSignal))
    for it in eachindex(timeSignal)
        sourceFull[:, iField, it] .= amplitude .* timeSignal[it] .* spatialWeights
    end
    return sourceFull
end

function point_source_weights(preparedLin, point::CartesianIndex; normalise=:none)
    weights = zeros(Float64, preparedLin.NforcePoints)
    LI = LinearIndices(preparedLin.spaceShape)
    weights[LI[point]] = 1.0
    return _normalise_source_weights!(weights; normalise)
end

function hyperplane_source_weights(preparedLin; fixed=Dict{Int,Int}(), profile=(I -> 1.0), normalise=:sum)
    weights = zeros(Float64, preparedLin.NforcePoints)
    LI = LinearIndices(preparedLin.spaceShape)
    for I in CartesianIndices(preparedLin.spaceShape)
        coords = Tuple(I)
        inside = all(coords[d] == v for (d, v) in fixed)
        if inside
            weights[LI[I]] = profile(I)
        end
    end
    return _normalise_source_weights!(weights; normalise)
end

function mask_source_weights(preparedLin, mask; profile=(I -> 1.0), normalise=:sum)
    weights = zeros(Float64, preparedLin.NforcePoints)
    LI = LinearIndices(preparedLin.spaceShape)

    if mask isa AbstractArray{Bool}
        size(mask) == preparedLin.spaceShape || error("Boolean mask size should be $(preparedLin.spaceShape)")
        for I in CartesianIndices(mask)
            mask[I] && (weights[LI[I]] = profile(I))
        end
    else
        for I in mask
            weights[LI[I]] = profile(I)
        end
    end

    return _normalise_source_weights!(weights; normalise)
end


using LinearAlgebra, SparseArrays
using CairoMakie
CairoMakie.activate!()

function apply_forced_boundary_for_video!(A, b; leftValue=0.0, rightValue=0.0)
    A[1, :] .= 0
    A[1, 1] = 1
    b[1] = leftValue

    A[end, :] .= 0
    A[end, end] = 1
    b[end] = rightValue
    return A, b
end

function propagate_linear_frames(
    preparedLin,
    sourceFull,
    Nt;
    initialCondition=0.0,
    boundaryConditionForced=false,
    store_every=1,
    blowup_limit=Inf,
)
    NField = preparedLin.NField
    NForceField = hasproperty(preparedLin, :NForceField) ? preparedLin.NForceField : preparedLin.NField
    NpointsSpace = preparedLin.NpointsSpace
    timePointsUsedForOneStep = preparedLin.timePointsUsedForOneStep
    NknownTime = max(timePointsUsedForOneStep - 1, 0)

    size(sourceFull, 1) == preparedLin.NforcePoints || error("sourceFull first dimension mismatch")
    size(sourceFull, 2) == NForceField || error("sourceFull second dimension mismatch")
    size(sourceFull, 3) >= Nt + timePointsUsedForOneStep - 1 || error("sourceFull has too few time samples")

    knownField = fill(Float64(initialCondition), size(preparedLin.known_lhs_template))
    knownForce = fill(Float64(initialCondition), size(preparedLin.known_rhs_template))
    unknownField = fill(Float64(initialCondition), NpointsSpace, NField)

    A = sparse(preparedLin.A_template)
    b0 = copy(preparedLin.b_template)
    if boundaryConditionForced
        apply_forced_boundary_for_video!(A, b0; leftValue=0.0, rightValue=0.0)
    end
    factor = lu(A)
    b = copy(preparedLin.b_template)

    frames = Vector{Array{Float64}}()
    for it in 1:Nt
        knownForce .= sourceFull[:, :, it:it+timePointsUsedForOneStep-1]
        knownInputs = vcat(vec(knownField), vec(knownForce))
        preparedLin.b_fun!(b, knownInputs)
        if boundaryConditionForced
            b[1] = 0.0
            b[end] = 0.0
        end

        u = factor \ b
        unknownField .= reshape(real.(u), NpointsSpace, NField)

        umax = maximum(abs, unknownField)
        if !isfinite(umax) || umax > blowup_limit
            @warn "Stopping propagation because the wavefield blew up" it umax blowup_limit
            push!(frames, reshape(copy(unknownField[:, 1]), preparedLin.spaceShape...))
            break
        end

        if it == 1 || it % store_every == 0 || it == Nt
            push!(frames, reshape(copy(unknownField[:, 1]), preparedLin.spaceShape...))
        end

        if NknownTime > 0
            if NknownTime > 1
                knownField[:, :, 1:end-1] .= knownField[:, :, 2:end]
            end
            knownField[:, :, end] .= unknownField
        end
    end
    return frames
end

function _finite_copy(A; fillvalue=0.0)
    B = Array{Float64}(undef, size(A))
    replaced = 0
    for I in eachindex(A)
        v = Float64(real(A[I]))
        if isfinite(v)
            B[I] = v
        else
            B[I] = fillvalue
            replaced += 1
        end
    end
    return B, replaced
end

function finite_frame_diagnostics(frames)
    total_bad = 0
    first_bad = nothing
    finite_max = 0.0
    for (i, frame) in pairs(frames)
        bad_here = count(!isfinite, frame)
        if bad_here > 0 && first_bad === nothing
            first_bad = i
        end
        total_bad += bad_here
        for v in frame
            if isfinite(v)
                finite_max = max(finite_max, abs(Float64(v)))
            end
        end
    end
    return (total_bad=total_bad, first_bad=first_bad, finite_max=finite_max)
end

function _background_for_plot(background, spaceShape)
    background === nothing && return nothing
    bg = Array(background)
    if size(bg) == spaceShape
        clean_bg, replaced = _finite_copy(bg)
    elseif length(bg) == prod(spaceShape)
        clean_bg, replaced = _finite_copy(reshape(bg, spaceShape...))
    else
        @warn "background size $(size(bg)) does not match wavefield spaceShape $spaceShape; ignoring background"
        return nothing
    end
    replaced > 0 && @warn "background had $replaced non-finite values; replaced by 0 for plotting"
    return clean_bg
end

function record_wave_video(
    frames;
    videoFile=joinpath(@__DIR__, "point_source_propagation.mp4"),
    background=nothing,
    sourcePoint=nothing,
    framerate=20,
    title="point source propagation",
    colormap=:balance,
    background_colormap=:grays,
    wave_alpha=0.72,
    clim=nothing,
)
    isempty(frames) && error("frames is empty")
    spaceShape = size(frames[1])
    bg = _background_for_plot(background, spaceShape)
    nd = length(spaceShape)

    diag = finite_frame_diagnostics(frames)
    if diag.total_bad > 0
        @warn "wavefield frames contain $(diag.total_bad) non-finite values; first affected stored frame is $(diag.first_bad). Replacing by 0 for plotting."
    end
    clean_frames = [first(_finite_copy(f)) for f in frames]

    if clim === nothing
        vmax = diag.finite_max
        clim = vmax == 0 ? 1.0 : vmax
    end

    fig = Figure(size=(900, 680))
    ax = Axis(fig[1, 1], aspect=DataAspect(), title=title)

    if nd == 1
        x = 1:spaceShape[1]
        yobs = Observable(vec(clean_frames[1]))
        lines!(ax, x, yobs, color=:dodgerblue3, linewidth=2)
        ylims!(ax, -clim, clim)
        if sourcePoint !== nothing
            vlines!(ax, [Tuple(sourcePoint)[1]], color=:red, linewidth=2)
        end
        record(fig, videoFile, eachindex(frames); framerate=framerate) do iframe
            yobs[] = vec(clean_frames[iframe])
            ax.title = "$title  |  frame $iframe/$(length(frames))"
        end
    elseif nd == 2
        if bg !== nothing
            heatmap!(ax, bg; colormap=background_colormap)
        end
        frame_obs = Observable(clean_frames[1])
        hm = heatmap!(ax, frame_obs; colormap=colormap, colorrange=(-clim, clim), alpha=bg === nothing ? 1.0 : wave_alpha)
        Colorbar(fig[1, 2], hm, label="wavefield")
        if sourcePoint !== nothing
            xy = Tuple(sourcePoint)
            scatter!(ax, [xy[1]], [xy[2]], color=:red, markersize=14)
        end
        record(fig, videoFile, eachindex(frames); framerate=framerate) do iframe
            frame_obs[] = clean_frames[iframe]
            ax.title = "$title  |  frame $iframe/$(length(frames))"
        end
    else
        error("Video helper currently supports 1D/2D frames. For 3D/4D, select a slice before recording.")
    end

    return videoFile
end


function _finite_plot_matrix(A; fillvalue=0.0)
    B = Array{Float64}(undef, size(A))
    nbad = 0
    finite_max = 0.0
    for I in eachindex(A)
        v = Float64(real(A[I]))
        if isfinite(v)
            B[I] = v
            finite_max = max(finite_max, abs(v))
        else
            B[I] = fillvalue
            nbad += 1
        end
    end
    return B, nbad, finite_max
end

function wavefield_snapshot_report(frames)
    rows = NamedTuple[]
    for (i, frame) in pairs(frames)
        _, nbad, finite_max = _finite_plot_matrix(frame)
        push!(rows, (frame=i, nbad=nbad, finite_max=finite_max, minimum=minimum(skipmissing(vec(ifelse.(isfinite.(frame), frame, missing)))), maximum=maximum(skipmissing(vec(ifelse.(isfinite.(frame), frame, missing))))))
    end
    return rows
end


function _snapshot_background_for_plot(background, spaceShape)
    background === nothing && return nothing
    bg = Array(background)
    if size(bg) == spaceShape
        B, replaced, _ = _finite_plot_matrix(bg)
    elseif length(bg) == prod(spaceShape)
        B, replaced, _ = _finite_plot_matrix(reshape(bg, spaceShape...))
    else
        @warn "background size $(size(bg)) does not match wavefield spaceShape $spaceShape; ignoring background"
        return nothing
    end
    replaced > 0 && @warn "background had $replaced non-finite values; replaced by 0 for plotting"
    return B
end

function plot_wave_snapshots(
    frames;
    indices=nothing,
    background=nothing,
    sourcePoint=nothing,
    clim=nothing,
    ncols=3,
    title="wavefield snapshots",
    colormap=:balance,
    background_colormap=:grays,
    wave_alpha=0.72,
)
    isempty(frames) && error("frames is empty")
    if indices === nothing
        indices = unique(round.(Int, range(1, length(frames), length=min(9, length(frames)))))
    end
    indices = collect(indices)

    clean_frames = Array{Float64}[]
    nbad = Int[]
    finite_maxima = Float64[]
    for i in indices
        B, bad, fmax = _finite_plot_matrix(frames[i])
        push!(clean_frames, B)
        push!(nbad, bad)
        push!(finite_maxima, fmax)
    end

    if clim === nothing
        vmax = maximum(finite_maxima)
        clim = vmax == 0 ? 1.0 : vmax
    end

    bg = _snapshot_background_for_plot(background, size(frames[1]))
    n = length(indices)
    nrows = cld(n, ncols)
    fig = Figure(size=(320*ncols, 300*nrows))

    for (k, iframe) in enumerate(indices)
        r = cld(k, ncols)
        c = k - (r - 1) * ncols
        ax = Axis(fig[r, c], aspect=DataAspect(), title="frame $iframe | bad=$(nbad[k])")
        if length(size(frames[1])) == 1
            lines!(ax, vec(clean_frames[k]), color=:dodgerblue3, linewidth=2)
            ylims!(ax, -clim, clim)
            if sourcePoint !== nothing
                vlines!(ax, [Tuple(sourcePoint)[1]], color=:red, linewidth=2)
            end
        elseif length(size(frames[1])) == 2
            if bg !== nothing
                heatmap!(ax, bg; colormap=background_colormap)
            end
            heatmap!(ax, clean_frames[k]; colormap=colormap, colorrange=(-clim, clim), alpha=bg === nothing ? 1.0 : wave_alpha)
            if sourcePoint !== nothing
                xy = Tuple(sourcePoint)
                scatter!(ax, [xy[1]], [xy[2]], color=:red, markersize=10)
            end
        else
            error("Snapshot helper supports 1D/2D frames. Select a slice first for higher dimensions.")
        end
    end

    Label(fig[0, :], title)
    return fig
end


function save_wave_snapshot_pdf(
    frames;
    filename=joinpath(@__DIR__, "wave_snapshots.pdf"),
    indices=nothing,
    background=nothing,
    sourcePoint=nothing,
    clim=nothing,
    ncols=3,
    title="wavefield snapshots",
    kwargs...,
)
    fig = plot_wave_snapshots(
        frames;
        indices=indices,
        background=background,
        sourcePoint=sourcePoint,
        clim=clim,
        ncols=ncols,
        title=title,
        kwargs...,
    )
    save(filename, fig)
    empty!(fig.scene)  # release most displayed scene resources
    GC.gc()
    return filename
end


using Statistics

function cfl_diagnostics(model_velocity, Δnum; cfl_safety=0.45, ppw=10, samples_per_period=20)
    v = vec(Float64.(model_velocity))
    finite_v = filter(isfinite, v)
    dt = Float64(Δnum[end])
    dx = Float64.(Δnum[1:end-1])
    dxmin = minimum(dx)
    dxmax = maximum(dx)
    vmax = maximum(abs, finite_v)
    vmin = minimum(abs, filter(!iszero, finite_v))
    cfl = vmax * dt / dxmin

    # MKS units: velocity [m/s], dx [m], dt [s], frequency [Hz].
    # For a Ricker wavelet in flexOPT, f0 is the dominant/peak frequency.
    fmax_spatial = vmin / (ppw * dxmax)
    fmax_temporal = 1 / (samples_per_period * dt)
    suggested_ricker_f0 = min(fmax_spatial, fmax_temporal)

    return (
        v_min=minimum(finite_v),
        v_median=median(finite_v),
        v_max=maximum(finite_v),
        dt=dt,
        dxmin=dxmin,
        dxmax=dxmax,
        cfl_max=cfl,
        suggested_dt_2D=cfl_safety * dxmin / vmax,
        ppw=ppw,
        samples_per_period=samples_per_period,
        fmax_spatial=fmax_spatial,
        fmax_temporal=fmax_temporal,
        suggested_ricker_f0=suggested_ricker_f0,
        suggested_ricker_t0=1.5 / suggested_ricker_f0,
    )
end

function ricker_time_signal(Nt, Δt; f0, t0=1.5/f0, timePointsUsedForOneStep=1)
    ntime = Nt + timePointsUsedForOneStep - 1
    t = (0:ntime-1) .* Δt
    return t, Ricker.(t, t0, f0)
end

function dominant_frequency_scan(signal, Δt; nfreq=512)
    y = Float64.(signal) .- mean(Float64.(signal))
    n = length(y)
    duration = n * Δt
    fmin = 1 / duration
    fmax = 0.5 / Δt
    freqs = range(fmin, fmax; length=nfreq)
    t = (0:n-1) .* Δt
    spectrum = [abs(sum(y .* exp.(-2im * π * f .* t))) for f in freqs]
    imax = argmax(spectrum)
    return (f_peak=freqs[imax], peak_amplitude=spectrum[imax], frequencies=freqs, spectrum=spectrum)
end

function ricker_diagnostics(model_velocity, Δnum; f0=nothing, Nt=200, timePointsUsedForOneStep=1, ppw=10, samples_per_period=20, t0=nothing)
    cfl = cfl_diagnostics(model_velocity, Δnum; ppw=ppw, samples_per_period=samples_per_period)
    f0_used = f0 === nothing ? cfl.suggested_ricker_f0 : Float64(f0)
    t0_used = t0 === nothing ? 1.5 / f0_used : Float64(t0)
    t, signal = ricker_time_signal(Nt, cfl.dt; f0=f0_used, t0=t0_used, timePointsUsedForOneStep=timePointsUsedForOneStep)
    freqinfo = dominant_frequency_scan(signal, cfl.dt)
    return (
        f0=f0_used,
        t0=t0_used,
        duration=t[end],
        dominant_frequency_scan=freqinfo.f_peak,
        wavelength_min_velocity=cfl.v_min / f0_used,
        points_per_wavelength_min_velocity=(cfl.v_min / f0_used) / cfl.dxmax,
        samples_per_period=1 / (f0_used * cfl.dt),
        max_abs=maximum(abs, signal),
        first_peak_time=t[argmax(abs.(signal))],
        cfl=cfl,
    )
end

function frame_motion_report(frames)
    rows = NamedTuple[]
    previous = nothing
    for (i, frame) in pairs(frames)
        _, nbad, fmax = _finite_plot_matrix(frame)
        dnorm = previous === nothing ? 0.0 : norm(frame .- previous)
        dmax = previous === nothing ? 0.0 : maximum(abs, frame .- previous)
        push!(rows, (frame=i, nbad=nbad, maxabs=fmax, norm=norm(frame), diff_norm=dnorm, diff_max=dmax))
        previous = frame
    end
    return rows
end

function difference_frames(frames)
    length(frames) >= 2 || error("need at least two frames")
    return [frames[i] .- frames[i-1] for i in 2:length(frames)]
end


function _sparse_norms(A)
    return (
        size=size(A),
        nnz=nnz(A),
        maxabs=nnz(A) == 0 ? 0.0 : maximum(abs, nonzeros(A)),
        norm=norm(A),
    )
end

function diagnose_prepared_linear_system(preparedLin)
    println("spaceShape = ", preparedLin.spaceShape)
    println("NpointsSpace = ", preparedLin.NpointsSpace)
    println("NField = ", preparedLin.NField)
    println("NForceField = ", hasproperty(preparedLin, :NForceField) ? preparedLin.NForceField : preparedLin.NField)
    println("timePointsUsedForOneStep = ", preparedLin.timePointsUsedForOneStep)
    println("A_unknown: ", _sparse_norms(preparedLin.A_unknown))
    println("L_known:   ", _sparse_norms(preparedLin.L_known))
    println("R_force:   ", _sparse_norms(preparedLin.R_force))
    if hasproperty(preparedLin, :right_operator)
        println("right_operator.size = ", preparedLin.right_operator.size)
        println("right_operator.nnz = ", length(preparedLin.right_operator.table.vals))
    end
    row_nnz_A = vec(sum(abs.(preparedLin.A_unknown) .> 0; dims=2))
    row_nnz_R = vec(sum(abs.(preparedLin.R_force) .> 0; dims=2))
    println("rows with A entries = ", count(!iszero, row_nnz_A), " / ", length(row_nnz_A))
    println("rows with R entries = ", count(!iszero, row_nnz_R), " / ", length(row_nnz_R))
    return nothing
end

function source_field_response(preparedLin, sourcePoint, timeSignal; amplitude=1.0, it=nothing)
    LI = LinearIndices(preparedLin.spaceShape)
    src_linear = LI[sourcePoint]
    last_start = length(timeSignal) - preparedLin.timePointsUsedForOneStep + 1
    last_start >= 1 || error("timeSignal is too short for timePointsUsedForOneStep=$(preparedLin.timePointsUsedForOneStep)")
    if it === nothing
        it = min(argmax(abs.(timeSignal)), last_start)
    else
        1 <= it <= last_start || error("it should be in 1:$last_start for this timeSignal")
    end

    out = NamedTuple[]
    NForceField = hasproperty(preparedLin, :NForceField) ? preparedLin.NForceField : preparedLin.NField
    for iField in 1:NForceField
        w = zeros(Float64, preparedLin.NforcePoints)
        w[src_linear] = 1.0
        sourceFull = make_sourceFull(preparedLin, w, timeSignal; iField=iField, amplitude=amplitude)
        knownForce = sourceFull[:, :, it:it+preparedLin.timePointsUsedForOneStep-1]
        knownInputs = vcat(vec(zero(preparedLin.known_lhs_template)), vec(knownForce))
        b = copy(preparedLin.b_template)
        preparedLin.b_fun!(b, knownInputs)
        push!(out, (
            iField=iField,
            time_index=it,
            source_norm=norm(knownForce),
            b_norm=norm(b),
            b_maximum=maximum(abs, b),
            b_nonzero=count(!iszero, b),
        ))
    end
    return out
end

function one_step_response(preparedLin, sourceFull; it=1, boundaryConditionForced=false)
    knownField = zero(preparedLin.known_lhs_template)
    knownForce = sourceFull[:, :, it:it+preparedLin.timePointsUsedForOneStep-1]
    knownInputs = vcat(vec(knownField), vec(knownForce))
    b = copy(preparedLin.b_template)
    preparedLin.b_fun!(b, knownInputs)

    A = sparse(preparedLin.A_template)
    if boundaryConditionForced
        apply_forced_boundary_for_video!(A, b; leftValue=0.0, rightValue=0.0)
    end
    u = A \ b
    ufield = reshape(real.(u), preparedLin.NpointsSpace, preparedLin.NField)
    return (
        b_norm=norm(b),
        b_maximum=maximum(abs, b),
        b_nonzero=count(!iszero, b),
        u_norm=norm(u),
        u_maximum=maximum(abs, u),
        u_nonzero=count(!iszero, u),
        ufield=ufield,
    )
end


function _matrix_row_abs_sums(A)
    return vec(sum(abs.(A); dims=2))
end

function operator_growth_diagnostics(preparedLin)
    A = sparse(preparedLin.A_unknown)
    L = sparse(preparedLin.L_known)
    R = sparse(preparedLin.R_force)
    diagA = abs.(diag(A))
    rowA = _matrix_row_abs_sums(A)
    offdiagA = rowA .- diagA
    rowL = _matrix_row_abs_sums(L)
    rowR = _matrix_row_abs_sums(R)
    return (
        A_size=size(A),
        A_nnz=nnz(A),
        A_diag_min=isempty(diagA) ? 0.0 : minimum(diagA),
        A_diag_median=isempty(diagA) ? 0.0 : median(diagA),
        A_diag_max=isempty(diagA) ? 0.0 : maximum(diagA),
        A_offdiag_over_diag_max=maximum(offdiagA ./ max.(diagA, eps(Float64))),
        L_row_sum_max=maximum(rowL),
        R_row_sum_max=maximum(rowR),
        R_over_A_diag_max=maximum(rowR ./ max.(diagA, eps(Float64))),
        condest_dense_sample=size(A,1) <= 5000 ? cond(Matrix(A)) : missing,
    )
end

function homogeneous_zero_test(preparedLin; Nt=20, store_every=1)
    sourceFull_zero = zeros(eltype(preparedLin.known_rhs_template), preparedLin.NforcePoints, preparedLin.NForceField, Nt + preparedLin.timePointsUsedForOneStep - 1)
    frames_zero = propagate_linear_frames(preparedLin, sourceFull_zero, Nt; store_every=store_every, blowup_limit=1e12)
    return wavefield_snapshot_report(frames_zero)
end

function source_amplitude_sweep(preparedLin, sourcePoint, timeSignal; amplitudes=(1e-12, 1e-10, 1e-8, 1e-6), iField=1, Nt=20)
    rows = NamedTuple[]
    w = point_source_weights(preparedLin, sourcePoint)
    for amp in amplitudes
        sourceFull = make_sourceFull(preparedLin, w, timeSignal; iField=iField, amplitude=amp)
        frames = propagate_linear_frames(preparedLin, sourceFull, Nt; store_every=max(1, Nt ÷ 5), blowup_limit=1e30)
        report = wavefield_snapshot_report(frames)
        push!(rows, (amplitude=amp, last=report[end], nframes=length(frames)))
    end
    return rows
end


function _distance_from_source_table(frame, sourcePoint; threshold_fraction=1e-6)
    f = abs.(Float64.(frame))
    fmax = maximum(f)
    threshold = threshold_fraction * max(fmax, eps(Float64))
    active = findall(>(threshold), f)
    if isempty(active)
        return (nactive=0, max_distance=0.0, mean_distance=0.0, fmax=fmax)
    end
    distances = [sqrt(sum((Tuple(I) .- Tuple(sourcePoint)).^2)) for I in active]
    return (
        nactive=length(active),
        max_distance=maximum(distances),
        mean_distance=mean(distances),
        fmax=fmax,
    )
end

function propagation_radius_report(frames, sourcePoint; threshold_fraction=1e-6)
    return [(frame=i, _distance_from_source_table(frame, sourcePoint; threshold_fraction=threshold_fraction)...) for (i, frame) in pairs(frames)]
end

function impulse_sourceFull(preparedLin, sourcePoint; iField=1, amplitude=1.0, Nt=20, impulse_time=1)
    timeSignal = zeros(Float64, Nt + preparedLin.timePointsUsedForOneStep - 1)
    timeSignal[impulse_time] = 1.0
    w = point_source_weights(preparedLin, sourcePoint)
    return make_sourceFull(preparedLin, w, timeSignal; iField=iField, amplitude=amplitude)
end

function impulse_propagation_test(preparedLin, sourcePoint; iField=1, amplitude=1.0, Nt=20, store_every=1, threshold_fraction=1e-8)
    sourceFull = impulse_sourceFull(preparedLin, sourcePoint; iField=iField, amplitude=amplitude, Nt=Nt)
    frames = propagate_linear_frames(preparedLin, sourceFull, Nt; store_every=store_every, blowup_limit=1e20)
    return (
        frames=frames,
        snapshots=wavefield_snapshot_report(frames),
        radius=propagation_radius_report(frames, sourcePoint; threshold_fraction=threshold_fraction),
    )
end


function frame_extrema_report(frames, sourcePoint)
    rows = NamedTuple[]
    for (i, frame) in pairs(frames)
        A = Float64.(frame)
        absA = abs.(A)
        imax = argmax(absA)
        maxpoint = CartesianIndices(size(A))[imax]
        distance = sqrt(sum((Tuple(maxpoint) .- Tuple(sourcePoint)).^2))
        push!(rows, (
            frame=i,
            maxpoint=maxpoint,
            distance_from_source=distance,
            value=A[imax],
            maxabs=absA[imax],
            minimum=minimum(A),
            maximum=maximum(A),
        ))
    end
    return rows
end


function _get_numop(numOps, which::Symbol)
    ops = numOps isa AbstractDict ? numOps["numericalOperators"] : numOps.numericalOperators
    return getproperty(ops, which)
end

function _point_linear_index(op, point)
    νWhole = op.geometry.νWhole
    if point isa Integer
        1 <= point <= length(νWhole) || error("point integer should be in 1:$(length(νWhole))")
        return Int(point), νWhole[Int(point)]
    elseif point isa CartesianIndex
        point in νWhole || error("point $point is not in op.geometry.νWhole")
        pointLinear = LinearIndices(νWhole)[point]
        return Int(pointLinear), point
    else
        error("point should be an Int linear point index or a CartesianIndex")
    end
end

function _time_role(iT, activeTimePoints)
    if iT == activeTimePoints
        return :future_unknown
    elseif iT == activeTimePoints - 1
        return :present_known
    else
        return Symbol("past_known_tminus$(activeTimePoints - iT)")
    end
end

function coefficient_rows_at_point(op, point; iExpr=1, atol=0.0)
    pointLinear, pointCI = _point_linear_index(op, point)
    nPoints = length(op.geometry.νWhole)
    activeTimePoints = op.geometry.activeTimePoints
    spaceShape = Tuple(op.geometry.wholeRegionPointsSpace)
    Nexpr = op.size[1] ÷ nPoints
    Nfield = op.size[2] ÷ (nPoints * activeTimePoints)

    1 <= iExpr <= Nexpr || error("iExpr should be in 1:$Nexpr")

    residualLI = LinearIndices((Nexpr, nPoints))
    targetRow = residualLI[iExpr, pointLinear]
    fieldTimeSpaceCI = CartesianIndices((Nfield, activeTimePoints, spaceShape...))

    rows = NamedTuple[]
    for k in eachindex(op.table.vals)
        Int(op.table.rows[k]) == targetRow || continue
        val = op.table.vals[k]
        abs(val) > atol || continue

        ci = fieldTimeSpaceCI[Int(op.table.cols[k])]
        iField = ci[1]
        iT = ci[2]
        neighbour = CartesianIndex(Tuple(ci)[3:end])
        offset = Tuple(neighbour - pointCI)
        timeRole = _time_role(iT, activeTimePoints)

        push!(rows, (
            k = k,
            residual_row = targetRow,
            expr = iExpr,
            point = pointCI,
            field = iField,
            time_slot = iT,
            time_role = timeRole,
            neighbour = neighbour,
            offset = offset,
            coef = val,
            abscoef = abs(val),
        ))
    end

    sort!(rows, by = r -> (r.time_slot, r.field, r.offset))
    return rows
end

function inspect_operator_coefficients_at_point(numOps, point; which=:residual, iExpr=1, atol=0.0)
    op = _get_numop(numOps, which)
    rows = coefficient_rows_at_point(op, point; iExpr=iExpr, atol=atol)
    println("operator = ", which)
    println("point = ", point, ", iExpr = ", iExpr, ", ncoef = ", length(rows))
    for r in rows
        println(
            "t=", r.time_slot,
            " (", r.time_role, ")",
            " field=", r.field,
            " offset=", r.offset,
            " neighbour=", r.neighbour,
            " coef=", r.coef,
        )
    end
    return rows
end

function coefficient_table_by_time(rows)
    slots = sort(unique(getproperty.(rows, :time_slot)))
    return [(
        time_slot=s,
        time_role=first(r.time_role for r in rows if r.time_slot == s),
        n=length(filter(r -> r.time_slot == s, rows)),
        sumcoef=sum(r.coef for r in rows if r.time_slot == s),
        sumabs=sum(r.abscoef for r in rows if r.time_slot == s),
    ) for s in slots]
end

function coefficient_rows_for_role(rows, role::Symbol)
    return filter(r -> r.time_role == role, rows)
end


function source_time_samples(Nt, Δt, timePointsUsedForOneStep; wavelet=nothing, t0=12Δt, f0=0.04)
    ntime = Nt + timePointsUsedForOneStep - 1
    t = (0:ntime-1) .* Δt
    if wavelet === nothing
        return Ricker.(t, t0, f0)
    else
        return wavelet.(t)
    end
end

function _normalise_source_weights!(w; normalise=:none)
    if normalise == :none || normalise === false
        return w
    elseif normalise == :sum
        s = sum(w)
        !iszero(s) && (w ./= s)
    elseif normalise == :l1
        s = sum(abs, w)
        !iszero(s) && (w ./= s)
    elseif normalise == :max
        s = maximum(abs, w)
        !iszero(s) && (w ./= s)
    else
        error("normalise should be :none, :sum, :l1, or :max")
    end
    return w
end

function make_sourceFull(preparedLin, spatialWeights, timeSignal; iField=1, amplitude=1.0)
    length(spatialWeights) == preparedLin.NforcePoints ||
        error("spatialWeights should have length preparedLin.NforcePoints=$(preparedLin.NforcePoints)")
    NForceField = hasproperty(preparedLin, :NForceField) ? preparedLin.NForceField : preparedLin.NField
    1 <= iField <= NForceField || error("iField should be between 1 and $(NForceField)")

    T = promote_type(eltype(spatialWeights), eltype(timeSignal), typeof(amplitude))
    sourceFull = zeros(T, preparedLin.NforcePoints, NForceField, length(timeSignal))
    for it in eachindex(timeSignal)
        sourceFull[:, iField, it] .= amplitude .* timeSignal[it] .* spatialWeights
    end
    return sourceFull
end

function point_source_weights(preparedLin, point::CartesianIndex; normalise=:none)
    weights = zeros(Float64, preparedLin.NforcePoints)
    LI = LinearIndices(preparedLin.spaceShape)
    weights[LI[point]] = 1.0
    return _normalise_source_weights!(weights; normalise)
end

function hyperplane_source_weights(preparedLin; fixed=Dict{Int,Int}(), profile=(I -> 1.0), normalise=:sum)
    weights = zeros(Float64, preparedLin.NforcePoints)
    LI = LinearIndices(preparedLin.spaceShape)
    for I in CartesianIndices(preparedLin.spaceShape)
        coords = Tuple(I)
        inside = all(coords[d] == v for (d, v) in fixed)
        if inside
            weights[LI[I]] = profile(I)
        end
    end
    return _normalise_source_weights!(weights; normalise)
end

function mask_source_weights(preparedLin, mask; profile=(I -> 1.0), normalise=:sum)
    weights = zeros(Float64, preparedLin.NforcePoints)
    LI = LinearIndices(preparedLin.spaceShape)

    if mask isa AbstractArray{Bool}
        size(mask) == preparedLin.spaceShape || error("Boolean mask size should be $(preparedLin.spaceShape)")
        for I in CartesianIndices(mask)
            mask[I] && (weights[LI[I]] = profile(I))
        end
    else
        for I in mask
            weights[LI[I]] = profile(I)
        end
    end

    return _normalise_source_weights!(weights; normalise)
end


using LinearAlgebra, SparseArrays
using CairoMakie
CairoMakie.activate!()

function apply_forced_boundary_for_video!(A, b; leftValue=0.0, rightValue=0.0)
    A[1, :] .= 0
    A[1, 1] = 1
    b[1] = leftValue

    A[end, :] .= 0
    A[end, end] = 1
    b[end] = rightValue
    return A, b
end

function propagate_linear_frames(
    preparedLin,
    sourceFull,
    Nt;
    initialCondition=0.0,
    boundaryConditionForced=false,
    store_every=1,
    blowup_limit=Inf,
)
    NField = preparedLin.NField
    NForceField = hasproperty(preparedLin, :NForceField) ? preparedLin.NForceField : preparedLin.NField
    NpointsSpace = preparedLin.NpointsSpace
    timePointsUsedForOneStep = preparedLin.timePointsUsedForOneStep
    NknownTime = max(timePointsUsedForOneStep - 1, 0)

    size(sourceFull, 1) == preparedLin.NforcePoints || error("sourceFull first dimension mismatch")
    size(sourceFull, 2) == NForceField || error("sourceFull second dimension mismatch")
    size(sourceFull, 3) >= Nt + timePointsUsedForOneStep - 1 || error("sourceFull has too few time samples")

    knownField = fill(Float64(initialCondition), size(preparedLin.known_lhs_template))
    knownForce = fill(Float64(initialCondition), size(preparedLin.known_rhs_template))
    unknownField = fill(Float64(initialCondition), NpointsSpace, NField)

    A = sparse(preparedLin.A_template)
    b0 = copy(preparedLin.b_template)
    if boundaryConditionForced
        apply_forced_boundary_for_video!(A, b0; leftValue=0.0, rightValue=0.0)
    end
    factor = lu(A)
    b = copy(preparedLin.b_template)

    frames = Vector{Array{Float64}}()
    for it in 1:Nt
        knownForce .= sourceFull[:, :, it:it+timePointsUsedForOneStep-1]
        knownInputs = vcat(vec(knownField), vec(knownForce))
        preparedLin.b_fun!(b, knownInputs)
        if boundaryConditionForced
            b[1] = 0.0
            b[end] = 0.0
        end

        u = factor \ b
        unknownField .= reshape(real.(u), NpointsSpace, NField)

        umax = maximum(abs, unknownField)
        if !isfinite(umax) || umax > blowup_limit
            @warn "Stopping propagation because the wavefield blew up" it umax blowup_limit
            push!(frames, reshape(copy(unknownField[:, 1]), preparedLin.spaceShape...))
            break
        end

        if it == 1 || it % store_every == 0 || it == Nt
            push!(frames, reshape(copy(unknownField[:, 1]), preparedLin.spaceShape...))
        end

        if NknownTime > 0
            if NknownTime > 1
                knownField[:, :, 1:end-1] .= knownField[:, :, 2:end]
            end
            knownField[:, :, end] .= unknownField
        end
    end
    return frames
end

function _finite_copy(A; fillvalue=0.0)
    B = Array{Float64}(undef, size(A))
    replaced = 0
    for I in eachindex(A)
        v = Float64(real(A[I]))
        if isfinite(v)
            B[I] = v
        else
            B[I] = fillvalue
            replaced += 1
        end
    end
    return B, replaced
end

function finite_frame_diagnostics(frames)
    total_bad = 0
    first_bad = nothing
    finite_max = 0.0
    for (i, frame) in pairs(frames)
        bad_here = count(!isfinite, frame)
        if bad_here > 0 && first_bad === nothing
            first_bad = i
        end
        total_bad += bad_here
        for v in frame
            if isfinite(v)
                finite_max = max(finite_max, abs(Float64(v)))
            end
        end
    end
    return (total_bad=total_bad, first_bad=first_bad, finite_max=finite_max)
end

function _background_for_plot(background, spaceShape)
    background === nothing && return nothing
    bg = Array(background)
    if size(bg) == spaceShape
        clean_bg, replaced = _finite_copy(bg)
    elseif length(bg) == prod(spaceShape)
        clean_bg, replaced = _finite_copy(reshape(bg, spaceShape...))
    else
        @warn "background size $(size(bg)) does not match wavefield spaceShape $spaceShape; ignoring background"
        return nothing
    end
    replaced > 0 && @warn "background had $replaced non-finite values; replaced by 0 for plotting"
    return clean_bg
end

function record_wave_video(
    frames;
    videoFile=joinpath(@__DIR__, "point_source_propagation.mp4"),
    background=nothing,
    sourcePoint=nothing,
    framerate=20,
    title="point source propagation",
    colormap=:balance,
    background_colormap=:grays,
    wave_alpha=0.72,
    clim=nothing,
)
    isempty(frames) && error("frames is empty")
    spaceShape = size(frames[1])
    bg = _background_for_plot(background, spaceShape)
    nd = length(spaceShape)

    diag = finite_frame_diagnostics(frames)
    if diag.total_bad > 0
        @warn "wavefield frames contain $(diag.total_bad) non-finite values; first affected stored frame is $(diag.first_bad). Replacing by 0 for plotting."
    end
    clean_frames = [first(_finite_copy(f)) for f in frames]

    if clim === nothing
        vmax = diag.finite_max
        clim = vmax == 0 ? 1.0 : vmax
    end

    fig = Figure(size=(900, 680))
    ax = Axis(fig[1, 1], aspect=DataAspect(), title=title)

    if nd == 1
        x = 1:spaceShape[1]
        yobs = Observable(vec(clean_frames[1]))
        lines!(ax, x, yobs, color=:dodgerblue3, linewidth=2)
        ylims!(ax, -clim, clim)
        if sourcePoint !== nothing
            vlines!(ax, [Tuple(sourcePoint)[1]], color=:red, linewidth=2)
        end
        record(fig, videoFile, eachindex(frames); framerate=framerate) do iframe
            yobs[] = vec(clean_frames[iframe])
            ax.title = "$title  |  frame $iframe/$(length(frames))"
        end
    elseif nd == 2
        if bg !== nothing
            heatmap!(ax, bg; colormap=background_colormap)
        end
        frame_obs = Observable(clean_frames[1])
        hm = heatmap!(ax, frame_obs; colormap=colormap, colorrange=(-clim, clim), alpha=bg === nothing ? 1.0 : wave_alpha)
        Colorbar(fig[1, 2], hm, label="wavefield")
        if sourcePoint !== nothing
            xy = Tuple(sourcePoint)
            scatter!(ax, [xy[1]], [xy[2]], color=:red, markersize=14)
        end
        record(fig, videoFile, eachindex(frames); framerate=framerate) do iframe
            frame_obs[] = clean_frames[iframe]
            ax.title = "$title  |  frame $iframe/$(length(frames))"
        end
    else
        error("Video helper currently supports 1D/2D frames. For 3D/4D, select a slice before recording.")
    end

    return videoFile
end


function _finite_plot_matrix(A; fillvalue=0.0)
    B = Array{Float64}(undef, size(A))
    nbad = 0
    finite_max = 0.0
    for I in eachindex(A)
        v = Float64(real(A[I]))
        if isfinite(v)
            B[I] = v
            finite_max = max(finite_max, abs(v))
        else
            B[I] = fillvalue
            nbad += 1
        end
    end
    return B, nbad, finite_max
end

function wavefield_snapshot_report(frames)
    rows = NamedTuple[]
    for (i, frame) in pairs(frames)
        _, nbad, finite_max = _finite_plot_matrix(frame)
        push!(rows, (frame=i, nbad=nbad, finite_max=finite_max, minimum=minimum(skipmissing(vec(ifelse.(isfinite.(frame), frame, missing)))), maximum=maximum(skipmissing(vec(ifelse.(isfinite.(frame), frame, missing))))))
    end
    return rows
end


function _snapshot_background_for_plot(background, spaceShape)
    background === nothing && return nothing
    bg = Array(background)
    if size(bg) == spaceShape
        B, replaced, _ = _finite_plot_matrix(bg)
    elseif length(bg) == prod(spaceShape)
        B, replaced, _ = _finite_plot_matrix(reshape(bg, spaceShape...))
    else
        @warn "background size $(size(bg)) does not match wavefield spaceShape $spaceShape; ignoring background"
        return nothing
    end
    replaced > 0 && @warn "background had $replaced non-finite values; replaced by 0 for plotting"
    return B
end

function plot_wave_snapshots(
    frames;
    indices=nothing,
    background=nothing,
    sourcePoint=nothing,
    clim=nothing,
    ncols=3,
    title="wavefield snapshots",
    colormap=:balance,
    background_colormap=:grays,
    wave_alpha=0.72,
)
    isempty(frames) && error("frames is empty")
    if indices === nothing
        indices = unique(round.(Int, range(1, length(frames), length=min(9, length(frames)))))
    end
    indices = collect(indices)

    clean_frames = Array{Float64}[]
    nbad = Int[]
    finite_maxima = Float64[]
    for i in indices
        B, bad, fmax = _finite_plot_matrix(frames[i])
        push!(clean_frames, B)
        push!(nbad, bad)
        push!(finite_maxima, fmax)
    end

    if clim === nothing
        vmax = maximum(finite_maxima)
        clim = vmax == 0 ? 1.0 : vmax
    end

    bg = _snapshot_background_for_plot(background, size(frames[1]))
    n = length(indices)
    nrows = cld(n, ncols)
    fig = Figure(size=(320*ncols, 300*nrows))

    for (k, iframe) in enumerate(indices)
        r = cld(k, ncols)
        c = k - (r - 1) * ncols
        ax = Axis(fig[r, c], aspect=DataAspect(), title="frame $iframe | bad=$(nbad[k])")
        if length(size(frames[1])) == 1
            lines!(ax, vec(clean_frames[k]), color=:dodgerblue3, linewidth=2)
            ylims!(ax, -clim, clim)
            if sourcePoint !== nothing
                vlines!(ax, [Tuple(sourcePoint)[1]], color=:red, linewidth=2)
            end
        elseif length(size(frames[1])) == 2
            if bg !== nothing
                heatmap!(ax, bg; colormap=background_colormap)
            end
            heatmap!(ax, clean_frames[k]; colormap=colormap, colorrange=(-clim, clim), alpha=bg === nothing ? 1.0 : wave_alpha)
            if sourcePoint !== nothing
                xy = Tuple(sourcePoint)
                scatter!(ax, [xy[1]], [xy[2]], color=:red, markersize=10)
            end
        else
            error("Snapshot helper supports 1D/2D frames. Select a slice first for higher dimensions.")
        end
    end

    Label(fig[0, :], title)
    return fig
end


function save_wave_snapshot_pdf(
    frames;
    filename=joinpath(@__DIR__, "wave_snapshots.pdf"),
    indices=nothing,
    background=nothing,
    sourcePoint=nothing,
    clim=nothing,
    ncols=3,
    title="wavefield snapshots",
    kwargs...,
)
    fig = plot_wave_snapshots(
        frames;
        indices=indices,
        background=background,
        sourcePoint=sourcePoint,
        clim=clim,
        ncols=ncols,
        title=title,
        kwargs...,
    )
    save(filename, fig)
    empty!(fig.scene)  # release most displayed scene resources
    GC.gc()
    return filename
end


using Statistics

function cfl_diagnostics(model_velocity, Δnum; cfl_safety=0.45, ppw=10, samples_per_period=20)
    v = vec(Float64.(model_velocity))
    finite_v = filter(isfinite, v)
    dt = Float64(Δnum[end])
    dx = Float64.(Δnum[1:end-1])
    dxmin = minimum(dx)
    dxmax = maximum(dx)
    vmax = maximum(abs, finite_v)
    vmin = minimum(abs, filter(!iszero, finite_v))
    cfl = vmax * dt / dxmin

    # MKS units: velocity [m/s], dx [m], dt [s], frequency [Hz].
    # For a Ricker wavelet in flexOPT, f0 is the dominant/peak frequency.
    fmax_spatial = vmin / (ppw * dxmax)
    fmax_temporal = 1 / (samples_per_period * dt)
    suggested_ricker_f0 = min(fmax_spatial, fmax_temporal)

    return (
        v_min=minimum(finite_v),
        v_median=median(finite_v),
        v_max=maximum(finite_v),
        dt=dt,
        dxmin=dxmin,
        dxmax=dxmax,
        cfl_max=cfl,
        suggested_dt_2D=cfl_safety * dxmin / vmax,
        ppw=ppw,
        samples_per_period=samples_per_period,
        fmax_spatial=fmax_spatial,
        fmax_temporal=fmax_temporal,
        suggested_ricker_f0=suggested_ricker_f0,
        suggested_ricker_t0=1.5 / suggested_ricker_f0,
    )
end

function ricker_time_signal(Nt, Δt; f0, t0=1.5/f0, timePointsUsedForOneStep=1)
    ntime = Nt + timePointsUsedForOneStep - 1
    t = (0:ntime-1) .* Δt
    return t, Ricker.(t, t0, f0)
end

function dominant_frequency_scan(signal, Δt; nfreq=512)
    y = Float64.(signal) .- mean(Float64.(signal))
    n = length(y)
    duration = n * Δt
    fmin = 1 / duration
    fmax = 0.5 / Δt
    freqs = range(fmin, fmax; length=nfreq)
    t = (0:n-1) .* Δt
    spectrum = [abs(sum(y .* exp.(-2im * π * f .* t))) for f in freqs]
    imax = argmax(spectrum)
    return (f_peak=freqs[imax], peak_amplitude=spectrum[imax], frequencies=freqs, spectrum=spectrum)
end

function ricker_diagnostics(model_velocity, Δnum; f0=nothing, Nt=200, timePointsUsedForOneStep=1, ppw=10, samples_per_period=20, t0=nothing)
    cfl = cfl_diagnostics(model_velocity, Δnum; ppw=ppw, samples_per_period=samples_per_period)
    f0_used = f0 === nothing ? cfl.suggested_ricker_f0 : Float64(f0)
    t0_used = t0 === nothing ? 1.5 / f0_used : Float64(t0)
    t, signal = ricker_time_signal(Nt, cfl.dt; f0=f0_used, t0=t0_used, timePointsUsedForOneStep=timePointsUsedForOneStep)
    freqinfo = dominant_frequency_scan(signal, cfl.dt)
    return (
        f0=f0_used,
        t0=t0_used,
        duration=t[end],
        dominant_frequency_scan=freqinfo.f_peak,
        wavelength_min_velocity=cfl.v_min / f0_used,
        points_per_wavelength_min_velocity=(cfl.v_min / f0_used) / cfl.dxmax,
        samples_per_period=1 / (f0_used * cfl.dt),
        max_abs=maximum(abs, signal),
        first_peak_time=t[argmax(abs.(signal))],
        cfl=cfl,
    )
end

function frame_motion_report(frames)
    rows = NamedTuple[]
    previous = nothing
    for (i, frame) in pairs(frames)
        _, nbad, fmax = _finite_plot_matrix(frame)
        dnorm = previous === nothing ? 0.0 : norm(frame .- previous)
        dmax = previous === nothing ? 0.0 : maximum(abs, frame .- previous)
        push!(rows, (frame=i, nbad=nbad, maxabs=fmax, norm=norm(frame), diff_norm=dnorm, diff_max=dmax))
        previous = frame
    end
    return rows
end

function difference_frames(frames)
    length(frames) >= 2 || error("need at least two frames")
    return [frames[i] .- frames[i-1] for i in 2:length(frames)]
end


function _sparse_norms(A)
    return (
        size=size(A),
        nnz=nnz(A),
        maxabs=nnz(A) == 0 ? 0.0 : maximum(abs, nonzeros(A)),
        norm=norm(A),
    )
end

function diagnose_prepared_linear_system(preparedLin)
    println("spaceShape = ", preparedLin.spaceShape)
    println("NpointsSpace = ", preparedLin.NpointsSpace)
    println("NField = ", preparedLin.NField)
    println("NForceField = ", hasproperty(preparedLin, :NForceField) ? preparedLin.NForceField : preparedLin.NField)
    println("timePointsUsedForOneStep = ", preparedLin.timePointsUsedForOneStep)
    println("A_unknown: ", _sparse_norms(preparedLin.A_unknown))
    println("L_known:   ", _sparse_norms(preparedLin.L_known))
    println("R_force:   ", _sparse_norms(preparedLin.R_force))
    if hasproperty(preparedLin, :right_operator)
        println("right_operator.size = ", preparedLin.right_operator.size)
        println("right_operator.nnz = ", length(preparedLin.right_operator.table.vals))
    end
    row_nnz_A = vec(sum(abs.(preparedLin.A_unknown) .> 0; dims=2))
    row_nnz_R = vec(sum(abs.(preparedLin.R_force) .> 0; dims=2))
    println("rows with A entries = ", count(!iszero, row_nnz_A), " / ", length(row_nnz_A))
    println("rows with R entries = ", count(!iszero, row_nnz_R), " / ", length(row_nnz_R))
    return nothing
end

function source_field_response(preparedLin, sourcePoint, timeSignal; amplitude=1.0, it=nothing)
    LI = LinearIndices(preparedLin.spaceShape)
    src_linear = LI[sourcePoint]
    last_start = length(timeSignal) - preparedLin.timePointsUsedForOneStep + 1
    last_start >= 1 || error("timeSignal is too short for timePointsUsedForOneStep=$(preparedLin.timePointsUsedForOneStep)")
    if it === nothing
        it = min(argmax(abs.(timeSignal)), last_start)
    else
        1 <= it <= last_start || error("it should be in 1:$last_start for this timeSignal")
    end

    out = NamedTuple[]
    NForceField = hasproperty(preparedLin, :NForceField) ? preparedLin.NForceField : preparedLin.NField
    for iField in 1:NForceField
        w = zeros(Float64, preparedLin.NforcePoints)
        w[src_linear] = 1.0
        sourceFull = make_sourceFull(preparedLin, w, timeSignal; iField=iField, amplitude=amplitude)
        knownForce = sourceFull[:, :, it:it+preparedLin.timePointsUsedForOneStep-1]
        knownInputs = vcat(vec(zero(preparedLin.known_lhs_template)), vec(knownForce))
        b = copy(preparedLin.b_template)
        preparedLin.b_fun!(b, knownInputs)
        push!(out, (
            iField=iField,
            time_index=it,
            source_norm=norm(knownForce),
            b_norm=norm(b),
            b_maximum=maximum(abs, b),
            b_nonzero=count(!iszero, b),
        ))
    end
    return out
end

function one_step_response(preparedLin, sourceFull; it=1, boundaryConditionForced=false)
    knownField = zero(preparedLin.known_lhs_template)
    knownForce = sourceFull[:, :, it:it+preparedLin.timePointsUsedForOneStep-1]
    knownInputs = vcat(vec(knownField), vec(knownForce))
    b = copy(preparedLin.b_template)
    preparedLin.b_fun!(b, knownInputs)

    A = sparse(preparedLin.A_template)
    if boundaryConditionForced
        apply_forced_boundary_for_video!(A, b; leftValue=0.0, rightValue=0.0)
    end
    u = A \ b
    ufield = reshape(real.(u), preparedLin.NpointsSpace, preparedLin.NField)
    return (
        b_norm=norm(b),
        b_maximum=maximum(abs, b),
        b_nonzero=count(!iszero, b),
        u_norm=norm(u),
        u_maximum=maximum(abs, u),
        u_nonzero=count(!iszero, u),
        ufield=ufield,
    )
end


function _matrix_row_abs_sums(A)
    return vec(sum(abs.(A); dims=2))
end

function operator_growth_diagnostics(preparedLin)
    A = sparse(preparedLin.A_unknown)
    L = sparse(preparedLin.L_known)
    R = sparse(preparedLin.R_force)
    diagA = abs.(diag(A))
    rowA = _matrix_row_abs_sums(A)
    offdiagA = rowA .- diagA
    rowL = _matrix_row_abs_sums(L)
    rowR = _matrix_row_abs_sums(R)
    return (
        A_size=size(A),
        A_nnz=nnz(A),
        A_diag_min=isempty(diagA) ? 0.0 : minimum(diagA),
        A_diag_median=isempty(diagA) ? 0.0 : median(diagA),
        A_diag_max=isempty(diagA) ? 0.0 : maximum(diagA),
        A_offdiag_over_diag_max=maximum(offdiagA ./ max.(diagA, eps(Float64))),
        L_row_sum_max=maximum(rowL),
        R_row_sum_max=maximum(rowR),
        R_over_A_diag_max=maximum(rowR ./ max.(diagA, eps(Float64))),
        condest_dense_sample=size(A,1) <= 5000 ? cond(Matrix(A)) : missing,
    )
end

function homogeneous_zero_test(preparedLin; Nt=20, store_every=1)
    sourceFull_zero = zeros(eltype(preparedLin.known_rhs_template), preparedLin.NforcePoints, preparedLin.NForceField, Nt + preparedLin.timePointsUsedForOneStep - 1)
    frames_zero = propagate_linear_frames(preparedLin, sourceFull_zero, Nt; store_every=store_every, blowup_limit=1e12)
    return wavefield_snapshot_report(frames_zero)
end

function source_amplitude_sweep(preparedLin, sourcePoint, timeSignal; amplitudes=(1e-12, 1e-10, 1e-8, 1e-6), iField=1, Nt=20)
    rows = NamedTuple[]
    w = point_source_weights(preparedLin, sourcePoint)
    for amp in amplitudes
        sourceFull = make_sourceFull(preparedLin, w, timeSignal; iField=iField, amplitude=amp)
        frames = propagate_linear_frames(preparedLin, sourceFull, Nt; store_every=max(1, Nt ÷ 5), blowup_limit=1e30)
        report = wavefield_snapshot_report(frames)
        push!(rows, (amplitude=amp, last=report[end], nframes=length(frames)))
    end
    return rows
end


function _distance_from_source_table(frame, sourcePoint; threshold_fraction=1e-6)
    f = abs.(Float64.(frame))
    fmax = maximum(f)
    threshold = threshold_fraction * max(fmax, eps(Float64))
    active = findall(>(threshold), f)
    if isempty(active)
        return (nactive=0, max_distance=0.0, mean_distance=0.0, fmax=fmax)
    end
    distances = [sqrt(sum((Tuple(I) .- Tuple(sourcePoint)).^2)) for I in active]
    return (
        nactive=length(active),
        max_distance=maximum(distances),
        mean_distance=mean(distances),
        fmax=fmax,
    )
end

function propagation_radius_report(frames, sourcePoint; threshold_fraction=1e-6)
    return [(frame=i, _distance_from_source_table(frame, sourcePoint; threshold_fraction=threshold_fraction)...) for (i, frame) in pairs(frames)]
end

function impulse_sourceFull(preparedLin, sourcePoint; iField=1, amplitude=1.0, Nt=20, impulse_time=1)
    timeSignal = zeros(Float64, Nt + preparedLin.timePointsUsedForOneStep - 1)
    timeSignal[impulse_time] = 1.0
    w = point_source_weights(preparedLin, sourcePoint)
    return make_sourceFull(preparedLin, w, timeSignal; iField=iField, amplitude=amplitude)
end

function impulse_propagation_test(preparedLin, sourcePoint; iField=1, amplitude=1.0, Nt=20, store_every=1, threshold_fraction=1e-8)
    sourceFull = impulse_sourceFull(preparedLin, sourcePoint; iField=iField, amplitude=amplitude, Nt=Nt)
    frames = propagate_linear_frames(preparedLin, sourceFull, Nt; store_every=store_every, blowup_limit=1e20)
    return (
        frames=frames,
        snapshots=wavefield_snapshot_report(frames),
        radius=propagation_radius_report(frames, sourcePoint; threshold_fraction=threshold_fraction),
    )
end


function frame_extrema_report(frames, sourcePoint)
    rows = NamedTuple[]
    for (i, frame) in pairs(frames)
        A = Float64.(frame)
        absA = abs.(A)
        imax = argmax(absA)
        maxpoint = CartesianIndices(size(A))[imax]
        distance = sqrt(sum((Tuple(maxpoint) .- Tuple(sourcePoint)).^2))
        push!(rows, (
            frame=i,
            maxpoint=maxpoint,
            distance_from_source=distance,
            value=A[imax],
            maxabs=absA[imax],
            minimum=minimum(A),
            maximum=maximum(A),
        ))
    end
    return rows
end


function _get_numop(numOps, which::Symbol)
    ops = numOps isa AbstractDict ? numOps["numericalOperators"] : numOps.numericalOperators
    return getproperty(ops, which)
end

function _point_linear_index(op, point)
    νWhole = op.geometry.νWhole
    if point isa Integer
        1 <= point <= length(νWhole) || error("point integer should be in 1:$(length(νWhole))")
        return Int(point), νWhole[Int(point)]
    elseif point isa CartesianIndex
        point in νWhole || error("point $point is not in op.geometry.νWhole")
        pointLinear = LinearIndices(νWhole)[point]
        return Int(pointLinear), point
    else
        error("point should be an Int linear point index or a CartesianIndex")
    end
end

function _time_role(iT, activeTimePoints)
    if iT == activeTimePoints
        return :future_unknown
    elseif iT == activeTimePoints - 1
        return :present_known
    else
        return Symbol("past_known_tminus$(activeTimePoints - iT)")
    end
end

function coefficient_rows_at_point(op, point; iExpr=1, atol=0.0)
    pointLinear, pointCI = _point_linear_index(op, point)
    nPoints = length(op.geometry.νWhole)
    activeTimePoints = op.geometry.activeTimePoints
    spaceShape = Tuple(op.geometry.wholeRegionPointsSpace)
    Nexpr = op.size[1] ÷ nPoints
    Nfield = op.size[2] ÷ (nPoints * activeTimePoints)

    1 <= iExpr <= Nexpr || error("iExpr should be in 1:$Nexpr")

    residualLI = LinearIndices((Nexpr, nPoints))
    targetRow = residualLI[iExpr, pointLinear]
    fieldTimeSpaceCI = CartesianIndices((Nfield, activeTimePoints, spaceShape...))

    rows = NamedTuple[]
    for k in eachindex(op.table.vals)
        Int(op.table.rows[k]) == targetRow || continue
        val = op.table.vals[k]
        abs(val) > atol || continue

        ci = fieldTimeSpaceCI[Int(op.table.cols[k])]
        iField = ci[1]
        iT = ci[2]
        neighbour = CartesianIndex(Tuple(ci)[3:end])
        offset = Tuple(neighbour - pointCI)
        timeRole = _time_role(iT, activeTimePoints)

        push!(rows, (
            k = k,
            residual_row = targetRow,
            expr = iExpr,
            point = pointCI,
            field = iField,
            time_slot = iT,
            time_role = timeRole,
            neighbour = neighbour,
            offset = offset,
            coef = val,
            abscoef = abs(val),
        ))
    end

    sort!(rows, by = r -> (r.time_slot, r.field, r.offset))
    return rows
end

function inspect_operator_coefficients_at_point(numOps, point; which=:residual, iExpr=1, atol=0.0)
    op = _get_numop(numOps, which)
    rows = coefficient_rows_at_point(op, point; iExpr=iExpr, atol=atol)
    println("operator = ", which)
    println("point = ", point, ", iExpr = ", iExpr, ", ncoef = ", length(rows))
    for r in rows
        println(
            "t=", r.time_slot,
            " (", r.time_role, ")",
            " field=", r.field,
            " offset=", r.offset,
            " neighbour=", r.neighbour,
            " coef=", r.coef,
        )
    end
    return rows
end

function coefficient_table_by_time(rows)
    slots = sort(unique(getproperty.(rows, :time_slot)))
    return [(
        time_slot=s,
        time_role=first(r.time_role for r in rows if r.time_slot == s),
        n=length(filter(r -> r.time_slot == s, rows)),
        sumcoef=sum(r.coef for r in rows if r.time_slot == s),
        sumabs=sum(r.abscoef for r in rows if r.time_slot == s),
    ) for s in slots]
end

function coefficient_rows_for_role(rows, role::Symbol)
    return filter(r -> r.time_role == role, rows)
end


function source_time_samples(Nt, Δt, timePointsUsedForOneStep; wavelet=nothing, t0=12Δt, f0=0.04)
    ntime = Nt + timePointsUsedForOneStep - 1
    t = (0:ntime-1) .* Δt
    if wavelet === nothing
        return Ricker.(t, t0, f0)
    else
        return wavelet.(t)
    end
end

function _normalise_source_weights!(w; normalise=:none)
    if normalise == :none || normalise === false
        return w
    elseif normalise == :sum
        s = sum(w)
        !iszero(s) && (w ./= s)
    elseif normalise == :l1
        s = sum(abs, w)
        !iszero(s) && (w ./= s)
    elseif normalise == :max
        s = maximum(abs, w)
        !iszero(s) && (w ./= s)
    else
        error("normalise should be :none, :sum, :l1, or :max")
    end
    return w
end

function make_sourceFull(preparedLin, spatialWeights, timeSignal; iField=1, amplitude=1.0)
    length(spatialWeights) == preparedLin.NforcePoints ||
        error("spatialWeights should have length preparedLin.NforcePoints=$(preparedLin.NforcePoints)")
    NForceField = hasproperty(preparedLin, :NForceField) ? preparedLin.NForceField : preparedLin.NField
    1 <= iField <= NForceField || error("iField should be between 1 and $(NForceField)")

    T = promote_type(eltype(spatialWeights), eltype(timeSignal), typeof(amplitude))
    sourceFull = zeros(T, preparedLin.NforcePoints, NForceField, length(timeSignal))
    for it in eachindex(timeSignal)
        sourceFull[:, iField, it] .= amplitude .* timeSignal[it] .* spatialWeights
    end
    return sourceFull
end

function point_source_weights(preparedLin, point::CartesianIndex; normalise=:none)
    weights = zeros(Float64, preparedLin.NforcePoints)
    LI = LinearIndices(preparedLin.spaceShape)
    weights[LI[point]] = 1.0
    return _normalise_source_weights!(weights; normalise)
end

function hyperplane_source_weights(preparedLin; fixed=Dict{Int,Int}(), profile=(I -> 1.0), normalise=:sum)
    weights = zeros(Float64, preparedLin.NforcePoints)
    LI = LinearIndices(preparedLin.spaceShape)
    for I in CartesianIndices(preparedLin.spaceShape)
        coords = Tuple(I)
        inside = all(coords[d] == v for (d, v) in fixed)
        if inside
            weights[LI[I]] = profile(I)
        end
    end
    return _normalise_source_weights!(weights; normalise)
end

function mask_source_weights(preparedLin, mask; profile=(I -> 1.0), normalise=:sum)
    weights = zeros(Float64, preparedLin.NforcePoints)
    LI = LinearIndices(preparedLin.spaceShape)

    if mask isa AbstractArray{Bool}
        size(mask) == preparedLin.spaceShape || error("Boolean mask size should be $(preparedLin.spaceShape)")
        for I in CartesianIndices(mask)
            mask[I] && (weights[LI[I]] = profile(I))
        end
    else
        for I in mask
            weights[LI[I]] = profile(I)
        end
    end

    return _normalise_source_weights!(weights; normalise)
end


using LinearAlgebra, SparseArrays
using CairoMakie
CairoMakie.activate!()

function apply_forced_boundary_for_video!(A, b; leftValue=0.0, rightValue=0.0)
    A[1, :] .= 0
    A[1, 1] = 1
    b[1] = leftValue

    A[end, :] .= 0
    A[end, end] = 1
    b[end] = rightValue
    return A, b
end

function propagate_linear_frames(
    preparedLin,
    sourceFull,
    Nt;
    initialCondition=0.0,
    boundaryConditionForced=false,
    store_every=1,
    blowup_limit=Inf,
)
    NField = preparedLin.NField
    NForceField = hasproperty(preparedLin, :NForceField) ? preparedLin.NForceField : preparedLin.NField
    NpointsSpace = preparedLin.NpointsSpace
    timePointsUsedForOneStep = preparedLin.timePointsUsedForOneStep
    NknownTime = max(timePointsUsedForOneStep - 1, 0)

    size(sourceFull, 1) == preparedLin.NforcePoints || error("sourceFull first dimension mismatch")
    size(sourceFull, 2) == NForceField || error("sourceFull second dimension mismatch")
    size(sourceFull, 3) >= Nt + timePointsUsedForOneStep - 1 || error("sourceFull has too few time samples")

    knownField = fill(Float64(initialCondition), size(preparedLin.known_lhs_template))
    knownForce = fill(Float64(initialCondition), size(preparedLin.known_rhs_template))
    unknownField = fill(Float64(initialCondition), NpointsSpace, NField)

    A = sparse(preparedLin.A_template)
    b0 = copy(preparedLin.b_template)
    if boundaryConditionForced
        apply_forced_boundary_for_video!(A, b0; leftValue=0.0, rightValue=0.0)
    end
    factor = lu(A)
    b = copy(preparedLin.b_template)

    frames = Vector{Array{Float64}}()
    for it in 1:Nt
        knownForce .= sourceFull[:, :, it:it+timePointsUsedForOneStep-1]
        knownInputs = vcat(vec(knownField), vec(knownForce))
        preparedLin.b_fun!(b, knownInputs)
        if boundaryConditionForced
            b[1] = 0.0
            b[end] = 0.0
        end

        u = factor \ b
        unknownField .= reshape(real.(u), NpointsSpace, NField)

        umax = maximum(abs, unknownField)
        if !isfinite(umax) || umax > blowup_limit
            @warn "Stopping propagation because the wavefield blew up" it umax blowup_limit
            push!(frames, reshape(copy(unknownField[:, 1]), preparedLin.spaceShape...))
            break
        end

        if it == 1 || it % store_every == 0 || it == Nt
            push!(frames, reshape(copy(unknownField[:, 1]), preparedLin.spaceShape...))
        end

        if NknownTime > 0
            if NknownTime > 1
                knownField[:, :, 1:end-1] .= knownField[:, :, 2:end]
            end
            knownField[:, :, end] .= unknownField
        end
    end
    return frames
end

function _finite_copy(A; fillvalue=0.0)
    B = Array{Float64}(undef, size(A))
    replaced = 0
    for I in eachindex(A)
        v = Float64(real(A[I]))
        if isfinite(v)
            B[I] = v
        else
            B[I] = fillvalue
            replaced += 1
        end
    end
    return B, replaced
end

function finite_frame_diagnostics(frames)
    total_bad = 0
    first_bad = nothing
    finite_max = 0.0
    for (i, frame) in pairs(frames)
        bad_here = count(!isfinite, frame)
        if bad_here > 0 && first_bad === nothing
            first_bad = i
        end
        total_bad += bad_here
        for v in frame
            if isfinite(v)
                finite_max = max(finite_max, abs(Float64(v)))
            end
        end
    end
    return (total_bad=total_bad, first_bad=first_bad, finite_max=finite_max)
end

function _background_for_plot(background, spaceShape)
    background === nothing && return nothing
    bg = Array(background)
    if size(bg) == spaceShape
        clean_bg, replaced = _finite_copy(bg)
    elseif length(bg) == prod(spaceShape)
        clean_bg, replaced = _finite_copy(reshape(bg, spaceShape...))
    else
        @warn "background size $(size(bg)) does not match wavefield spaceShape $spaceShape; ignoring background"
        return nothing
    end
    replaced > 0 && @warn "background had $replaced non-finite values; replaced by 0 for plotting"
    return clean_bg
end

function record_wave_video(
    frames;
    videoFile=joinpath(@__DIR__, "point_source_propagation.mp4"),
    background=nothing,
    sourcePoint=nothing,
    framerate=20,
    title="point source propagation",
    colormap=:balance,
    background_colormap=:grays,
    wave_alpha=0.72,
    clim=nothing,
)
    isempty(frames) && error("frames is empty")
    spaceShape = size(frames[1])
    bg = _background_for_plot(background, spaceShape)
    nd = length(spaceShape)

    diag = finite_frame_diagnostics(frames)
    if diag.total_bad > 0
        @warn "wavefield frames contain $(diag.total_bad) non-finite values; first affected stored frame is $(diag.first_bad). Replacing by 0 for plotting."
    end
    clean_frames = [first(_finite_copy(f)) for f in frames]

    if clim === nothing
        vmax = diag.finite_max
        clim = vmax == 0 ? 1.0 : vmax
    end

    fig = Figure(size=(900, 680))
    ax = Axis(fig[1, 1], aspect=DataAspect(), title=title)

    if nd == 1
        x = 1:spaceShape[1]
        yobs = Observable(vec(clean_frames[1]))
        lines!(ax, x, yobs, color=:dodgerblue3, linewidth=2)
        ylims!(ax, -clim, clim)
        if sourcePoint !== nothing
            vlines!(ax, [Tuple(sourcePoint)[1]], color=:red, linewidth=2)
        end
        record(fig, videoFile, eachindex(frames); framerate=framerate) do iframe
            yobs[] = vec(clean_frames[iframe])
            ax.title = "$title  |  frame $iframe/$(length(frames))"
        end
    elseif nd == 2
        if bg !== nothing
            heatmap!(ax, bg; colormap=background_colormap)
        end
        frame_obs = Observable(clean_frames[1])
        hm = heatmap!(ax, frame_obs; colormap=colormap, colorrange=(-clim, clim), alpha=bg === nothing ? 1.0 : wave_alpha)
        Colorbar(fig[1, 2], hm, label="wavefield")
        if sourcePoint !== nothing
            xy = Tuple(sourcePoint)
            scatter!(ax, [xy[1]], [xy[2]], color=:red, markersize=14)
        end
        record(fig, videoFile, eachindex(frames); framerate=framerate) do iframe
            frame_obs[] = clean_frames[iframe]
            ax.title = "$title  |  frame $iframe/$(length(frames))"
        end
    else
        error("Video helper currently supports 1D/2D frames. For 3D/4D, select a slice before recording.")
    end

    return videoFile
end


function _finite_plot_matrix(A; fillvalue=0.0)
    B = Array{Float64}(undef, size(A))
    nbad = 0
    finite_max = 0.0
    for I in eachindex(A)
        v = Float64(real(A[I]))
        if isfinite(v)
            B[I] = v
            finite_max = max(finite_max, abs(v))
        else
            B[I] = fillvalue
            nbad += 1
        end
    end
    return B, nbad, finite_max
end

function wavefield_snapshot_report(frames)
    rows = NamedTuple[]
    for (i, frame) in pairs(frames)
        _, nbad, finite_max = _finite_plot_matrix(frame)
        push!(rows, (frame=i, nbad=nbad, finite_max=finite_max, minimum=minimum(skipmissing(vec(ifelse.(isfinite.(frame), frame, missing)))), maximum=maximum(skipmissing(vec(ifelse.(isfinite.(frame), frame, missing))))))
    end
    return rows
end


function _snapshot_background_for_plot(background, spaceShape)
    background === nothing && return nothing
    bg = Array(background)
    if size(bg) == spaceShape
        B, replaced, _ = _finite_plot_matrix(bg)
    elseif length(bg) == prod(spaceShape)
        B, replaced, _ = _finite_plot_matrix(reshape(bg, spaceShape...))
    else
        @warn "background size $(size(bg)) does not match wavefield spaceShape $spaceShape; ignoring background"
        return nothing
    end
    replaced > 0 && @warn "background had $replaced non-finite values; replaced by 0 for plotting"
    return B
end

function plot_wave_snapshots(
    frames;
    indices=nothing,
    background=nothing,
    sourcePoint=nothing,
    clim=nothing,
    ncols=3,
    title="wavefield snapshots",
    colormap=:balance,
    background_colormap=:grays,
    wave_alpha=0.72,
)
    isempty(frames) && error("frames is empty")
    if indices === nothing
        indices = unique(round.(Int, range(1, length(frames), length=min(9, length(frames)))))
    end
    indices = collect(indices)

    clean_frames = Array{Float64}[]
    nbad = Int[]
    finite_maxima = Float64[]
    for i in indices
        B, bad, fmax = _finite_plot_matrix(frames[i])
        push!(clean_frames, B)
        push!(nbad, bad)
        push!(finite_maxima, fmax)
    end

    if clim === nothing
        vmax = maximum(finite_maxima)
        clim = vmax == 0 ? 1.0 : vmax
    end

    bg = _snapshot_background_for_plot(background, size(frames[1]))
    n = length(indices)
    nrows = cld(n, ncols)
    fig = Figure(size=(320*ncols, 300*nrows))

    for (k, iframe) in enumerate(indices)
        r = cld(k, ncols)
        c = k - (r - 1) * ncols
        ax = Axis(fig[r, c], aspect=DataAspect(), title="frame $iframe | bad=$(nbad[k])")
        if length(size(frames[1])) == 1
            lines!(ax, vec(clean_frames[k]), color=:dodgerblue3, linewidth=2)
            ylims!(ax, -clim, clim)
            if sourcePoint !== nothing
                vlines!(ax, [Tuple(sourcePoint)[1]], color=:red, linewidth=2)
            end
        elseif length(size(frames[1])) == 2
            if bg !== nothing
                heatmap!(ax, bg; colormap=background_colormap)
            end
            heatmap!(ax, clean_frames[k]; colormap=colormap, colorrange=(-clim, clim), alpha=bg === nothing ? 1.0 : wave_alpha)
            if sourcePoint !== nothing
                xy = Tuple(sourcePoint)
                scatter!(ax, [xy[1]], [xy[2]], color=:red, markersize=10)
            end
        else
            error("Snapshot helper supports 1D/2D frames. Select a slice first for higher dimensions.")
        end
    end

    Label(fig[0, :], title)
    return fig
end


function save_wave_snapshot_pdf(
    frames;
    filename=joinpath(@__DIR__, "wave_snapshots.pdf"),
    indices=nothing,
    background=nothing,
    sourcePoint=nothing,
    clim=nothing,
    ncols=3,
    title="wavefield snapshots",
    kwargs...,
)
    fig = plot_wave_snapshots(
        frames;
        indices=indices,
        background=background,
        sourcePoint=sourcePoint,
        clim=clim,
        ncols=ncols,
        title=title,
        kwargs...,
    )
    save(filename, fig)
    empty!(fig.scene)  # release most displayed scene resources
    GC.gc()
    return filename
end


using Statistics

function cfl_diagnostics(model_velocity, Δnum; cfl_safety=0.45, ppw=10, samples_per_period=20)
    v = vec(Float64.(model_velocity))
    finite_v = filter(isfinite, v)
    dt = Float64(Δnum[end])
    dx = Float64.(Δnum[1:end-1])
    dxmin = minimum(dx)
    dxmax = maximum(dx)
    vmax = maximum(abs, finite_v)
    vmin = minimum(abs, filter(!iszero, finite_v))
    cfl = vmax * dt / dxmin

    # MKS units: velocity [m/s], dx [m], dt [s], frequency [Hz].
    # For a Ricker wavelet in flexOPT, f0 is the dominant/peak frequency.
    fmax_spatial = vmin / (ppw * dxmax)
    fmax_temporal = 1 / (samples_per_period * dt)
    suggested_ricker_f0 = min(fmax_spatial, fmax_temporal)

    return (
        v_min=minimum(finite_v),
        v_median=median(finite_v),
        v_max=maximum(finite_v),
        dt=dt,
        dxmin=dxmin,
        dxmax=dxmax,
        cfl_max=cfl,
        suggested_dt_2D=cfl_safety * dxmin / vmax,
        ppw=ppw,
        samples_per_period=samples_per_period,
        fmax_spatial=fmax_spatial,
        fmax_temporal=fmax_temporal,
        suggested_ricker_f0=suggested_ricker_f0,
        suggested_ricker_t0=1.5 / suggested_ricker_f0,
    )
end

function ricker_time_signal(Nt, Δt; f0, t0=1.5/f0, timePointsUsedForOneStep=1)
    ntime = Nt + timePointsUsedForOneStep - 1
    t = (0:ntime-1) .* Δt
    return t, Ricker.(t, t0, f0)
end

function dominant_frequency_scan(signal, Δt; nfreq=512)
    y = Float64.(signal) .- mean(Float64.(signal))
    n = length(y)
    duration = n * Δt
    fmin = 1 / duration
    fmax = 0.5 / Δt
    freqs = range(fmin, fmax; length=nfreq)
    t = (0:n-1) .* Δt
    spectrum = [abs(sum(y .* exp.(-2im * π * f .* t))) for f in freqs]
    imax = argmax(spectrum)
    return (f_peak=freqs[imax], peak_amplitude=spectrum[imax], frequencies=freqs, spectrum=spectrum)
end

function ricker_diagnostics(model_velocity, Δnum; f0=nothing, Nt=200, timePointsUsedForOneStep=1, ppw=10, samples_per_period=20, t0=nothing)
    cfl = cfl_diagnostics(model_velocity, Δnum; ppw=ppw, samples_per_period=samples_per_period)
    f0_used = f0 === nothing ? cfl.suggested_ricker_f0 : Float64(f0)
    t0_used = t0 === nothing ? 1.5 / f0_used : Float64(t0)
    t, signal = ricker_time_signal(Nt, cfl.dt; f0=f0_used, t0=t0_used, timePointsUsedForOneStep=timePointsUsedForOneStep)
    freqinfo = dominant_frequency_scan(signal, cfl.dt)
    return (
        f0=f0_used,
        t0=t0_used,
        duration=t[end],
        dominant_frequency_scan=freqinfo.f_peak,
        wavelength_min_velocity=cfl.v_min / f0_used,
        points_per_wavelength_min_velocity=(cfl.v_min / f0_used) / cfl.dxmax,
        samples_per_period=1 / (f0_used * cfl.dt),
        max_abs=maximum(abs, signal),
        first_peak_time=t[argmax(abs.(signal))],
        cfl=cfl,
    )
end

function frame_motion_report(frames)
    rows = NamedTuple[]
    previous = nothing
    for (i, frame) in pairs(frames)
        _, nbad, fmax = _finite_plot_matrix(frame)
        dnorm = previous === nothing ? 0.0 : norm(frame .- previous)
        dmax = previous === nothing ? 0.0 : maximum(abs, frame .- previous)
        push!(rows, (frame=i, nbad=nbad, maxabs=fmax, norm=norm(frame), diff_norm=dnorm, diff_max=dmax))
        previous = frame
    end
    return rows
end

function difference_frames(frames)
    length(frames) >= 2 || error("need at least two frames")
    return [frames[i] .- frames[i-1] for i in 2:length(frames)]
end


function _sparse_norms(A)
    return (
        size=size(A),
        nnz=nnz(A),
        maxabs=nnz(A) == 0 ? 0.0 : maximum(abs, nonzeros(A)),
        norm=norm(A),
    )
end

function diagnose_prepared_linear_system(preparedLin)
    println("spaceShape = ", preparedLin.spaceShape)
    println("NpointsSpace = ", preparedLin.NpointsSpace)
    println("NField = ", preparedLin.NField)
    println("NForceField = ", hasproperty(preparedLin, :NForceField) ? preparedLin.NForceField : preparedLin.NField)
    println("timePointsUsedForOneStep = ", preparedLin.timePointsUsedForOneStep)
    println("A_unknown: ", _sparse_norms(preparedLin.A_unknown))
    println("L_known:   ", _sparse_norms(preparedLin.L_known))
    println("R_force:   ", _sparse_norms(preparedLin.R_force))
    if hasproperty(preparedLin, :right_operator)
        println("right_operator.size = ", preparedLin.right_operator.size)
        println("right_operator.nnz = ", length(preparedLin.right_operator.table.vals))
    end
    row_nnz_A = vec(sum(abs.(preparedLin.A_unknown) .> 0; dims=2))
    row_nnz_R = vec(sum(abs.(preparedLin.R_force) .> 0; dims=2))
    println("rows with A entries = ", count(!iszero, row_nnz_A), " / ", length(row_nnz_A))
    println("rows with R entries = ", count(!iszero, row_nnz_R), " / ", length(row_nnz_R))
    return nothing
end

function source_field_response(preparedLin, sourcePoint, timeSignal; amplitude=1.0, it=nothing)
    LI = LinearIndices(preparedLin.spaceShape)
    src_linear = LI[sourcePoint]
    last_start = length(timeSignal) - preparedLin.timePointsUsedForOneStep + 1
    last_start >= 1 || error("timeSignal is too short for timePointsUsedForOneStep=$(preparedLin.timePointsUsedForOneStep)")
    if it === nothing
        it = min(argmax(abs.(timeSignal)), last_start)
    else
        1 <= it <= last_start || error("it should be in 1:$last_start for this timeSignal")
    end

    out = NamedTuple[]
    NForceField = hasproperty(preparedLin, :NForceField) ? preparedLin.NForceField : preparedLin.NField
    for iField in 1:NForceField
        w = zeros(Float64, preparedLin.NforcePoints)
        w[src_linear] = 1.0
        sourceFull = make_sourceFull(preparedLin, w, timeSignal; iField=iField, amplitude=amplitude)
        knownForce = sourceFull[:, :, it:it+preparedLin.timePointsUsedForOneStep-1]
        knownInputs = vcat(vec(zero(preparedLin.known_lhs_template)), vec(knownForce))
        b = copy(preparedLin.b_template)
        preparedLin.b_fun!(b, knownInputs)
        push!(out, (
            iField=iField,
            time_index=it,
            source_norm=norm(knownForce),
            b_norm=norm(b),
            b_maximum=maximum(abs, b),
            b_nonzero=count(!iszero, b),
        ))
    end
    return out
end

function one_step_response(preparedLin, sourceFull; it=1, boundaryConditionForced=false)
    knownField = zero(preparedLin.known_lhs_template)
    knownForce = sourceFull[:, :, it:it+preparedLin.timePointsUsedForOneStep-1]
    knownInputs = vcat(vec(knownField), vec(knownForce))
    b = copy(preparedLin.b_template)
    preparedLin.b_fun!(b, knownInputs)

    A = sparse(preparedLin.A_template)
    if boundaryConditionForced
        apply_forced_boundary_for_video!(A, b; leftValue=0.0, rightValue=0.0)
    end
    u = A \ b
    ufield = reshape(real.(u), preparedLin.NpointsSpace, preparedLin.NField)
    return (
        b_norm=norm(b),
        b_maximum=maximum(abs, b),
        b_nonzero=count(!iszero, b),
        u_norm=norm(u),
        u_maximum=maximum(abs, u),
        u_nonzero=count(!iszero, u),
        ufield=ufield,
    )
end


function _matrix_row_abs_sums(A)
    return vec(sum(abs.(A); dims=2))
end

function operator_growth_diagnostics(preparedLin)
    A = sparse(preparedLin.A_unknown)
    L = sparse(preparedLin.L_known)
    R = sparse(preparedLin.R_force)
    diagA = abs.(diag(A))
    rowA = _matrix_row_abs_sums(A)
    offdiagA = rowA .- diagA
    rowL = _matrix_row_abs_sums(L)
    rowR = _matrix_row_abs_sums(R)
    return (
        A_size=size(A),
        A_nnz=nnz(A),
        A_diag_min=isempty(diagA) ? 0.0 : minimum(diagA),
        A_diag_median=isempty(diagA) ? 0.0 : median(diagA),
        A_diag_max=isempty(diagA) ? 0.0 : maximum(diagA),
        A_offdiag_over_diag_max=maximum(offdiagA ./ max.(diagA, eps(Float64))),
        L_row_sum_max=maximum(rowL),
        R_row_sum_max=maximum(rowR),
        R_over_A_diag_max=maximum(rowR ./ max.(diagA, eps(Float64))),
        condest_dense_sample=size(A,1) <= 5000 ? cond(Matrix(A)) : missing,
    )
end

function homogeneous_zero_test(preparedLin; Nt=20, store_every=1)
    sourceFull_zero = zeros(eltype(preparedLin.known_rhs_template), preparedLin.NforcePoints, preparedLin.NForceField, Nt + preparedLin.timePointsUsedForOneStep - 1)
    frames_zero = propagate_linear_frames(preparedLin, sourceFull_zero, Nt; store_every=store_every, blowup_limit=1e12)
    return wavefield_snapshot_report(frames_zero)
end

function source_amplitude_sweep(preparedLin, sourcePoint, timeSignal; amplitudes=(1e-12, 1e-10, 1e-8, 1e-6), iField=1, Nt=20)
    rows = NamedTuple[]
    w = point_source_weights(preparedLin, sourcePoint)
    for amp in amplitudes
        sourceFull = make_sourceFull(preparedLin, w, timeSignal; iField=iField, amplitude=amp)
        frames = propagate_linear_frames(preparedLin, sourceFull, Nt; store_every=max(1, Nt ÷ 5), blowup_limit=1e30)
        report = wavefield_snapshot_report(frames)
        push!(rows, (amplitude=amp, last=report[end], nframes=length(frames)))
    end
    return rows
end


function _distance_from_source_table(frame, sourcePoint; threshold_fraction=1e-6)
    f = abs.(Float64.(frame))
    fmax = maximum(f)
    threshold = threshold_fraction * max(fmax, eps(Float64))
    active = findall(>(threshold), f)
    if isempty(active)
        return (nactive=0, max_distance=0.0, mean_distance=0.0, fmax=fmax)
    end
    distances = [sqrt(sum((Tuple(I) .- Tuple(sourcePoint)).^2)) for I in active]
    return (
        nactive=length(active),
        max_distance=maximum(distances),
        mean_distance=mean(distances),
        fmax=fmax,
    )
end

function propagation_radius_report(frames, sourcePoint; threshold_fraction=1e-6)
    return [(frame=i, _distance_from_source_table(frame, sourcePoint; threshold_fraction=threshold_fraction)...) for (i, frame) in pairs(frames)]
end

function impulse_sourceFull(preparedLin, sourcePoint; iField=1, amplitude=1.0, Nt=20, impulse_time=1)
    timeSignal = zeros(Float64, Nt + preparedLin.timePointsUsedForOneStep - 1)
    timeSignal[impulse_time] = 1.0
    w = point_source_weights(preparedLin, sourcePoint)
    return make_sourceFull(preparedLin, w, timeSignal; iField=iField, amplitude=amplitude)
end

function impulse_propagation_test(preparedLin, sourcePoint; iField=1, amplitude=1.0, Nt=20, store_every=1, threshold_fraction=1e-8)
    sourceFull = impulse_sourceFull(preparedLin, sourcePoint; iField=iField, amplitude=amplitude, Nt=Nt)
    frames = propagate_linear_frames(preparedLin, sourceFull, Nt; store_every=store_every, blowup_limit=1e20)
    return (
        frames=frames,
        snapshots=wavefield_snapshot_report(frames),
        radius=propagation_radius_report(frames, sourcePoint; threshold_fraction=threshold_fraction),
    )
end


function frame_extrema_report(frames, sourcePoint)
    rows = NamedTuple[]
    for (i, frame) in pairs(frames)
        A = Float64.(frame)
        absA = abs.(A)
        imax = argmax(absA)
        maxpoint = CartesianIndices(size(A))[imax]
        distance = sqrt(sum((Tuple(maxpoint) .- Tuple(sourcePoint)).^2))
        push!(rows, (
            frame=i,
            maxpoint=maxpoint,
            distance_from_source=distance,
            value=A[imax],
            maxabs=absA[imax],
            minimum=minimum(A),
            maximum=maximum(A),
        ))
    end
    return rows
end


function _get_numop(numOps, which::Symbol)
    ops = numOps isa AbstractDict ? numOps["numericalOperators"] : numOps.numericalOperators
    return getproperty(ops, which)
end

function _point_linear_index(op, point)
    νWhole = op.geometry.νWhole
    if point isa Integer
        1 <= point <= length(νWhole) || error("point integer should be in 1:$(length(νWhole))")
        return Int(point), νWhole[Int(point)]
    elseif point isa CartesianIndex
        point in νWhole || error("point $point is not in op.geometry.νWhole")
        pointLinear = LinearIndices(νWhole)[point]
        return Int(pointLinear), point
    else
        error("point should be an Int linear point index or a CartesianIndex")
    end
end

function _time_role(iT, activeTimePoints)
    if iT == activeTimePoints
        return :future_unknown
    elseif iT == activeTimePoints - 1
        return :present_known
    else
        return Symbol("past_known_tminus$(activeTimePoints - iT)")
    end
end

function coefficient_rows_at_point(op, point; iExpr=1, atol=0.0)
    pointLinear, pointCI = _point_linear_index(op, point)
    nPoints = length(op.geometry.νWhole)
    activeTimePoints = op.geometry.activeTimePoints
    spaceShape = Tuple(op.geometry.wholeRegionPointsSpace)
    Nexpr = op.size[1] ÷ nPoints
    Nfield = op.size[2] ÷ (nPoints * activeTimePoints)

    1 <= iExpr <= Nexpr || error("iExpr should be in 1:$Nexpr")

    residualLI = LinearIndices((Nexpr, nPoints))
    targetRow = residualLI[iExpr, pointLinear]
    fieldTimeSpaceCI = CartesianIndices((Nfield, activeTimePoints, spaceShape...))

    rows = NamedTuple[]
    for k in eachindex(op.table.vals)
        Int(op.table.rows[k]) == targetRow || continue
        val = op.table.vals[k]
        abs(val) > atol || continue

        ci = fieldTimeSpaceCI[Int(op.table.cols[k])]
        iField = ci[1]
        iT = ci[2]
        neighbour = CartesianIndex(Tuple(ci)[3:end])
        offset = Tuple(neighbour - pointCI)
        timeRole = _time_role(iT, activeTimePoints)

        push!(rows, (
            k = k,
            residual_row = targetRow,
            expr = iExpr,
            point = pointCI,
            field = iField,
            time_slot = iT,
            time_role = timeRole,
            neighbour = neighbour,
            offset = offset,
            coef = val,
            abscoef = abs(val),
        ))
    end

    sort!(rows, by = r -> (r.time_slot, r.field, r.offset))
    return rows
end

function inspect_operator_coefficients_at_point(numOps, point; which=:residual, iExpr=1, atol=0.0)
    op = _get_numop(numOps, which)
    rows = coefficient_rows_at_point(op, point; iExpr=iExpr, atol=atol)
    println("operator = ", which)
    println("point = ", point, ", iExpr = ", iExpr, ", ncoef = ", length(rows))
    for r in rows
        println(
            "t=", r.time_slot,
            " (", r.time_role, ")",
            " field=", r.field,
            " offset=", r.offset,
            " neighbour=", r.neighbour,
            " coef=", r.coef,
        )
    end
    return rows
end

function coefficient_table_by_time(rows)
    slots = sort(unique(getproperty.(rows, :time_slot)))
    return [(
        time_slot=s,
        time_role=first(r.time_role for r in rows if r.time_slot == s),
        n=length(filter(r -> r.time_slot == s, rows)),
        sumcoef=sum(r.coef for r in rows if r.time_slot == s),
        sumabs=sum(r.abscoef for r in rows if r.time_slot == s),
    ) for s in slots]
end

function coefficient_rows_for_role(rows, role::Symbol)
    return filter(r -> r.time_role == role, rows)
end


function prepare_fd2d_acoustic_baseline(velocity, Δnum; source_scale=:dt2)
    spaceShape = size(velocity)
    length(spaceShape) == 2 || error("baseline helper currently expects a 2D velocity model")
    dx, dz, dt = Float64.(Δnum)
    nPoints = prod(spaceShape)
    LI = LinearIndices(spaceShape)

    rowsA = Int[]; colsA = Int[]; valsA = Float64[]
    rowsL = Int[]; colsL = Int[]; valsL = Float64[]
    rowsR = Int[]; colsR = Int[]; valsR = Float64[]

    # known_lhs_template is (point, field, time) with time 1=past and time 2=present.
    col_known(point, it) = point + (it - 1) * nPoints

    for I in CartesianIndices(spaceShape)
        p = LI[I]
        v = Float64(velocity[I])
        cx = (v * dt / dx)^2
        cz = (v * dt / dz)^2

        # A_unknown * u_future = b, here A_unknown is identity.
        push!(rowsA, p); push!(colsA, p); push!(valsA, 1.0)

        # Residual form:
        # u_future + u_past + (-2 + 2cx + 2cz)u_present
        #   - cx*(u_left + u_right) - cz*(u_down + u_up) - source_term = 0
        # prepareNumericalLinearSystem uses b = -L_known*known + R_force*source.
        push!(rowsL, p); push!(colsL, col_known(p, 1)); push!(valsL, 1.0)
        push!(rowsL, p); push!(colsL, col_known(p, 2)); push!(valsL, -2.0 + 2cx + 2cz)

        i, j = Tuple(I)
        if i > 1
            q = LI[CartesianIndex(i - 1, j)]
            push!(rowsL, p); push!(colsL, col_known(q, 2)); push!(valsL, -cx)
        end
        if i < spaceShape[1]
            q = LI[CartesianIndex(i + 1, j)]
            push!(rowsL, p); push!(colsL, col_known(q, 2)); push!(valsL, -cx)
        end
        if j > 1
            q = LI[CartesianIndex(i, j - 1)]
            push!(rowsL, p); push!(colsL, col_known(q, 2)); push!(valsL, -cz)
        end
        if j < spaceShape[2]
            q = LI[CartesianIndex(i, j + 1)]
            push!(rowsL, p); push!(colsL, col_known(q, 2)); push!(valsL, -cz)
        end

        # Source field is one scalar component, same point indexing.
        rscale = source_scale == :dt2 ? dt^2 : Float64(source_scale)
        push!(rowsR, p); push!(colsR, p); push!(valsR, rscale)
    end

    A_unknown = sparse(rowsA, colsA, valsA, nPoints, nPoints)
    L_known = sparse(rowsL, colsL, valsL, nPoints, 2nPoints)
    R_force = sparse(rowsR, colsR, valsR, nPoints, nPoints * 3)

    b_template = zeros(Float64, nPoints)
    known_lhs_template = zeros(Float64, nPoints, 1, 2)
    known_rhs_template = zeros(Float64, nPoints, 1, 3)

    function b_fun!(b, knownInputs)
        nKnownField = length(known_lhs_template)
        knownFieldVec = @view knownInputs[1:nKnownField]
        knownForceVec = @view knownInputs[nKnownField+1:nKnownField+length(known_rhs_template)]
        b .= 0.0
        mul!(b, L_known, knownFieldVec, -1.0, 1.0)
        mul!(b, R_force, knownForceVec, 1.0, 1.0)
        return b
    end

    function A_fun!(avec, knownInputs)
        avec .= vec(A_unknown)
        return avec
    end

    return (
        is_numerical=true,
        is_fd2d_baseline=true,
        A_unknown=A_unknown,
        L_known=L_known,
        R_force=R_force,
        A_fun! = A_fun!,
        b_fun! = b_fun!,
        A_template=copy(A_unknown),
        b_template=b_template,
        known_lhs_template=known_lhs_template,
        known_rhs_template=known_rhs_template,
        spaceShape=spaceShape,
        NpointsSpace=nPoints,
        NforcePoints=nPoints,
        NField=1,
        NForceField=1,
        timePointsUsedForOneStep=3,
    )
end


function crop_frames_around_source(frames, sourcePoint; radius=80)
    ndims(frames[1]) == 2 || error("crop helper currently expects 2D frames")
    nx, nz = size(frames[1])
    sx, sz = Tuple(sourcePoint)
    xr = max(1, sx-radius):min(nx, sx+radius)
    zr = max(1, sz-radius):min(nz, sz+radius)
    cropped = [frame[xr, zr] for frame in frames]
    cropped_source = CartesianIndex(sx - first(xr) + 1, sz - first(zr) + 1)
    return cropped, cropped_source, (xr=xr, zr=zr)
end

function source_line_report(frames, sourcePoint; axis=1)
    rows = NamedTuple[]
    for (i, frame) in pairs(frames)
        if axis == 1
            line = frame[:, Tuple(sourcePoint)[2]]
        else
            line = frame[Tuple(sourcePoint)[1], :]
        end
        push!(rows, (
            frame=i,
            maxabs=maximum(abs, line),
            argmax=argmax(abs.(line)),
            minimum=minimum(line),
            maximum=maximum(line),
        ))
    end
    return rows
end


## 4. Shared Ricker Source

In [ ]:
# Ricker source diagnostics shared by FD and OPT tests.
# MKS units: dx,dz [m], dt [s], velocity [m/s], f0 [Hz].
cfl_info = cfl_diagnostics(models[1], Δnum)
ricker_info = ricker_diagnostics(models[1], Δnum; Nt=Nt_debug, timePointsUsedForOneStep=pointsInTime)
source_f0 = ricker_info.f0
run_duration = (Nt_debug + pointsInTime - 2) * Δnum[end]
source_t0 = min(ricker_info.t0, 0.30 * run_duration)

@show cfl_info
@show ricker_info
@show run_duration source_f0 source_t0


## 5. Tiny Homogeneous FD Sanity Check

In [ ]:
# Tiny homogeneous sanity check. This should visibly propagate.
toy_shape = (201, 201)
toy_velocity_value = vmed
toy_velocity = fill(toy_velocity_value, toy_shape)
toy_dx = 100.0
toy_dt = cfl_safety * toy_dx / toy_velocity_value
Δtoy = (toy_dx, toy_dx, toy_dt)

preparedToy = prepare_fd2d_acoustic_baseline(toy_velocity, Δtoy)
toy_sourcePoint = CartesianIndex(cld(toy_shape[1], 2), cld(toy_shape[2], 2))
Nt_toy = 180
store_every_toy = 3
f0_toy = 4.0
t0_toy = 0.20

toy_timeSignal = source_time_samples(Nt_toy, Δtoy[end], preparedToy.timePointsUsedForOneStep; t0=t0_toy, f0=f0_toy)
toy_sourceFull = make_sourceFull(preparedToy, point_source_weights(preparedToy, toy_sourcePoint), toy_timeSignal; iField=1, amplitude=1.0)
frames_toy = propagate_linear_frames(preparedToy, toy_sourceFull, Nt_toy; store_every=store_every_toy, blowup_limit=1e12)

toy_report = wavefield_snapshot_report(frames_toy)
toy_radius = propagation_radius_report(frames_toy, toy_sourcePoint; threshold_fraction=1e-4)
@show Δtoy maximum(abs, toy_timeSignal) length(frames_toy)
@show toy_report[1] toy_report[end]
toy_radius


In [ ]:
fig_toy = plot_wave_snapshots(frames_toy; sourcePoint=toy_sourcePoint, title="toy homogeneous 3x3 FD point source")
fig_toy


## 6. Marmousi Baseline 3x3 FD

In [ ]:
# Known-good 3x3 finite-difference operator on the Marmousi grid.
preparedFD = prepare_fd2d_acoustic_baseline(models[1], Δnum)
diagnose_prepared_linear_system(preparedFD)
fd_growth_diag = operator_growth_diagnostics(preparedFD)
@show fd_growth_diag

fd_timeSignal = source_time_samples(Nt_debug, Δnum[end], preparedFD.timePointsUsedForOneStep; t0=source_t0, f0=source_f0)
fd_sourceFull = make_sourceFull(preparedFD, point_source_weights(preparedFD, sourcePoint), fd_timeSignal; iField=1, amplitude=source_amplitude_fd)

frames_fd = propagate_linear_frames(preparedFD, fd_sourceFull, Nt_debug; store_every=store_every_debug, blowup_limit=1e12)
report_fd = wavefield_snapshot_report(frames_fd)
fd_radius = propagation_radius_report(frames_fd, sourcePoint; threshold_fraction=1e-6)

@show maximum(abs, fd_timeSignal) maximum(abs, fd_sourceFull) length(frames_fd)
@show report_fd[1] report_fd[end]
fd_radius


In [ ]:
frames_fd_zoom, fd_source_zoom, fd_crop_ranges = crop_frames_around_source(frames_fd, sourcePoint; radius=zoom_radius)
@show fd_crop_ranges
fig_fd_zoom = plot_wave_snapshots(frames_fd_zoom; sourcePoint=fd_source_zoom, title="Marmousi baseline 3x3 FD zoom")
fig_fd_zoom


## 7. Build OPT Numerical Operators

In [ ]:
# Build the OPT recipe and numerical operators. This is the expensive part.
famousEquationType = "2DacousticTime"
orderBspace = 1
orderBtime = 1
pointsInSpace = 3
supplementaryOrder = 2

fieldItpl = (ptsSpace=1, ptsTime=1, offsetSpace=1, offsetTime=1, YorderBspace=0, YorderBtime=0)
materItpl = (ptsSpace=1, ptsTime=1, offsetSpace=1, offsetTime=1, YorderBspace=0, YorderBtime=0)

concreteParametersForOPTConstruction = @strdict famousEquationType Δ orderBtime orderBspace pointsInSpace pointsInTime supplementaryOrder fieldItpl materItpl
optRec = makeOPTsemiSymbolic(concreteParametersForOPTConstruction)
recette = optRec["recette"]

modelPoints = getModelPoints(models[1], pointsInTime, recette.numbersOfTheSystem.numbersOfTheSystemL.timeMarching)
tmpModel = (models=models, modelPoints=modelPoints, Δ=Δ, modelName="Marmousi_debug")
params = @strdict optRec=optRec modelFam=tmpModel absorbingBoundaries=nothing maskedRegionInSpace=nothing representation="matrixfree"

numOpt = numericalOperatorConstruction(params)
numOps = numOpt["numOperators"]
preparedLin = prepareLinearSystem(numOps)

@show preparedLin.spaceShape preparedLin.NpointsSpace preparedLin.NforcePoints preparedLin.NField preparedLin.NForceField preparedLin.timePointsUsedForOneStep


In [ ]:
recette.lhs.Ajiννᶜ

## 8. OPT Diagnostics

In [ ]:
diagnose_prepared_linear_system(preparedLin)
opt_growth_diag = operator_growth_diagnostics(preparedLin)
@show opt_growth_diag

zero_report_opt = homogeneous_zero_test(preparedLin; Nt=20, store_every=5)
@show zero_report_opt[1] zero_report_opt[end]


## 9. Choose OPT Source Component

In [ ]:
timeSignal_probe = source_time_samples(50, Δnum[end], preparedLin.timePointsUsedForOneStep; t0=source_t0, f0=source_f0)
field_responses = source_field_response(preparedLin, sourcePoint, timeSignal_probe; amplitude=1.0)
best_source_field = argmax(getproperty.(field_responses, :b_norm))
@show best_source_field
field_responses


## 10. OPT Impulse Test

In [ ]:
impulse_opt = impulse_propagation_test(
    preparedLin,
    sourcePoint;
    iField=best_source_field,
    amplitude=1e-12,
    Nt=20,
    store_every=1,
    threshold_fraction=1e-8,
)
impulse_extrema_opt = frame_extrema_report(impulse_opt.frames, sourcePoint)
@show impulse_opt.snapshots[1] impulse_opt.snapshots[end]
@show impulse_extrema_opt[1] impulse_extrema_opt[end]
impulse_opt.radius


## 11. OPT Ricker Propagation

In [ ]:
timeSignal_opt = source_time_samples(Nt_debug, Δnum[end], preparedLin.timePointsUsedForOneStep; t0=source_t0, f0=source_f0)
sourceFull_opt = make_sourceFull(preparedLin, point_source_weights(preparedLin, sourcePoint), timeSignal_opt; iField=best_source_field, amplitude=source_amplitude_opt)

frames_opt = propagate_linear_frames(preparedLin, sourceFull_opt, Nt_debug; store_every=store_every_debug, blowup_limit=1e12)
report_opt = wavefield_snapshot_report(frames_opt)
opt_radius = propagation_radius_report(frames_opt, sourcePoint; threshold_fraction=1e-6)

@show maximum(abs, timeSignal_opt) maximum(abs, sourceFull_opt) length(frames_opt)
@show report_opt[1] report_opt[end]
opt_radius


In [ ]:
frames_opt_zoom, opt_source_zoom, opt_crop_ranges = crop_frames_around_source(frames_opt, sourcePoint; radius=zoom_radius)
@show opt_crop_ranges
fig_opt_zoom = plot_wave_snapshots(frames_opt_zoom; sourcePoint=opt_source_zoom, title="OPT numerical operator zoom")
fig_opt_zoom


## 12. Compare FD And OPT

In [ ]:
comparison = (
    fd_last = report_fd[end],
    opt_last = report_opt[end],
    fd_radius_last = fd_radius[end],
    opt_radius_last = opt_radius[end],
    fd_growth = fd_growth_diag,
    opt_growth = opt_growth_diag,
)
comparison


## 13. Inspect OPT Coefficients At Source

In [ ]:
inspectPoint = sourcePoint
coef_rows_right = inspect_operator_coefficients_at_point(numOps, inspectPoint; which=:right, iExpr=1, atol=0.0)
coef_rows_left = inspect_operator_coefficients_at_point(numOps, inspectPoint; which=:left, iExpr=1, atol=0.0)

(
    right_by_time = coefficient_table_by_time(coef_rows_right),
    left_by_time = coefficient_table_by_time(coef_rows_left),
)


## 14. Compare OPT 27 Coefficients Against 3x3x3 FD

This prints the local left-hand stencil at `sourcePoint` and compares it with the standard 3x3 finite-difference stencil. The comparison is normalized by the future-center coefficient. If the time slots are permuted, the signs are flipped, or the spatial coefficients are missing, it should be visible here.

In [ ]:
function fd_reference_stencil_at_point(velocity, Δnum, point)
    dx, dz, dt = Float64.(Δnum)
    v = Float64(velocity[point])
    cx = (v * dt / dx)^2
    cz = (v * dt / dz)^2
    return [
        (time_role=:past_known_tminus2, time_slot=1, offset=(0, 0), coef=1.0),
        (time_role=:present_known, time_slot=2, offset=(0, 0), coef=-2.0 + 2cx + 2cz),
        (time_role=:present_known, time_slot=2, offset=(-1, 0), coef=-cx),
        (time_role=:present_known, time_slot=2, offset=(1, 0), coef=-cx),
        (time_role=:present_known, time_slot=2, offset=(0, -1), coef=-cz),
        (time_role=:present_known, time_slot=2, offset=(0, 1), coef=-cz),
        (time_role=:future_unknown, time_slot=3, offset=(0, 0), coef=1.0),
    ]
end

function compact_stencil_table(rows; field=1, atol=0.0)
    kept = filter(r -> r.field == field && abs(r.coef) > atol, rows)
    return sort([
        (
            time_slot=r.time_slot,
            time_role=r.time_role,
            offset=r.offset,
            coef=r.coef,
            abscoef=abs(r.coef),
        ) for r in kept
    ], by = r -> (r.time_slot, r.offset))
end

function normalize_stencil_by_future_center(table)
    pivots = [r.coef for r in table if r.time_role == :future_unknown && r.offset == (0, 0)]
    isempty(pivots) && return table, missing
    pivot = first(pivots)
    return [merge(r, (; coef_norm = r.coef / pivot)) for r in table], pivot
end

function opt_fd_stencil_comparison(numOps, velocity, Δnum, point; iExpr=1, field=1, atol=0.0)
    opt_rows = inspect_operator_coefficients_at_point(numOps, point; which=:left, iExpr=iExpr, atol=atol)
    opt_table = compact_stencil_table(opt_rows; field=field, atol=atol)
    opt_norm, opt_pivot = normalize_stencil_by_future_center(opt_table)

    fd_table = fd_reference_stencil_at_point(velocity, Δnum, point)
    fd_norm, fd_pivot = normalize_stencil_by_future_center(fd_table)

    return (
        opt_pivot=opt_pivot,
        fd_pivot=fd_pivot,
        opt_table=opt_table,
        opt_normalized=opt_norm,
        fd_normalized=fd_norm,
        opt_by_time=coefficient_table_by_time(opt_rows),
    )
end

function print_stencil_table(table; title="stencil")
    println(title)
    for r in table
        if hasproperty(r, :coef_norm)
            println("t=", r.time_slot, " ", r.time_role, " offset=", r.offset, " coef=", r.coef, " coef_norm=", r.coef_norm)
        else
            println("t=", r.time_slot, " ", r.time_role, " offset=", r.offset, " coef=", r.coef)
        end
    end
    return nothing
end

function print_opt_geometry_time_layout(numOps)
    left = numOps.numericalOperators.left
    g = left.geometry
    println("activeTimePoints = ", g.activeTimePoints)
    println("timePointsUsedForOneStep = ", g.timePointsUsedForOneStep)
    println("first geometry local points:")
    for (i, p) in pairs(g.localPointsIndices[1])
        println(i, " => ", p)
    end
    println("middlepoint geometry 1 = ", g.middlepoints[1])
    return nothing
end


In [ ]:
print_opt_geometry_time_layout(numOps)

stencil_cmp = opt_fd_stencil_comparison(numOps, models[1], Δnum, sourcePoint; iExpr=1, field=1, atol=0.0)
@show stencil_cmp.opt_pivot stencil_cmp.fd_pivot
@show stencil_cmp.opt_by_time

print_stencil_table(stencil_cmp.fd_normalized; title="FD normalized reference stencil")
print_stencil_table(stencil_cmp.opt_normalized; title="OPT normalized left stencil at sourcePoint")


### Stencil Moment Summary

For OPT/Galerkin stencils, coefficients may be spread over all 3 time slots. This moment summary checks whether each time slice has reasonable total weight and whether the stencil has directional bias.

In [ ]:
function stencil_moment_report(table)
    slots = sort(unique(getproperty.(table, :time_slot)))
    rows = NamedTuple[]
    for s in slots
        subset = filter(r -> r.time_slot == s, table)
        sumcoef = sum(r.coef for r in subset)
        sumabs = sum(abs(r.coef) for r in subset)
        mx = sum(r.coef * r.offset[1] for r in subset)
        mz = sum(r.coef * r.offset[2] for r in subset)
        mxx = sum(r.coef * r.offset[1]^2 for r in subset)
        mzz = sum(r.coef * r.offset[2]^2 for r in subset)
        mxz = sum(r.coef * r.offset[1] * r.offset[2] for r in subset)
        push!(rows, (
            time_slot=s,
            time_role=first(r.time_role for r in subset),
            sumcoef=sumcoef,
            sumabs=sumabs,
            first_moment_x=mx,
            first_moment_z=mz,
            second_moment_xx=mxx,
            second_moment_zz=mzz,
            cross_moment_xz=mxz,
        ))
    end
    return rows
end

function normalized_stencil_moments(table)
    normalized, pivot = normalize_stencil_by_future_center(table)
    return stencil_moment_report(normalized), pivot
end

function print_stencil_moments(label, table)
    moments, pivot = normalized_stencil_moments(table)
    println(label, " pivot = ", pivot)
    for r in moments
        println(r)
    end
    return moments
end


In [ ]:
fd_moments = stencil_moment_report(stencil_cmp.fd_normalized)
opt_moments = stencil_moment_report(stencil_cmp.opt_normalized)

println("FD normalized moments")
foreach(println, fd_moments)
println("OPT normalized moments")
foreach(println, opt_moments)

# For a collocated explicit FD wave scheme, future/past spatial moments are zero and
# the present slot carries the spatial Laplacian. OPT is allowed to have mass/stiffness
# spread over all time slots, but large first moments indicate directional bias/asymmetry.
(fd_moments=fd_moments, opt_moments=opt_moments)


## 15. Homogeneous OPT Stencil Test

This rebuilds the numerical OPT operator with `v = median(Marmousi)` everywhere. If the stencil remains asymmetric, the issue is in the OPT recipe / space-time basis / time split. If it becomes symmetric, the issue is material mapping in the heterogeneous case.

In [ ]:
# Homogeneous OPT test using the same recipe and grid size.
# If this is still asymmetric, the issue is in the recipe/time-space Galerkin stencil itself,
# not in Marmousi material mapping.
hom_velocity = fill(vmed, size(models[1]))
homModel = (models=[hom_velocity], modelPoints=modelPoints, Δ=Δ, modelName="homogeneous_OPT_debug")
homParams = @strdict optRec=optRec modelFam=homModel absorbingBoundaries=nothing maskedRegionInSpace=nothing representation="matrixfree"

numOpt_hom = numericalOperatorConstruction(homParams)
numOps_hom = numOpt_hom["numOperators"]
preparedHom = prepareLinearSystem(numOps_hom)

hom_stencil_cmp = opt_fd_stencil_comparison(numOps_hom, hom_velocity, Δnum, sourcePoint; iExpr=1, field=1, atol=0.0)
@show hom_stencil_cmp.opt_pivot hom_stencil_cmp.fd_pivot
@show hom_stencil_cmp.opt_by_time

print_stencil_table(hom_stencil_cmp.fd_normalized; title="FD normalized homogeneous reference stencil")
print_stencil_table(hom_stencil_cmp.opt_normalized; title="OPT normalized homogeneous stencil at sourcePoint")

hom_opt_moments = print_stencil_moments("OPT homogeneous", hom_stencil_cmp.opt_table)
hom_opt_moments


In [ ]:
# Optional propagation with the homogeneous OPT operator.
hom_field_responses = source_field_response(preparedHom, sourcePoint, timeSignal_probe; amplitude=1.0)
hom_best_source_field = argmax(getproperty.(hom_field_responses, :b_norm))
@show hom_best_source_field

timeSignal_hom = source_time_samples(Nt_debug, Δnum[end], preparedHom.timePointsUsedForOneStep; t0=source_t0, f0=source_f0)
sourceFull_hom = make_sourceFull(preparedHom, point_source_weights(preparedHom, sourcePoint), timeSignal_hom; iField=hom_best_source_field, amplitude=source_amplitude_opt)
frames_hom_opt = propagate_linear_frames(preparedHom, sourceFull_hom, Nt_debug; store_every=store_every_debug, blowup_limit=1e12)
report_hom_opt = wavefield_snapshot_report(frames_hom_opt)
@show length(frames_hom_opt) report_hom_opt[1] report_hom_opt[end]

frames_hom_zoom, hom_source_zoom, hom_crop_ranges = crop_frames_around_source(frames_hom_opt, sourcePoint; radius=zoom_radius)
fig_hom_opt_zoom = plot_wave_snapshots(frames_hom_zoom; sourcePoint=hom_source_zoom, title="homogeneous OPT numerical operator zoom")
fig_hom_opt_zoom


## 16. OPT Control-Parameter Stencil Comparison

This tests whether the asymmetry is caused by an underdetermined/ill-conditioned Taylor inversion. Compare current `(pointsInSpace=3, supplementaryOrder=2)` against `(3,0)` and `(5,2)`.

In [ ]:
function build_opt_for_controls(; pointsInSpace_test=3, supplementaryOrder_test=0, model_velocity=models[1])
    concrete = @strdict famousEquationType Δ orderBtime orderBspace pointsInTime fieldItpl materItpl
    concrete["pointsInSpace"] = pointsInSpace_test
    concrete["supplementaryOrder"] = supplementaryOrder_test
    opt = makeOPTsemiSymbolic(concrete)
    rec = opt["recette"]
    mp = getModelPoints(model_velocity, pointsInTime, rec.numbersOfTheSystem.numbersOfTheSystemL.timeMarching)
    tm = (models=[model_velocity], modelPoints=mp, Δ=Δ, modelName="OPT_points$(pointsInSpace_test)_supp$(supplementaryOrder_test)")
    par = @strdict optRec=opt modelFam=tm absorbingBoundaries=nothing maskedRegionInSpace=nothing representation="matrixfree"
    no = numericalOperatorConstruction(par)
    nop = no["numOperators"]
    pl = prepareLinearSystem(nop)
    return opt, nop, pl
end

function compare_opt_controls_at_source(cases)
    rows = NamedTuple[]
    for case in cases
        label, pspace, supp = case
        println("\n--- ", label, " pointsInSpace=", pspace, " supplementaryOrder=", supp, " ---")
        opt_case, numOps_case, prepared_case = build_opt_for_controls(pointsInSpace_test=pspace, supplementaryOrder_test=supp)
        cmp = opt_fd_stencil_comparison(numOps_case, models[1], Δnum, sourcePoint; iExpr=1, field=1, atol=0.0)
        moments, pivot = normalized_stencil_moments(cmp.opt_table)
        @show cmp.opt_pivot cmp.opt_by_time
        for m in moments
            println(m)
        end
        push!(rows, (
            label=label,
            pointsInSpace=pspace,
            supplementaryOrder=supp,
            opt_pivot=cmp.opt_pivot,
            by_time=cmp.opt_by_time,
            moments=moments,
        ))
    end
    return rows
end


In [ ]:
opt_control_cases = [
    ("current", 3, 2),
    ("no_supplementary", 3, 0),
    ("more_points", 5, 2),
]

opt_control_comparison = compare_opt_controls_at_source(opt_control_cases)
opt_control_comparison
